In [ ]:
# BigAlpha 2026 submission_v6 — flat 5-minute package
#
# All runtime modules are embedded below.  Weight shards are ASCII text files
# at this same directory level and are reassembled in memory by checkpoint.py.
from __future__ import annotations

import sys
import types
from pathlib import Path

PACKAGE_ROOT = Path.cwd()
EMBEDDED_MODULE_SOURCES = {'ba_bar_encoder': '"""Bar-level field encoder."""\n\nimport torch\nimport torch.nn as nn\n\n\nclass BarEncoder(nn.Module):\n    def __init__(self, num_fields: int, d_model: int, dropout: float = 0.1):\n        super().__init__()\n        self.embedding = nn.Linear(num_fields, d_model)\n        self.norm = nn.LayerNorm(d_model)\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, bars: torch.Tensor) -> torch.Tensor:\n        # bars: [B, D, P, F] -> [B, D, P, d_model]\n        x = self.embedding(bars)\n        x = self.norm(x)\n        x = self.dropout(x)\n        return x', 'ba_config': '"""Configuration management for BigAlpha-SSPT-30m."""\n\nimport os\nimport yaml\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional\n\n\ndef deep_merge(base: dict, override: dict) -> dict:\n    result = base.copy()\n    for k, v in override.items():\n        if k in result and isinstance(result[k], dict) and isinstance(v, dict):\n            result[k] = deep_merge(result[k], v)\n        else:\n            result[k] = v\n    return result\n\n\ndef load_config(config_path: str, overrides: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:\n    with open(config_path, "r") as f:\n        config = yaml.safe_load(f)\n\n    # Support both the research package (<repo>/bigalpha_sspt/utils/config.py)\n    # and the flat modules embedded in the one-file submission notebook.\n    module_path = Path(__file__).resolve()\n    if module_path.parent.name == "utils" and module_path.parent.parent.name == "bigalpha_sspt":\n        project_root = str(module_path.parents[2])\n    else:\n        project_root = str(module_path.parent)\n    config["_project_root"] = project_root\n\n    defaults = {\n        "_project_root": project_root,\n        "data": {\n            # Submission/private-board runs must not inherit a developer\n            # filesystem path.  A supplied config wins; otherwise callers\n            # explicitly provide BIGALPHA_DATA_DIR.\n            "data_dir": os.environ.get("BIGALPHA_DATA_DIR", ""),\n            "lookback_days": 120,\n            "min_history_days": 20,\n            "prediction_horizon_days": 1,\n            "bars_per_day": 8,\n            "cache_dir": os.path.join(project_root, "artifacts", "cache"),\n            "barra_exposure_path": "",\n            "barra_neutralization": {\n                "regression": "ols",\n                "add_intercept": True,\n                "precondition": True,\n                "include_industry": True,\n                "require_industry": False,\n                "industry_column": "industry_level1_code",\n                "weight_column": "weights",\n                "condition_number_threshold": 1e8,\n                "svd_rcond": 1e-10,\n                "proxy_exposure": {\n                    "enabled": True,\n                    "fallback_when_official_missing": True,\n                    "cache_path": "",\n                    "beta_window_days": 60,\n                    "beta_min_periods": 20,\n                    "residual_vol_window_days": 60,\n                    "residual_vol_min_periods": 20,\n                    "liquidity_window_days": 20,\n                    "size_window_days": 60,\n                    "momentum_lookback_days": 120,\n                    "momentum_skip_days": 20,\n                    "market_return_weight": "amount",\n                    "standardize": True,\n                },\n            },\n        },\n        "model": {\n            "d_model": 640,\n            "intraday_encoder_type": "transformer",\n            "intraday_layers": 4,\n            "intraday_gru_bidirectional": True,\n            "intraday_gru_hidden_size": None,\n            "temporal_layers": 10,\n            "num_heads": 10,\n            "ff_mult": 4,\n            "dropout": 0.10,\n            "attention_dropout": 0.05,\n            "activation": "gelu",\n            "pre_norm": True,\n            "use_cross_section_module": False,\n            "num_market_latents": 16,\n            "cross_section_layers": 2,\n            "cross_section_ff_mult": 4,\n        },\n        "pretrain": {\n            "use_ma_prediction": True,\n            "ma_windows": [3, 5, 10, 20],\n            "ma_target_field": "close",\n            "ma_loss_weight": 1.0,\n            "use_masked_bar": True,\n            "mask_ratio": 0.20,\n            "block_mask_probability": 0.70,\n            "masked_bar_loss_weight": 1.0,\n            "use_next_day_prediction": True,\n            "next_day_loss_weight": 0.3,\n            "ma_target_type": "relative",\n            "use_instrument_classification": False,\n            "use_industry_classification": False,\n        },\n        "loss": {\n            "regression_type": "huber",\n            "reg_loss_weight": 0.1,\n            "use_pairwise_rank": True,\n            "pair_sample_size": 32768,\n            "rank_loss_weight": 0.3,\n            "use_ic_loss": True,\n            "ic_loss_weight": 1.0,\n            "use_style_robust_ic": True,\n            "style_robust_ic_weight": 0.5,\n            "use_tail_loss": True,\n            "tail_fraction": 0.10,\n            "tail_margin": 0.50,\n            "tail_loss_weight": 0.1,\n            # Stage-2 objective: differentiable BARRA residual projection.\n            "use_risk_neutralized_target": False,\n            "risk_neutralized_target_weight": 0.5,\n        },\n        "training": {\n            "seed": 42,\n            "batch_size": 64,\n            "epochs": 100,\n            "lr": 1e-4,\n            "weight_decay": 0.01,\n            "warmup_epochs": 5,\n            "precision": "bf16",\n            "gradient_checkpointing": True,\n            "gradient_accumulation_steps": 4,\n            "max_grad_norm": 1.0,\n            "compile": False,\n            "use_sdpa": True,\n            "num_workers": 4,\n            "pin_memory": True,\n            "persistent_workers": True,\n            "group_by_date": False,\n            # Experiment-0 contract: one optimizer step/validation forward is\n            # one complete signal-date cross-section.  Disabled by default so\n            # historical configurations remain reproducible.\n            "date_level_training": False,\n            "date_level_validation": False,\n            "date_level_ddp_mode": "replicate",\n            "local_encoder_chunk_size": 0,\n            "compile": False,\n            "compile_mode": "reduce-overhead",\n            "compile_dynamic": True,\n            "valid_batch_size": 512,\n            "pretrained_checkpoint": "",\n            "reset_prediction_head": True,\n            "output_dir": "",\n            "run_name": "",\n            "log_interval": 50,\n            "eval_interval": 500,\n            "save_interval": 2000,\n            "eval_every_epochs": 1,\n        },\n        "split": {\n            "train_start": "2019-01-01",\n            "train_end": "2023-09-30",\n            "valid_start": "2023-10-01",\n            "valid_end": "2024-02-29",\n            "test_start": "2024-03-01",\n            "test_end": "2024-12-31",\n        },\n        "fields": {\n            "selected": [\n                "open", "high", "low", "close", "volume", "amount",\n                "deal_number",\n                "ask_price1", "bid_price1",\n                "ask_volume1", "bid_volume1",\n            ],\n            "excluded": [\n                "date", "instrument_id", "adjust_factor",\n                "ask_price2", "ask_price3", "bid_price2", "bid_price3",\n                "ask_volume2", "ask_volume3", "bid_volume2", "bid_volume3",\n                "ask_num_orders1", "ask_num_orders2", "ask_num_orders3",\n                "bid_num_orders1", "bid_num_orders2", "bid_num_orders3",\n            ],\n            "log_transform": ["volume", "amount", "ask_volume1", "bid_volume1", "deal_number"],\n            "preprocessing": {\n                "scaler": "standard",\n                "clip_outliers": True,\n                "clip_quantile_low": 0.001,\n                "clip_quantile_high": 0.999,\n                "fill_na": True,\n                "min_std_threshold": 1e-8,\n                "scaler_fit_dates": 48,\n                "scaler_fit_instruments_per_date": 64,\n                "scaler_fit_max_rows": 2000000,\n            },\n        },\n    }\n\n    config = deep_merge(defaults, config)\n    if not config["data"].get("data_dir"):\n        config["data"]["data_dir"] = os.environ.get("BIGALPHA_DATA_DIR", "")\n\n    if overrides:\n        _apply_overrides(config, overrides)\n\n    return config\n\n\ndef _apply_overrides(config: dict, overrides: dict, prefix: str = ""):\n    for k, v in overrides.items():\n        if isinstance(v, dict):\n            _apply_overrides(config, v, f"{prefix}{k}.")\n        else:\n            parts = (f"{prefix}{k}").split(".")\n            d = config\n            for p in parts[:-1]:\n                if p not in d:\n                    d[p] = {}\n                d = d[p]\n            d[parts[-1]] = v\n\n\ndef save_resolved_config(config: dict, path: str):\n    config_to_save = {k: v for k, v in config.items() if not k.startswith("_")}\n    with open(path, "w") as f:\n        yaml.dump(config_to_save, f, default_flow_style=False, allow_unicode=True)\n\n\ndef load_fields_config(config_path: str) -> dict:\n    with open(config_path, "r") as f:\n        return yaml.safe_load(f)\n', 'ba_data_collate': '"""Batch collate functions."""\n\nimport torch\nimport numpy as np\nfrom typing import List, Dict\n\n\ndef collate_fn_sft(batch: List[Dict]) -> Dict[str, torch.Tensor]:\n    bars_list = []\n    day_masks = []\n    labels = []\n    dates = []\n    instruments = []\n\n    max_days = max(item["bars"].shape[0] for item in batch)\n    P = batch[0]["bars"].shape[1]\n    F = batch[0]["bars"].shape[2]\n\n    for item in batch:\n        bars = item["bars"]\n        dm = item["day_mask"]\n        D = bars.shape[0]\n        if D < max_days:\n            pad = torch.zeros((max_days - D, P, F), dtype=bars.dtype)\n            bars = torch.cat([bars, pad], dim=0)\n            dm = torch.cat([dm, torch.zeros((max_days - D, P), dtype=dm.dtype)], dim=0)\n        bars_list.append(bars)\n        day_masks.append(dm)\n        labels.append(item.get("label", torch.tensor(0.0)))\n        dates.append(item.get("date", ""))\n        instruments.append(item.get("instrument", 0))\n\n    result = {\n        "bars": torch.stack(bars_list),\n        "day_mask": torch.stack(day_masks),\n        "labels": torch.stack(labels),\n        "dates": dates,\n        "instruments": instruments,\n    }\n    # Preserve optional dense supervision produced by pretraining datasets.\n    # Previously MA targets were silently dropped, so the configured MA head\n    # was instantiated but never optimized.\n    for key in ("ma_targets", "next_day_targets", "next_day_mask"):\n        if all(key in item for item in batch):\n            result[key] = torch.stack([item[key] for item in batch])\n    return result\n\n\ndef collate_fn_pretrain(batch: List[Dict]) -> Dict[str, torch.Tensor]:\n    result = collate_fn_sft(batch)\n    B, D, P, F = result["bars"].shape\n\n    mask_ratio = 0.20\n    block_prob = 0.70\n    \n    mask = torch.ones(B, D, P, F, dtype=torch.float32)\n    \n    for b in range(B):\n        # Reconstruction is defined only on observed bars.  Sampling from the\n        # padded D*P grid made most targets trivial zeros for short histories.\n        valid_indices = result["day_mask"][b].reshape(-1).nonzero(as_tuple=False).flatten()\n        if valid_indices.numel() == 0:\n            continue\n        num_mask = max(1, int(valid_indices.numel() * mask_ratio))\n        if torch.rand(1).item() < block_prob:\n            block_len = min(valid_indices.numel(), max(1, num_mask))\n            max_start = valid_indices.numel() - block_len + 1\n            start = int(torch.randint(0, max_start, (1,)).item())\n            masked_indices = valid_indices[start:start + block_len]\n        else:\n            order = torch.randperm(valid_indices.numel())[:num_mask]\n            masked_indices = valid_indices[order]\n        \n        for idx in masked_indices:\n            d = int(idx // P)\n            bar = int(idx % P)\n            mask[b, d, bar, :] = 0.0\n\n    masked_bars = result["bars"] * mask\n\n    result["mask_bars"] = mask\n    result["masked_bars"] = masked_bars\n    result["original_bars"] = result["bars"].clone()\n\n    return result\n', 'ba_data_schema': '"""Data schema definitions."""\n\nimport numpy as np\n\nOHLCV_FIELDS = ["open", "high", "low", "close", "volume", "amount"]\nMICROSTRUCTURE_FIELDS = ["deal_number", "ask_price1", "bid_price1", "ask_volume1", "bid_volume1"]\nID_COLUMNS = ["date", "instrument_id"]\nEXCLUDED_FIELDS = [\n    "adjust_factor",\n    "ask_price2", "ask_price3", "bid_price2", "bid_price3",\n    "ask_volume2", "ask_volume3", "bid_volume2", "bid_volume3",\n    "ask_num_orders1", "ask_num_orders2", "ask_num_orders3",\n    "bid_num_orders1", "bid_num_orders2", "bid_num_orders3",\n]\nDEFAULT_SELECTED_FIELDS = OHLCV_FIELDS + MICROSTRUCTURE_FIELDS\nLOG_TRANSFORM_FIELDS = ["volume", "amount", "deal_number", "ask_volume1", "bid_volume1"]\nBAR_TIMES = 8\nALL_FIELDS = ["date", "instrument_id", "adjust_factor", "high", "open", "low", "close", "deal_number",\n              "volume", "amount", "ask_price1", "ask_price2", "ask_price3", "bid_price1", "bid_price2",\n              "bid_price3", "ask_volume1", "ask_volume2", "ask_volume3", "bid_volume1", "bid_volume2",\n              "bid_volume3", "ask_num_orders1", "ask_num_orders2", "ask_num_orders3", "bid_num_orders1",\n              "bid_num_orders2", "bid_num_orders3"]', 'ba_data_samplers': '"""Samplers that preserve the cross-sectional training contract."""\n\nfrom collections import defaultdict\nfrom typing import Iterator, List\n\nimport numpy as np\nfrom torch.utils.data import Sampler\n\n\nclass DateBatchSampler(Sampler[List[int]]):\n    """Yield batches containing stocks from one trading date only.\n\n    IC, pairwise ranking and tail separation are cross-sectional objectives.\n    A normal shuffled DataLoader mixes dates, so most batches contain only a\n    handful of stocks per date and pairwise losses compare unrelated days.\n    This sampler keeps each batch on one date while still shuffling dates and\n    stocks deterministically per epoch.\n\n    With ``world_size > 1`` it forms a same-date global batch and yields one\n    contiguous shard per rank, so every DDP worker still sees the same date.\n    """\n\n    def __init__(self, dataset, batch_size: int, *, shuffle: bool = True,\n                 drop_last: bool = True, seed: int = 42,\n                 world_size: int = 1, rank: int = 0,\n                 pad_to_global_batch: bool = False,\n                 full_date: bool = False,\n                 shard_full_date: bool = False):\n        if batch_size < 2:\n            raise ValueError("batch_size must be at least 2 for cross-sectional losses")\n        self.dataset = dataset\n        self.batch_size = int(batch_size)\n        self.shuffle = shuffle\n        self.drop_last = drop_last\n        self.seed = int(seed)\n        self.world_size = int(world_size)\n        self.rank = int(rank)\n        self.pad_to_global_batch = bool(pad_to_global_batch)\n        self.full_date = bool(full_date)\n        self.shard_full_date = bool(shard_full_date)\n        if self.shard_full_date and not self.full_date:\n            raise ValueError("shard_full_date requires full_date=True")\n        if not 0 <= self.rank < self.world_size:\n            raise ValueError("rank must be in [0, world_size)")\n        self.epoch = 0\n        groups = defaultdict(list)\n        for idx, sample in enumerate(dataset.samples):\n            groups[sample[0]].append(idx)\n        self.groups = {date: indices for date, indices in groups.items()}\n\n    def set_epoch(self, epoch: int) -> None:\n        self.epoch = int(epoch)\n\n    def __iter__(self) -> Iterator[List[int]]:\n        rng = np.random.default_rng(self.seed + self.epoch)\n        dates = list(self.groups)\n        if self.shuffle:\n            rng.shuffle(dates)\n        for date in dates:\n            indices = np.asarray(self.groups[date], dtype=np.int64).copy()\n            if self.shuffle:\n                rng.shuffle(indices)\n            if self.full_date:\n                if self.shard_full_date:\n                    # All ranks receive the same date but distinct stock\n                    # shards. The model\'s ragged gather rebuilds the complete\n                    # cross-section before Market Latent; no repeat padding.\n                    local = indices[self.rank::self.world_size]\n                    if len(local):\n                        yield local.tolist()\n                elif len(indices) >= 2:\n                    # Replicated full-date reference mode.\n                    yield indices.tolist()\n                continue\n            global_batch = self.batch_size * self.world_size\n            for start in range(0, len(indices), global_batch):\n                global_slice = indices[start:start + global_batch]\n                if len(global_slice) < global_batch:\n                    if self.pad_to_global_batch:\n                        # Repeat stocks from the same date only. This keeps\n                        # every rank\'s shard the same size, which is required\n                        # by the differentiable Market Latent all-gather.\n                        if len(global_slice) < self.batch_size:\n                            continue\n                        repeats = np.resize(global_slice, global_batch)\n                        global_slice = repeats\n                    elif self.drop_last:\n                        continue\n                batch = global_slice[self.rank * self.batch_size:\n                                     (self.rank + 1) * self.batch_size].tolist()\n                if len(batch) < self.batch_size and self.drop_last:\n                    continue\n                if batch:\n                    yield batch\n\n    def __len__(self) -> int:\n        if self.full_date:\n            return sum(len(v) >= 2 for v in self.groups.values())\n        if self.pad_to_global_batch:\n            global_batch = self.batch_size * self.world_size\n            return sum(\n                ((len(v) + global_batch - 1) // global_batch)\n                for v in self.groups.values() if len(v) >= self.batch_size\n            )\n        if self.drop_last:\n            return sum(len(v) // (self.batch_size * self.world_size)\n                       for v in self.groups.values())\n        return sum((len(v) + self.batch_size * self.world_size - 1)\n                   // (self.batch_size * self.world_size)\n                   for v in self.groups.values())\n', 'ba_data_targets': '"""Target generation: labels, moving average targets, masked bar targets."""\n\nimport numpy as np\nimport pandas as pd\n\n\ndef compute_future_return(close_prices: np.ndarray, horizon: int = 1) -> np.ndarray:\n    """Compute future return for a ``[instrument, date]`` close matrix."""\n    assert close_prices.ndim == 2  # [num_instruments, num_dates]\n    result = np.full_like(close_prices, np.nan, dtype=np.float32)\n    if horizon < 1:\n        raise ValueError("horizon must be at least one trading day")\n    for t in range(close_prices.shape[1] - horizon):\n        current = close_prices[:, t]\n        future = close_prices[:, t + horizon]\n        valid = (current > 0) & (future > 0)\n        result[valid, t] = (future[valid] - current[valid]) / current[valid]\n    return result\n\n\ndef compute_daily_ma_target(close_prices: np.ndarray, window: int,\n                            relative: bool = False) -> np.ndarray:\n    """Compute daily moving average target using only past data.\n    close_prices: [num_instruments, num_dates]\n    Returns MA values for each date using window previous days.  When\n    ``relative`` is true, returns ``close[t] / MA[t] - 1``.  The relative\n    form is preferable for pretraining because absolute prices are not\n    comparable across instruments and would dominate the loss.\n    """\n    result = np.full_like(close_prices, np.nan, dtype=np.float32)\n    cumsum = np.cumsum(np.nan_to_num(close_prices, nan=0.0), axis=1)\n    valid_counts = np.cumsum(~np.isnan(close_prices), axis=1)\n    \n    for t in range(1, close_prices.shape[1]):\n        start = max(0, t - window)\n        if start == 0:\n            ma_sum = cumsum[:, t - 1]\n            count = valid_counts[:, t - 1]\n        else:\n            ma_sum = cumsum[:, t - 1] - cumsum[:, start - 1]\n            count = valid_counts[:, t - 1] - valid_counts[:, start - 1]\n        mask = count > 0\n        ma = np.full(close_prices.shape[0], np.nan, dtype=np.float32)\n        ma[mask] = ma_sum[mask] / count[mask]\n        if relative:\n            current = close_prices[:, t]\n            valid = np.isfinite(ma) & np.isfinite(current) & (ma > 0)\n            ma[valid] = current[valid] / ma[valid] - 1.0\n            ma[~valid] = np.nan\n        result[:, t] = ma\n    \n    return result\n\n\ndef compute_ma_targets(close_prices: np.ndarray, windows: list,\n                       relative: bool = False) -> np.ndarray:\n    """Compute multiple MA targets. Returns [num_instruments, num_dates, num_windows]."""\n    result = np.zeros((*close_prices.shape, len(windows)), dtype=np.float32)\n    for i, w in enumerate(windows):\n        result[:, :, i] = compute_daily_ma_target(close_prices, w, relative=relative)\n    return result\n\n\ndef generate_mask_targets(bars: np.ndarray, mask_ratio: float = 0.20, \n                           block_prob: float = 0.70, seed: int = None):\n    """Generate random masks for bar reconstruction pretraining.\n    bars: [batch, days, bars_per_day, num_fields]\n    Returns: masked_bars, mask (same shape)\n    """\n    if seed is not None:\n        rng = np.random.RandomState(seed)\n    else:\n        rng = np.random\n        \n    batch, days, bars_per_day, num_fields = bars.shape\n    mask = np.ones_like(bars, dtype=np.float32)\n    \n    for b in range(batch):\n        total_cells = days * bars_per_day\n        num_mask = int(total_cells * mask_ratio)\n        \n        if rng.random() < block_prob:\n            block_len = max(1, num_mask // 4)\n            start = rng.randint(0, max(1, total_cells - block_len))\n            flat = np.arange(total_cells)\n            masked_indices = flat[start:start + block_len]\n        else:\n            all_idx = np.arange(total_cells)\n            rng.shuffle(all_idx)\n            masked_indices = all_idx[:num_mask]\n        \n        for idx in masked_indices:\n            d = idx // bars_per_day\n            bar = idx % bars_per_day\n            mask[b, d, bar, :] = 0.0\n    \n    masked_bars = bars * mask\n    return masked_bars, mask\n\n\ndef compute_next_day_target(bars: np.ndarray) -> np.ndarray:\n    """Compute next-day prediction target: next day\'s field-wise average.\n    bars: [num_instruments, num_days, bars_per_day, num_fields]\n    Returns: [num_instruments, num_days, num_fields] - next day\'s avg per field\n    """\n    result = np.zeros_like(bars[:, :, 0, :], dtype=np.float32)\n    for t in range(bars.shape[1] - 1):\n        next_day = bars[:, t + 1, :, :]\n        valid_mask = (next_day != 0).any(axis=-1)\n        counts = valid_mask.sum(axis=1)\n        if np.any(counts > 0):\n            summed = (next_day * valid_mask[..., None]).sum(axis=1)\n            valid_rows = counts > 0\n            result[valid_rows, t, :] = summed[valid_rows] / counts[valid_rows, None]\n    return result\n', 'ba_environment': '"""Environment information capture."""\n\nimport json\nimport os\nimport platform\nimport sys\nfrom datetime import datetime\n\nimport torch\n\n\ndef get_environment_info() -> dict:\n    info = {\n        "timestamp": datetime.now().isoformat(),\n        "platform": platform.platform(),\n        "python_version": sys.version,\n        "pytorch_version": torch.__version__,\n        "cuda_available": torch.cuda.is_available(),\n    }\n    if torch.cuda.is_available():\n        info["cuda_version"] = torch.version.cuda\n        info["gpu_count"] = torch.cuda.device_count()\n        info["gpu_names"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\n        try:\n            info["gpu_memory_gb"] = [torch.cuda.get_device_properties(i).total_memory / 1e9 for i in range(torch.cuda.device_count())]\n        except:\n            pass\n        info["bf16_supported"] = torch.cuda.is_bf16_supported()\n    return info\n\n\ndef save_environment_info(output_dir: str):\n    info = get_environment_info()\n    os.makedirs(output_dir, exist_ok=True)\n    with open(os.path.join(output_dir, "environment.json"), "w") as f:\n        json.dump(info, f, indent=2)\n    return info', 'ba_ic_loss': '"""Pearson IC loss - cross-sectional correlation loss."""\n\nimport torch\nimport torch.nn as nn\n\n\ndef pearson_ic_loss(pred: torch.Tensor, target: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:\n    """\n    Compute negative Pearson IC (correlation) for cross-sectional ranking.\n    pred, target, mask: [N] for a single day\n    """\n    if mask is None:\n        mask = torch.ones_like(pred, dtype=torch.bool)\n\n    pred_valid = pred[mask]\n    target_valid = target[mask]\n\n    if len(pred_valid) < 2:\n        return torch.tensor(0.0, device=pred.device)\n\n    pred_centered = pred_valid - pred_valid.mean()\n    target_centered = target_valid - target_valid.mean()\n\n    pred_var = (pred_centered * pred_centered).mean()\n    target_var = (target_centered * target_centered).mean()\n\n    if pred_var < 1e-12 or target_var < 1e-12:\n        return torch.tensor(0.0, device=pred.device)\n\n    cov = (pred_centered * target_centered).mean()\n    corr = cov / (torch.sqrt(pred_var) * torch.sqrt(target_var))\n    return -corr\n\n\nclass ICLoss(nn.Module):\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor, date_groups: torch.Tensor) -> torch.Tensor:\n        unique_dates = date_groups.unique()\n        total_loss = 0.0\n        count = 0\n\n        for d in unique_dates:\n            day_mask = (date_groups == d)\n            day_pred = pred[day_mask]\n            day_target = target[day_mask]\n\n            if len(day_pred) < 2:\n                continue\n\n            day_loss = pearson_ic_loss(day_pred, day_target)\n            total_loss += day_loss\n            count += 1\n\n        if count == 0:\n            return torch.tensor(0.0, device=pred.device)\n\n        return total_loss / count\n\n\ndef compute_daily_ic(pred: torch.Tensor, target: torch.Tensor, date_groups: torch.Tensor) -> float:\n    unique_dates = date_groups.unique()\n    ics = []\n\n    for d in unique_dates:\n        day_mask = (date_groups == d)\n        day_pred = pred[day_mask]\n        day_target = target[day_mask]\n\n        if len(day_pred) < 2:\n            continue\n\n        pred_c = day_pred - day_pred.mean()\n        target_c = day_target - day_target.mean()\n\n        pv = (pred_c * pred_c).mean()\n        tv = (target_c * target_c).mean()\n\n        if pv < 1e-12 or tv < 1e-12:\n            continue\n\n        ic = (pred_c * target_c).mean() / (torch.sqrt(pv) * torch.sqrt(tv))\n        ics.append(ic.item())\n\n    if not ics:\n        return 0.0\n\n    return sum(ics) / len(ics)', 'ba_intraday_encoder': '"""Intraday (within-day) encoders."""\n\nimport torch\nimport torch.nn as nn\nfrom torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence\n\n\nclass IntradayEncoder(nn.Module):\n    def __init__(self, d_model: int, num_layers: int, num_heads: int,\n                 ff_mult: int = 4, dropout: float = 0.1,\n                 attention_dropout: float = 0.05, activation: str = "gelu",\n                 pre_norm: bool = True, bars_per_day: int = 8):\n        super().__init__()\n        self.d_model = d_model\n        self.num_layers = num_layers\n        self.num_heads = num_heads\n        self.pre_norm = pre_norm\n        self.bars_per_day = bars_per_day\n\n        self.bar_pos_embed = nn.Parameter(torch.randn(1, 1, bars_per_day, d_model) * 0.02)\n\n        self.day_cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)\n\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=d_model, nhead=num_heads,\n            dim_feedforward=d_model * ff_mult,\n            dropout=dropout, activation=activation,\n            batch_first=True, norm_first=pre_norm,\n        )\n        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)\n\n    def forward(self, bar_embeddings: torch.Tensor, bar_mask: torch.Tensor = None) -> torch.Tensor:\n        # bar_embeddings: [B, D, P, d_model]\n        # bar_mask: [B, D, P]\n        B, D, P, E = bar_embeddings.shape\n\n        bar_embeddings = bar_embeddings + self.bar_pos_embed[:, :, :P, :]\n\n        bar_flat = bar_embeddings.reshape(B * D, P, E)\n\n        cls_token = self.day_cls_token.expand(B * D, -1, -1)  # [B*D, 1, d_model]\n        bar_with_cls = torch.cat([cls_token, bar_flat], dim=1)  # [B*D, 1+P, d_model]\n\n        if bar_mask is not None:\n            src_mask = bar_mask.reshape(B * D, P)\n            cls_mask = torch.ones(B * D, 1, device=bar_mask.device, dtype=bar_mask.dtype)\n            full_mask = torch.cat([cls_mask, src_mask], dim=1)\n            src_key_padding_mask = (full_mask < 0.5)\n        else:\n            src_key_padding_mask = None\n\n        out = self.transformer(bar_with_cls, src_key_padding_mask=src_key_padding_mask)\n\n        day_cls = out[:, 0, :]  # [B*D, d_model]\n        if bar_mask is not None:\n            # Padding tokens are masked from attention above, but they still\n            # have non-zero output states.  Do not let them leak into the\n            # pooled daily representation.\n            valid = src_mask.to(dtype=out.dtype).unsqueeze(-1)\n            day_mean = (out[:, 1:, :] * valid).sum(dim=1)\n            day_mean = day_mean / valid.sum(dim=1).clamp(min=1.0)\n        else:\n            day_mean = out[:, 1:, :].mean(dim=1)  # [B*D, d_model]\n\n        day_repr = (day_cls + day_mean) / 2.0\n        day_repr = day_repr.reshape(B, D, E)  # [B, D, d_model]\n\n        return day_repr\n\n\nclass GRUIntradayEncoder(nn.Module):\n    """Mask-aware GRU alternative to the intraday Transformer.\n\n    The local data has only eight 30-minute bars per day, so a recurrent\n    encoder is a useful low-parameter ablation.  Packed sequences keep padded\n    bars from affecting the daily state.\n    """\n\n    def __init__(self, d_model: int, num_layers: int,\n                 dropout: float = 0.1, bars_per_day: int = 8,\n                 bidirectional: bool = True, hidden_size: int = None):\n        super().__init__()\n        self.d_model = d_model\n        self.num_layers = num_layers\n        self.bars_per_day = bars_per_day\n        self.bidirectional = bidirectional\n        self.num_directions = 2 if bidirectional else 1\n        if hidden_size is None:\n            hidden_size = max(1, d_model // self.num_directions)\n        self.hidden_size = int(hidden_size)\n        output_dim = self.hidden_size * self.num_directions\n\n        self.bar_pos_embed = nn.Parameter(torch.randn(1, 1, bars_per_day, d_model) * 0.02)\n        self.gru = nn.GRU(\n            input_size=d_model,\n            hidden_size=self.hidden_size,\n            num_layers=num_layers,\n            dropout=dropout if num_layers > 1 else 0.0,\n            batch_first=True,\n            bidirectional=bidirectional,\n        )\n        self.output_proj = (\n            nn.Identity() if output_dim == d_model else nn.Linear(output_dim, d_model)\n        )\n        self.norm = nn.LayerNorm(d_model)\n\n    def forward(self, bar_embeddings: torch.Tensor,\n                bar_mask: torch.Tensor = None) -> torch.Tensor:\n        # bar_embeddings: [B, D, P, d_model]\n        # bar_mask: [B, D, P]\n        B, D, P, E = bar_embeddings.shape\n        x = bar_embeddings + self.bar_pos_embed[:, :, :P, :]\n        x = x.reshape(B * D, P, E)\n\n        if bar_mask is None:\n            lengths = torch.full((B * D,), P, device=x.device, dtype=torch.long)\n            valid = torch.ones(B * D, P, 1, device=x.device, dtype=x.dtype)\n        else:\n            flat_mask = bar_mask.reshape(B * D, P)\n            lengths = flat_mask.sum(dim=1).to(dtype=torch.long)\n            valid = flat_mask.to(dtype=x.dtype).unsqueeze(-1)\n            x = x * valid\n\n        non_empty = lengths > 0\n        packed_lengths = lengths.clamp(min=1).cpu()\n        packed = pack_padded_sequence(\n            x, packed_lengths, batch_first=True, enforce_sorted=False,\n        )\n        packed_out, hidden = self.gru(packed)\n        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=P)\n\n        if self.bidirectional:\n            final_hidden = torch.cat([hidden[-2], hidden[-1]], dim=-1)\n        else:\n            final_hidden = hidden[-1]\n\n        valid_sum = valid.sum(dim=1).clamp(min=1.0)\n        day_mean = (out * valid).sum(dim=1) / valid_sum\n        day_repr = (final_hidden + day_mean) / 2.0\n        day_repr = self.output_proj(day_repr)\n        day_repr = self.norm(day_repr)\n        day_repr = torch.where(non_empty.unsqueeze(-1), day_repr, torch.zeros_like(day_repr))\n        return day_repr.reshape(B, D, E)\n', 'ba_logging': '"""Logging utilities."""\n\nimport json\nimport logging\nimport os\nimport sys\nfrom typing import Optional\n\n\nclass MetricsLogger:\n    def __init__(self, output_dir: str, filename: str = "metrics.jsonl"):\n        os.makedirs(output_dir, exist_ok=True)\n        self.filepath = os.path.join(output_dir, filename)\n        self._file = open(self.filepath, "a")\n\n    def log(self, metrics: dict, step: int):\n        record = {"step": step}\n        record.update({k: float(v) if hasattr(v, \'item\') else v for k, v in metrics.items()})\n        self._file.write(json.dumps(record) + "\\n")\n        self._file.flush()\n\n    def close(self):\n        self._file.close()\n\n\nclass CSVLogger:\n    def __init__(self, output_dir: str, filename: str = "training_log.csv"):\n        os.makedirs(output_dir, exist_ok=True)\n        self.filepath = os.path.join(output_dir, filename)\n        self._initialized = False\n\n    def log(self, metrics: dict, step: int):\n        record = {"step": step}\n        record.update({k: float(v) if hasattr(v, \'item\') else v for k, v in metrics.items()})\n        if not self._initialized:\n            with open(self.filepath, "w") as f:\n                f.write(",".join(record.keys()) + "\\n")\n            self._initialized = True\n        with open(self.filepath, "a") as f:\n            f.write(",".join(str(v) for v in record.values()) + "\\n")\n\n    def close(self):\n        """Compatibility hook for the trainer\'s logger shutdown path.\n\n        CSVLogger opens files per write, so there is no persistent handle to\n        close.  Keeping an explicit no-op close method makes a completed run\n        exit cleanly (and matches MetricsLogger\'s interface).\n        """\n        return None\n\n\ndef setup_logging(name: str = "bigalpha_submission", level: int = logging.INFO) -> logging.Logger:\n    logger = logging.getLogger(name)\n    if not logger.handlers:\n        logger.setLevel(level)\n        handler = logging.StreamHandler(sys.stdout)\n        handler.setLevel(level)\n        formatter = logging.Formatter(\n            "%(asctime)s [%(levelname)s] %(name)s: %(message)s",\n            datefmt="%Y-%m-%d %H:%M:%S",\n        )\n        handler.setFormatter(formatter)\n        logger.addHandler(handler)\n    return logger\n', 'ba_market_latent': '"""Optional market latent cross-section module."""\n\nimport torch\nimport torch.nn as nn\nimport torch.distributed as dist\n\ntry:\n    from torch.distributed.nn.functional import all_gather as differentiable_all_gather\nexcept (ImportError, AttributeError):\n    def differentiable_all_gather(tensor):\n        return (tensor,)\n\n\nclass MarketLatentModule(nn.Module):\n    def __init__(self, d_model: int, num_latents: int = 16, num_layers: int = 2,\n                 dropout: float = 0.1, num_heads: int = 4, ff_mult: int = 4,\n                 activation: str = "gelu", pre_norm: bool = True):\n        super().__init__()\n        self.d_model = d_model\n        self.num_latents = num_latents\n\n        self.latents = nn.Parameter(torch.randn(1, num_latents, d_model) * 0.02)\n\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=d_model, nhead=num_heads,\n            dim_feedforward=d_model * ff_mult, dropout=dropout,\n            activation=activation, batch_first=True, norm_first=pre_norm,\n        )\n        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)\n        self.norm = nn.LayerNorm(d_model)\n\n    def forward(self, stock_repr: torch.Tensor, mask: torch.Tensor = None,\n                distributed_gather: bool = True) -> torch.Tensor:\n        # stock_repr: [N, d_model] - stocks from same day\n        B, E = stock_repr.shape\n        assert E == self.d_model\n\n        # DateBatchSampler shards one same-date cross-section across DDP\n        # ranks. Gather shards before the latent transformer, then return the\n        # local slice; the differentiable collective preserves gradients.\n        local_n = stock_repr.shape[0]\n        use_distributed = bool(\n            distributed_gather and dist.is_available() and dist.is_initialized()\n        )\n        if use_distributed:\n            gathered = differentiable_all_gather(stock_repr.contiguous())\n            stock_repr_all = torch.cat(tuple(gathered), dim=0)\n            if mask is not None:\n                mask_list = [torch.empty_like(mask) for _ in range(dist.get_world_size())]\n                dist.all_gather(mask_list, mask.contiguous())\n                mask_all = torch.cat(mask_list, dim=0)\n            else:\n                mask_all = None\n        else:\n            stock_repr_all = stock_repr\n            mask_all = mask\n\n        sr = stock_repr_all.unsqueeze(0)  # [1, N_global, d_model]\n        latents = self.latents  # [1, K, d_model]\n\n        combined = torch.cat([latents, sr], dim=1)  # [1, K+N, d_model]\n\n        if mask_all is not None:\n            lat_mask = torch.ones(1, self.num_latents, device=mask_all.device, dtype=mask_all.dtype)\n            full_mask = torch.cat([lat_mask, mask_all.unsqueeze(0)], dim=1)\n            src_key_padding_mask = (full_mask < 0.5)\n        else:\n            src_key_padding_mask = None\n\n        out = self.transformer(combined, src_key_padding_mask=src_key_padding_mask)\n\n        stock_out = out[:, self.num_latents:, :]  # [1, N, d_model]\n        stock_repr_out = self.norm(stock_repr_all.unsqueeze(0) + stock_out)\n\n        stock_repr_out = stock_repr_out.squeeze(0)\n        if use_distributed:\n            rank = dist.get_rank()\n            return stock_repr_out[rank * local_n:(rank + 1) * local_n]\n        return stock_repr_out  # [N, d_model]\n', 'ba_pairwise_rank': '"""Pairwise ranking loss."""\n\nimport torch\nimport torch.nn as nn\n\n\nclass PairwiseRankingLoss(nn.Module):\n    def __init__(self, pair_sample_size: int = 32768, margin: float = 0.0):\n        super().__init__()\n        self.pair_sample_size = pair_sample_size\n        self.margin = margin\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor,\n                mask: torch.Tensor = None, rng_seed: int = None) -> torch.Tensor:\n        N = pred.shape[0]\n\n        if mask is not None:\n            pred = pred[mask.bool()]\n            target = target[mask.bool()]\n            N = pred.shape[0]\n\n        if N < 2:\n            return torch.tensor(0.0, device=pred.device)\n\n        max_pairs = N * (N - 1) // 2\n        num_pairs = min(self.pair_sample_size, max_pairs)\n\n        if rng_seed is not None:\n            generator = torch.Generator(device=pred.device)\n            generator.manual_seed(rng_seed)\n        else:\n            generator = None\n\n        if num_pairs >= max_pairs:\n            idx1 = torch.arange(N).unsqueeze(1).expand(N, N).triu(diagonal=1)\n            idx1 = idx1.nonzero(as_tuple=False)[:num_pairs, 0]\n            idx2 = torch.arange(N).unsqueeze(1).expand(N, N).triu(diagonal=1)\n            idx2 = idx2.nonzero(as_tuple=False)[:num_pairs, 1]\n        else:\n            idx1 = torch.randint(0, N, (num_pairs,), device=pred.device, generator=generator)\n            idx2 = torch.randint(0, N, (num_pairs,), device=pred.device, generator=generator)\n            same = idx1 == idx2\n            while same.any():\n                idx2[same] = torch.randint(0, N, (same.sum().item(),),\n                                            device=pred.device, generator=generator)\n                same = idx1 == idx2\n\n        pred_diff = pred[idx1] - pred[idx2]\n        target_diff = target[idx1] - target[idx2]\n\n        same_sign = (pred_diff > 0) != (target_diff > 0)\n        loss = torch.relu(self.margin - pred_diff * torch.sign(target_diff))\n        loss = loss.mean()\n\n        return loss', 'ba_parameters': '"""Parameter counting utilities."""\n\nimport json\nimport os\n\nimport torch\n\n\ndef count_parameters(model: torch.nn.Module) -> dict:\n    total = 0\n    trainable = 0\n    modules = {}\n    \n    for name, module in model.named_children():\n        m_total = sum(p.numel() for p in module.parameters())\n        m_trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)\n        total += m_total\n        trainable += m_trainable\n        modules[name] = {"total": m_total, "trainable": m_trainable}\n\n    result = {\n        "total_parameters": total,\n        "trainable_parameters": trainable,\n        "non_trainable_parameters": total - trainable,\n        "modules": modules,\n    }\n\n    total_m = total / 1e6\n    if total_m > 100:\n        raise ValueError(\n            f"Model has {total_m:.1f}M parameters, exceeding the 100M limit. "\n            f"Reduce model size."\n        )\n\n    param_size = total * 4\n    result["model_size_disk_mb_fp32"] = param_size / (1024 * 1024)\n    result["model_size_disk_mb_bf16"] = total * 2 / (1024 * 1024)\n    result["memory_forward_mb_fp32"] = param_size / (1024 * 1024) * 2\n    result["memory_forward_mb_bf16"] = total * 2 / (1024 * 1024) * 2\n    result["memory_optimizer_mb_fp32"] = param_size / (1024 * 1024) * 2\n\n    return result\n\n\ndef save_parameter_report(model: torch.nn.Module, output_dir: str):\n    report = count_parameters(model)\n    os.makedirs(output_dir, exist_ok=True)\n    with open(os.path.join(output_dir, "parameter_report.json"), "w") as f:\n        json.dump(report, f, indent=2)\n\n    print(f"\\n{\'=\'*60}")\n    print(f"Parameter Report")\n    print(f"{\'=\'*60}")\n    print(f"Total parameters:       {report[\'total_parameters\']:>12,} ({report[\'total_parameters\']/1e6:.2f}M)")\n    print(f"Trainable parameters:   {report[\'trainable_parameters\']:>12,} ({report[\'trainable_parameters\']/1e6:.2f}M)")\n    print(f"Non-trainable:          {report[\'non_trainable_parameters\']:>12,}")\n    print(f"{\'=\'*60}")\n    for name, counts in report["modules"].items():\n        print(f"  {name:<25s}: total={counts[\'total\']:>10,}, trainable={counts[\'trainable\']:>10,}")\n    print(f"{\'=\'*60}")\n    print(f"FP32 disk: {report[\'model_size_disk_mb_fp32\']:.1f} MB")\n    print(f"BF16 disk: {report[\'model_size_disk_mb_bf16\']:.1f} MB")\n    print(f"{\'=\'*60}\\n")\n    return report', 'ba_preprocessing': '"""Data preprocessing: scaling, normalization, missing value handling."""\n\nimport json\nimport os\nfrom typing import Dict, List, Optional\n\nimport numpy as np\n\n\nLOCAL_MONETARY_FIELDS = {\n    "open", "high", "low", "close", "amount",\n    "ask_price1", "ask_price2", "ask_price3",\n    "bid_price1", "bid_price2", "bid_price3",\n}\nLOCAL_OHLC_FIELDS = {"open", "high", "low", "close"}\n\n\ndef canonicalize_local_frame(data, feature_fields=None):\n    """Convert local e2e Feather rows to the shared yuan-based representation.\n\n    Local prices and ``amount`` are stored in cents. OHLC ``-1`` denotes a\n    missing value; book-price zero remains a legitimate missing-value marker.\n    """\n    import pandas as pd\n\n    out = data.copy()\n    if "date" in out.columns:\n        out["date"] = pd.to_datetime(out["date"], errors="raise")\n    for field in LOCAL_OHLC_FIELDS:\n        if field in out.columns:\n            out.loc[out[field] == -1, field] = np.nan\n    fields = feature_fields or list(out.columns)\n    for field in LOCAL_MONETARY_FIELDS.intersection(fields):\n        if field in out.columns:\n            out[field] = out[field].astype("float64") / 100.0\n    if "instrument_id" in out.columns:\n        out["key"] = out["instrument_id"].astype(str)\n    return out\n\n\nclass FieldScaler:\n    def __init__(self, scaler_type: str = "standard", clip_bounds: Optional[tuple] = None,\n                 min_std: float = 1e-8, log_fields: Optional[List[str]] = None,\n                 log_offset: float = 1.0, clip_quantiles: Optional[tuple] = None):\n        self.scaler_type = scaler_type\n        self.clip_bounds = clip_bounds\n        self.min_std = min_std\n        self.log_fields = set(log_fields or [])\n        self.log_offset = float(log_offset)\n        self.clip_quantiles = clip_quantiles\n        self.representation = "canonical_yuan"\n        self.stats = {}\n\n    def _prepare_column(self, col, name: str):\n        col = np.asarray(col, dtype=np.float32).copy()\n        if name in self.log_fields:\n            col = np.log(np.maximum(col, 0.0) + self.log_offset)\n        return col\n\n    def fit(self, data: np.ndarray, field_names: List[str]):\n        assert data.ndim == 2, f"Expected 2D data, got {data.ndim}D"\n        assert data.shape[1] == len(field_names), f"Field count mismatch: {data.shape[1]} vs {len(field_names)}"\n        \n        for i, name in enumerate(field_names):\n            col = self._prepare_column(data[:, i], name)\n            col = col[np.isfinite(col)]\n            if len(col) == 0:\n                self.stats[name] = {"mean": 0.0, "std": 1.0, "min": 0.0, "max": 1.0}\n                continue\n            stats = {}\n            if self.clip_quantiles is not None:\n                low_q, high_q = self.clip_quantiles\n                stats["clip_low"] = float(np.quantile(col, low_q))\n                stats["clip_high"] = float(np.quantile(col, high_q))\n                col = np.clip(col, stats["clip_low"], stats["clip_high"])\n            \n            if self.scaler_type == "standard":\n                mean = float(np.mean(col))\n                std = float(np.std(col))\n                std = max(std, self.min_std)\n                stats.update({"mean": mean, "std": std, "min": float(np.min(col)), "max": float(np.max(col))})\n                self.stats[name] = stats\n            elif self.scaler_type == "minmax":\n                col_min = float(np.min(col))\n                col_max = float(np.max(col))\n                rng = max(col_max - col_min, self.min_std)\n                stats.update({"min": col_min, "max": col_max, "range": rng})\n                self.stats[name] = stats\n            else:\n                raise ValueError(f"Unknown scaler type: {self.scaler_type}")\n\n    def transform(self, data: np.ndarray, field_names: List[str]) -> np.ndarray:\n        assert data.ndim == 2\n        assert data.shape[1] == len(field_names)\n        result = data.copy().astype(np.float32)\n        \n        for i, name in enumerate(field_names):\n            col = self._prepare_column(result[:, i], name)\n            stats = self.stats[name]\n            # Missing values are neutral *after* scaling.  Filling them with\n            # raw zero before standardization turns a missing price into\n            # ``-mean / std`` -- often an extreme artificial signal.\n            valid = np.isfinite(col)\n            transformed = np.zeros_like(col, dtype=np.float32)\n            if not np.any(valid):\n                result[:, i] = transformed\n                continue\n            work = col[valid]\n            if "clip_low" in stats:\n                work = np.clip(work, stats["clip_low"], stats["clip_high"])\n            \n            if self.scaler_type == "standard":\n                work = (work - stats["mean"]) / stats["std"]\n            elif self.scaler_type == "minmax":\n                work = (work - stats["min"]) / stats["range"]\n            \n            if self.clip_bounds:\n                low, high = self.clip_bounds\n                work = np.clip(work, low, high)\n            \n            transformed[valid] = work\n            result[:, i] = transformed\n        \n        return result\n\n    def save(self, path: str):\n        os.makedirs(os.path.dirname(path), exist_ok=True)\n        with open(path, "w") as f:\n            json.dump(self.to_dict(), f, indent=2)\n\n    def to_dict(self):\n        return {\n            "scaler_type": self.scaler_type,\n            "stats": self.stats,\n            "min_std": self.min_std,\n            "log_fields": sorted(self.log_fields),\n            "log_offset": self.log_offset,\n            "clip_quantiles": self.clip_quantiles,\n            "representation": self.representation,\n        }\n\n    def load(self, path: str):\n        with open(path, "r") as f:\n            data = json.load(f)\n        self.load_dict(data)\n\n    def load_dict(self, data: dict):\n        """Load scaler state from an embedded checkpoint payload."""\n        self.scaler_type = data["scaler_type"]\n        self.stats = data["stats"]\n        self.min_std = data.get("min_std", 1e-8)\n        self.log_fields = set(data.get("log_fields", []))\n        self.log_offset = float(data.get("log_offset", 1.0))\n        self.clip_quantiles = data.get("clip_quantiles")\n        self.representation = data.get("representation")\n\n\nclass LogTransform:\n    def __init__(self, fields: List[str], min_val: float = 0.0, offset: float = 1.0):\n        self.fields = set(fields)\n        self.min_val = min_val\n        self.offset = offset\n\n    def transform(self, data: np.ndarray, field_names: List[str]) -> np.ndarray:\n        result = data.copy().astype(np.float32)\n        for i, name in enumerate(field_names):\n            if name in self.fields:\n                result[:, i] = np.log(np.maximum(result[:, i], self.min_val) + self.offset)\n        return result\n\n\ndef clip_outliers(data: np.ndarray, low: float = 0.001, high: float = 0.999) -> np.ndarray:\n    result = data.copy().astype(np.float32)\n    for i in range(data.shape[1]):\n        col = result[:, i]\n        finite = col[np.isfinite(col)]\n        if len(finite) == 0:\n            continue\n        lo = np.quantile(finite, low)\n        hi = np.quantile(finite, high)\n        result[:, i] = np.clip(col, lo, hi)\n    return result\n', 'ba_pretrain_heads': '"""Pre-training task heads."""\n\nimport torch\nimport torch.nn as nn\n\n\nclass MAPredictionHead(nn.Module):\n    def __init__(self, d_model: int, num_windows: int):\n        super().__init__()\n        self.norm = nn.LayerNorm(d_model)\n        self.fc = nn.Sequential(\n            nn.Linear(d_model, d_model),\n            nn.GELU(),\n            nn.Dropout(0.1),\n            nn.Linear(d_model, num_windows),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.norm(x)\n        return self.fc(x)\n\n\nclass MaskedBarHead(nn.Module):\n    def __init__(self, d_model: int, bars_per_day: int, num_fields: int,\n                 num_layers: int = 2, num_heads: int = 4):\n        super().__init__()\n        self.recon_embed = nn.Linear(d_model, d_model)\n        self.bar_query = nn.Parameter(torch.randn(1, bars_per_day, d_model) * 0.02)\n\n        decoder_layer = nn.TransformerDecoderLayer(\n            d_model=d_model, nhead=num_heads,\n            dim_feedforward=d_model * 4, dropout=0.1,\n            activation="gelu", batch_first=True, norm_first=True,\n        )\n        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)\n        self.recon_head = nn.Linear(d_model, num_fields)\n\n    def forward(self, day_repr: torch.Tensor, day_mask: torch.Tensor = None) -> torch.Tensor:\n        # day_repr: [B*D, d_model] or [B, D, d_model]\n        if day_repr.dim() == 3:\n            B, D, E = day_repr.shape\n            day_repr = day_repr.reshape(B * D, E)\n            if day_mask is not None:\n                day_mask = day_mask.reshape(B * D, -1)\n\n        B_sq = day_repr.shape[0]  # B*D\n        bars_per_day = self.bar_query.shape[1]\n\n        memory = day_repr.unsqueeze(1)  # [B*D, 1, d_model]\n        tgt = self.bar_query.expand(B_sq, -1, -1)  # [B*D, P, d_model]\n\n        out = self.decoder(tgt, memory)  # [B*D, P, d_model]\n        recon = self.recon_head(out)  # [B*D, P, num_fields]\n\n        return recon\n\n\nclass NextDayHead(nn.Module):\n    def __init__(self, d_model: int, num_fields: int, bars_per_day: int = 8):\n        super().__init__()\n        self.norm = nn.LayerNorm(d_model)\n        self.fc = nn.Sequential(\n            nn.Linear(d_model, d_model),\n            nn.GELU(),\n            nn.Dropout(0.1),\n            nn.Linear(d_model, bars_per_day * num_fields),\n        )\n        self.bars_per_day = bars_per_day\n        self.num_fields = num_fields\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = self.norm(x)\n        out = self.fc(x)\n        return out.view(-1, self.bars_per_day, self.num_fields)', 'ba_pretrain_losses': '"""Pre-training loss functions."""\n\nimport torch\nimport torch.nn as nn\n\n\nclass MALoss(nn.Module):\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor,\n                mask: torch.Tensor = None) -> torch.Tensor:\n        if mask is None:\n            mask = torch.ones_like(target, dtype=torch.bool)\n        valid = (mask > 0.5) & torch.isfinite(target) & torch.isfinite(pred)\n        if valid.sum() == 0:\n            return torch.tensor(0.0, device=pred.device)\n        diff = pred[valid] - target[valid]\n        loss = (diff ** 2).mean()\n        return loss\n\n\nclass MaskedBarLoss(nn.Module):\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor,\n                mask: torch.Tensor) -> torch.Tensor:\n        masked = mask < 0.5\n        if masked.sum() == 0:\n            return torch.tensor(0.0, device=pred.device)\n        diff = pred[masked] - target[masked]\n        loss = (diff ** 2).mean()\n        return loss\n\n\nclass NextDayLoss(nn.Module):\n    def __init__(self):\n        super().__init__()\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor,\n                mask: torch.Tensor = None) -> torch.Tensor:\n        if mask is None:\n            mask = torch.ones_like(target, dtype=torch.bool)\n        valid = (mask > 0.5) & torch.isfinite(target) & torch.isfinite(pred)\n        if valid.sum() == 0:\n            return torch.tensor(0.0, device=pred.device)\n        diff = pred[valid] - target[valid]\n        loss = (diff ** 2).mean()\n        return loss\n\n\ndef compute_pretrain_losses(outputs: dict, batch: dict, pretrain_config: dict) -> dict:\n    losses = {}\n    total = None\n\n    if "ma_pred" in outputs and "ma_targets" in batch:\n        ma_loss = MALoss()(outputs["ma_pred"], batch["ma_targets"])\n        weight = pretrain_config.get("ma_loss_weight", 1.0)\n        losses["ma_loss"] = ma_loss.item()\n        total = weight * ma_loss if total is None else total + weight * ma_loss\n\n    if "bar_recon" in outputs and "mask_bars" in batch:\n        pred = outputs["bar_recon"]  # [B*D, P, F]\n        target = batch["original_bars"]  # [B, D, P, F]\n        mask = batch["mask_bars"]  # [B, D, P, F]\n        B, D, P, F = target.shape\n        target_flat = target.reshape(B * D, P, F)\n        mask_flat = mask.reshape(B * D, P, F)\n        masked_loss = MaskedBarLoss()(pred, target_flat, mask_flat)\n        weight = pretrain_config.get("masked_bar_loss_weight", 1.0)\n        losses["masked_bar_loss"] = masked_loss.item()\n        total = weight * masked_loss if total is None else total + weight * masked_loss\n\n    if "next_day_pred" in outputs and "next_day_targets" in batch:\n        nd_loss = NextDayLoss()(outputs["next_day_pred"], batch["next_day_targets"])\n        weight = pretrain_config.get("next_day_loss_weight", 0.3)\n        losses["next_day_loss"] = nd_loss.item()\n        total = weight * nd_loss if total is None else total + weight * nd_loss\n\n    if total is None:\n        # Keep the training loop differentiable even for a deliberately\n        # disabled/empty auxiliary-target configuration.\n        total = next(iter(outputs.values())).sum() * 0.0 if outputs else torch.tensor(0.0)\n    losses["total_pretrain_loss"] = total.item()\n\n    return losses, total\n', 'ba_regression': '"""Regression losses: MSE, Huber."""\n\nimport torch\nimport torch.nn as nn\n\n\ndef huber_loss(pred: torch.Tensor, target: torch.Tensor, delta: float = 1.0) -> torch.Tensor:\n    diff = pred - target\n    abs_diff = diff.abs()\n    quadratic = torch.clamp(abs_diff, max=delta)\n    linear = abs_diff - quadratic\n    loss = 0.5 * quadratic ** 2 + delta * linear\n    return loss.mean()\n\n\ndef mse_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n    return nn.functional.mse_loss(pred, target)\n\n\nclass RegressionLoss(nn.Module):\n    def __init__(self, loss_type: str = "huber", delta: float = 1.0):\n        super().__init__()\n        self.loss_type = loss_type\n        self.delta = delta\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:\n        if self.loss_type == "huber":\n            return huber_loss(pred, target, self.delta)\n        elif self.loss_type == "mse":\n            return mse_loss(pred, target)\n        else:\n            raise ValueError(f"Unknown loss type: {self.loss_type}")', 'ba_reproducibility': '"""Reproducibility utilities."""\n\nimport random\nimport numpy as np\nimport torch\n\n\ndef set_seed(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False', 'ba_scheduler': '"""Learning rate schedulers."""\n\nimport math\nimport torch\nimport torch.optim as optim\nfrom torch.optim.lr_scheduler import LambdaLR\n\n\ndef get_cosine_schedule_with_warmup(optimizer, warmup_steps: int, total_steps: int):\n    def lr_lambda(step):\n        if step < warmup_steps:\n            return float(step) / float(max(1, warmup_steps))\n        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))\n        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))\n    \n    return LambdaLR(optimizer, lr_lambda)', 'ba_score_head': '"""Residual gated score prediction head."""\n\nimport torch\nimport torch.nn as nn\n\n\nclass ScoreHead(nn.Module):\n    def __init__(self, d_model: int, use_nonlinear: bool = True, dropout: float = 0.1):\n        super().__init__()\n        self.d_model = d_model\n        self.use_nonlinear = use_nonlinear\n\n        self.norm = nn.LayerNorm(d_model)\n        self.linear_score = nn.Linear(d_model, 1)\n\n        if use_nonlinear:\n            self.gate_Wv = nn.Linear(d_model, d_model)\n            self.gate_Wg = nn.Linear(d_model, d_model)\n            self.gate_proj = nn.Linear(d_model, 1)\n\n        self.scale = nn.Parameter(torch.tensor(1.0))\n        self.dropout = nn.Dropout(dropout)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        # x: [B, d_model]\n        x = self.norm(x)\n\n        linear_score = self.linear_score(x)  # [B, 1]\n\n        if self.use_nonlinear:\n            v = self.gate_Wv(x)\n            g = torch.sigmoid(self.gate_Wg(x))\n            nonlinear_score = self.gate_proj(v * g)  # [B, 1]\n            score = linear_score + self.scale * nonlinear_score\n        else:\n            score = linear_score\n\n        return score.squeeze(-1)  # [B]', 'ba_style_neutralize': '"""BARRA风格剔除与本地评价模块。\n\n仅用于本地复刻平台后处理管线，不得作为正式提交依赖或推理输入。\n\n官方暴露表: bigalpha_2026_exposure\n训练入口可在官方表未配置/未找到时调用 data.proxy_exposures 构造\nproxy_barra_like 暴露；显式配置的官方路径仍会严格校验。\n"""\n\nimport numpy as np\nimport pandas as pd\nfrom typing import Dict, Tuple, Optional, List\n\nSTYLE_COLUMNS = [\n    "SIZE",\n    "BETA",\n    "MOMENTUM",\n    "RESVOL",\n    "LIQUIDTY",\n    "BTOP",\n    "EARNYILD",\n    "GROWTH",\n    "LEVERAGE",\n    "SIZENL",\n]\n\nINDUSTRY_COLUMN = "industry_level1_code"\nWEIGHT_COLUMN = "weights"\n\n\ndef _solve_ols(design: np.ndarray, target: np.ndarray,\n               weights: Optional[np.ndarray] = None,\n               condition_number_threshold: float = 1e8,\n               svd_rcond: float = 1e-10):\n    """Solve OLS and explicitly fall back to an SVD pseudoinverse.\n\n    The competition reference specifies OLS residuals and an SVD fallback for\n    ill-conditioned cross-sections.  ``numpy.linalg.lstsq`` hides that policy\n    behind its own cutoff, so we implement the decision and diagnostics here.\n    The returned condition number is ``s_max / s_min`` (larger is worse).\n    """\n    x = np.asarray(design, dtype=np.float64)\n    y = np.asarray(target, dtype=np.float64).reshape(-1)\n    if x.ndim != 2 or y.ndim != 1 or x.shape[0] != y.shape[0]:\n        raise ValueError("design and target have incompatible shapes")\n\n    if weights is not None:\n        w = np.asarray(weights, dtype=np.float64).reshape(-1)\n        valid_w = np.isfinite(w) & (w > 0)\n        if len(w) != len(y) or not valid_w.all():\n            raise ValueError("weights must be finite and strictly positive")\n        sqrt_w = np.sqrt(w / max(float(np.mean(w)), np.finfo(np.float64).eps))\n        x_solve = x * sqrt_w[:, None]\n        y_solve = y * sqrt_w\n    else:\n        x_solve, y_solve = x, y\n\n    if x_solve.size == 0:\n        raise ValueError("empty OLS design")\n    _, singular_values, _ = np.linalg.svd(x_solve, full_matrices=False)\n    s_max = float(singular_values[0]) if singular_values.size else 0.0\n    tolerance = max(s_max * float(svd_rcond), np.finfo(np.float64).eps)\n    rank = int(np.sum(singular_values > tolerance))\n    s_min = float(singular_values[-1]) if singular_values.size else 0.0\n    condition_number = (s_max / s_min if s_min > tolerance else float("inf"))\n    ill_conditioned = (\n        rank < x_solve.shape[1]\n        or not np.isfinite(condition_number)\n        or condition_number > float(condition_number_threshold)\n    )\n\n    if not ill_conditioned:\n        # Normal-equation solve is the ordinary full-rank OLS solution.  The\n        # condition check above prevents this branch for unstable designs.\n        gram = x_solve.T @ x_solve\n        rhs = x_solve.T @ y_solve\n        try:\n            beta = np.linalg.solve(gram, rhs)\n            solver = "ols_normal"\n        except np.linalg.LinAlgError:\n            ill_conditioned = True\n\n    if ill_conditioned:\n        # X^+ y = V diag(1/s) U^T y, with small singular values discarded.\n        u, s, vt = np.linalg.svd(x_solve, full_matrices=False)\n        inv_s = np.zeros_like(s)\n        invertible = s > tolerance\n        inv_s[invertible] = 1.0 / s[invertible]\n        beta = vt.T @ (inv_s * (u.T @ y_solve))\n        solver = "svd_pinv"\n\n    return beta, {\n        "rank": rank,\n        "condition_number": condition_number,\n        "solver": solver,\n        "used_svd_pinv": bool(ill_conditioned),\n        "svd_rcond": float(svd_rcond),\n    }\n\n\ndef _build_cross_section_design(\n    group: pd.DataFrame,\n    style_columns: List[str],\n    *,\n    add_intercept: bool,\n    precondition: bool,\n    include_industry: bool,\n    industry_column: str,\n) -> Tuple[np.ndarray, List[str], int]:\n    """Build a daily BARRA design matrix without the dummy-variable trap.\n\n    AlphaPurify uses an intercept plus ``drop_first`` categorical dummies,\n    while the CNE5 reference uses style exposures together with industry\n    one-hot columns.  Combining those conventions gives a full-rank design in\n    normal cross-sections and still leaves the SVD fallback for genuine rank\n    deficiency.\n    """\n    blocks = []\n    feature_names: List[str] = []\n    if add_intercept:\n        blocks.append(np.ones((len(group), 1), dtype=np.float64))\n        feature_names.append("INTERCEPT")\n\n    styles = group[style_columns].to_numpy(np.float64)\n    if precondition and styles.shape[1]:\n        mean = np.mean(styles, axis=0)\n        std = np.std(styles, axis=0)\n        std[std < 1e-12] = 1.0\n        styles = (styles - mean) / std\n    if styles.shape[1]:\n        blocks.append(styles)\n        feature_names.extend(style_columns)\n\n    n_industries = 0\n    if include_industry and industry_column in group.columns:\n        industry = group[industry_column].astype("string").str.strip()\n        categories = sorted(industry.dropna().unique().tolist())\n        n_industries = len(categories)\n        categorical = pd.Categorical(industry, categories=categories)\n        dummies = pd.get_dummies(categorical, dtype=np.float64)\n        # With an intercept, one reference industry must be omitted.  Without\n        # an intercept the full industry basis is identifiable and retained.\n        if add_intercept and len(dummies.columns):\n            dummies = dummies.iloc[:, 1:]\n        if dummies.shape[1]:\n            blocks.append(dummies.to_numpy(np.float64))\n            feature_names.extend(\n                [f"INDUSTRY[{column}]" for column in dummies.columns]\n            )\n\n    if not blocks:\n        raise ValueError("empty BARRA regression design")\n    return np.column_stack(blocks), feature_names, n_industries\n\n\ndef _weighted_r_squared(target: np.ndarray, residual: np.ndarray,\n                        weights: Optional[np.ndarray]) -> float:\n    """Compute an R² consistent with the OLS/WLS objective."""\n    if weights is None:\n        center = float(np.mean(target))\n        ss_total = float(np.sum((target - center) ** 2))\n        ss_res = float(np.sum(residual ** 2))\n    else:\n        w = np.asarray(weights, dtype=np.float64)\n        center = float(np.average(target, weights=w))\n        ss_total = float(np.sum(w * (target - center) ** 2))\n        ss_res = float(np.sum(w * residual ** 2))\n    return 1.0 - ss_res / ss_total if ss_total > 1e-12 else 0.0\n\n\ndef load_exposures(data_dir: str, start_date: Optional[str] = None,\n                   end_date: Optional[str] = None,\n                   exposure_path: Optional[str] = None) -> Optional[pd.DataFrame]:\n    """尝试从本地或缓存读取官方 BARRA 暴露数据。\n\n    优先读取本地文件 bigalpha_2026_exposure.feather。\n    若未配置且不存在则返回 None，并打印错误提示；显式配置的路径若\n    不存在或字段不完整会直接报错，避免训练悄悄退回非 BARRA 标签。\n    """\n    import os\n    configured = exposure_path or os.environ.get("BIGALPHA_BARRA_EXPOSURE_PATH")\n    if configured and not os.path.exists(configured):\n        raise FileNotFoundError(f"configured BARRA exposure file not found: {configured}")\n    possible_paths = [p for p in [configured] if p]\n    possible_paths += [\n        os.path.join(data_dir, "bigalpha_2026_exposure.feather"),\n        os.path.join(data_dir, "bigalpha_2026_exposure.parquet"),\n        os.path.join(data_dir, "bigalpha_2026_exposure.csv"),\n        os.path.join(data_dir, "barra_exposure.feather"),\n        os.path.join(os.path.dirname(data_dir), "bigalpha_2026_exposure.feather"),\n        os.path.join(os.path.dirname(data_dir), "..", "bigalpha_2026_exposure.feather"),\n    ]\n    for p in dict.fromkeys(os.path.abspath(p) for p in possible_paths):\n        if os.path.exists(p):\n            print(f"[style_neutralize] Loading exposures from: {p}")\n            suffix = os.path.splitext(p)[1].lower()\n            if suffix == ".feather":\n                import pyarrow.feather as feather\n                df = feather.read_feather(p)\n            elif suffix == ".parquet":\n                df = pd.read_parquet(p)\n            elif suffix == ".csv":\n                df = pd.read_csv(p)\n            else:\n                raise ValueError(f"unsupported BARRA exposure format: {p}")\n            return _canonicalize_exposures(df, start_date, end_date)\n\n    print("=" * 60)\n    print("[style_neutralize] WARNING: Official exposure table NOT FOUND.")\n    print("  The BARRA style neutralization requires the official table:")\n    print("    bigalpha_2026_exposure")\n    print("  containing columns: date, instrument, SIZE, BETA, MOMENTUM,")\n    print("  RESVOL, LIQUIDTY, BTOP, EARNYILD, GROWTH, LEVERAGE, SIZENL")\n    print("")\n    print("  To obtain: query via platform DAI:")\n    print("    SELECT date, instrument, SIZE, BETA, MOMENTUM, RESVOL,")\n    print("    LIQUIDTY, BTOP, EARNYILD, GROWTH, LEVERAGE, SIZENL")\n    print("    FROM bigalpha_2026_exposure")\n    print("    ORDER BY date, instrument")\n    print("")\n    print("  Without this data, style-neutralized training labels")\n    print("  cannot be computed.  Unconfigured training will fall back to raw returns.")\n    print("=" * 60)\n    return None\n\n\ndef _canonicalize_exposures(df: pd.DataFrame,\n                             start_date: Optional[str] = None,\n                             end_date: Optional[str] = None) -> pd.DataFrame:\n    """统一暴露数据格式。"""\n    df = df.copy()\n    required = {"date", "instrument", *STYLE_COLUMNS}\n    missing = sorted(required.difference(df.columns))\n    if missing:\n        raise ValueError(f"BARRA exposure table missing columns: {missing}")\n    df["date"] = pd.to_datetime(df["date"]).dt.normalize()\n    df["instrument"] = df["instrument"].astype("string").str.strip()\n    for column in [*STYLE_COLUMNS, "float_market_cap", WEIGHT_COLUMN, "ret"]:\n        if column in df.columns:\n            df[column] = pd.to_numeric(df[column], errors="coerce")\n    if INDUSTRY_COLUMN in df.columns:\n        df[INDUSTRY_COLUMN] = df[INDUSTRY_COLUMN].astype("string").str.strip()\n\n    if df.duplicated(["date", "instrument"]).any():\n        df = df.drop_duplicates(["date", "instrument"])\n\n    if start_date:\n        df = df[df["date"] >= pd.Timestamp(start_date).normalize()]\n    if end_date:\n        df = df[df["date"] <= pd.Timestamp(end_date).normalize()]\n\n    return df\n\n\ndef barra_neutralize_scores(\n    scores: pd.DataFrame,\n    exposures: pd.DataFrame,\n    config: Optional[Dict] = None,\n) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    """对每日 score 执行 BARRA 风格剔除管线。\n\n    Args:\n        scores: DataFrame with columns [date, instrument, score]\n        exposures: DataFrame with columns [date, instrument] + STYLE_COLUMNS\n        config: 可选配置 dict:\n            - regression: "ols" (default) | "wls"\n            - add_intercept: True (default)\n            - precondition: True (default)\n            - include_industry: True (when industry column exists)\n            - industry_column: "industry_level1_code"\n            - weight_column: "weights" (used by WLS)\n\n    Returns:\n        result_df: 逐股票结果 (date, instrument, score, score_winsorized,\n                   score_z, score_style_fitted, score_residual)\n        diag_df: 逐日回归诊断\n    """\n    config = config or {}\n    regression = str(config.get("regression", "ols")).lower()\n    if regression not in {"ols", "wls"}:\n        raise ValueError("regression must be \'ols\' or \'wls\'")\n    add_intercept = config.get("add_intercept", True)\n    precondition = config.get("precondition", True)\n    include_industry = config.get("include_industry", True)\n    require_industry = config.get("require_industry", False)\n    industry_column = config.get("industry_column", INDUSTRY_COLUMN)\n    weight_column = config.get("weight_column", WEIGHT_COLUMN)\n    condition_number_threshold = config.get("condition_number_threshold", 1e8)\n    svd_rcond = config.get("svd_rcond", 1e-10)\n\n    scores = scores.copy()\n    exposures = exposures.copy()\n    scores["date"] = pd.to_datetime(scores["date"]).dt.normalize()\n    exposures["date"] = pd.to_datetime(exposures["date"]).dt.normalize()\n    scores["instrument"] = scores["instrument"].astype("string").str.strip()\n    exposures["instrument"] = exposures["instrument"].astype("string").str.strip()\n\n    merged = scores.merge(exposures, on=["date", "instrument"], how="left")\n\n    if len(merged) == 0:\n        return pd.DataFrame(), pd.DataFrame()\n\n    available_styles = [c for c in STYLE_COLUMNS if c in merged.columns]\n    missing_styles = [c for c in STYLE_COLUMNS if c not in merged.columns]\n    if missing_styles:\n        print(f"[style_neutralize] Missing style columns: {missing_styles}")\n    if len(available_styles) == 0:\n        raise ValueError("No style columns available for regression")\n    if require_industry and industry_column not in merged.columns:\n        raise ValueError(f"required industry column is missing: {industry_column}")\n    use_industry = bool(include_industry and industry_column in merged.columns)\n    if regression == "wls" and weight_column not in merged.columns:\n        raise ValueError(f"WLS weight column is missing: {weight_column}")\n\n    daily_diag = []\n    results_list = []\n\n    for date, group in merged.groupby("date"):\n        n_total = len(group)\n        required_columns = ["score", *available_styles]\n        if use_industry:\n            required_columns.append(industry_column)\n        if regression == "wls":\n            required_columns.append(weight_column)\n        g = group.dropna(subset=required_columns).copy()\n        g = g[np.isfinite(g["score"].to_numpy(np.float64))]\n        g = g[\n            np.isfinite(g[available_styles].to_numpy(np.float64)).all(axis=1)\n        ]\n        if regression == "wls":\n            valid_weight = np.isfinite(g[weight_column].to_numpy(np.float64))\n            valid_weight &= g[weight_column].to_numpy(np.float64) > 0\n            g = g[valid_weight]\n\n        n_valid = len(g)\n        design, feature_names, n_industries = _build_cross_section_design(\n            g, available_styles,\n            add_intercept=add_intercept,\n            precondition=precondition,\n            include_industry=use_industry,\n            industry_column=industry_column,\n        ) if n_valid else (np.empty((0, 0)), [], 0)\n        if n_valid < max(10, design.shape[1] + 1):\n            daily_diag.append({\n                "date": date, "n_total": n_total, "n_valid": n_valid,\n                "n_features": design.shape[1], "n_industries": n_industries,\n                "regression": regression,\n                "rank": 0, "condition_number": np.nan, "r_squared": np.nan,\n                "residual_mean": np.nan, "residual_std": np.nan,\n            })\n            g = group.assign(\n                score_winsorized=np.nan, score_z=np.nan,\n                score_style_fitted=np.nan, score_residual=np.nan,\n            )\n            results_list.append(g)\n            continue\n\n        scores_arr = g["score"].to_numpy(np.float64)\n\n        # Step 1: Winsorize (per-day cross-section)\n        lo = np.quantile(scores_arr, 0.01)\n        hi = np.quantile(scores_arr, 0.99)\n        score_win = np.clip(scores_arr, lo, hi)\n\n        # Step 2: Z-score (population std)\n        mu = score_win.mean()\n        sigma = score_win.std(ddof=0)\n        if sigma < 1e-8:\n            score_z = score_win - mu\n        else:\n            score_z = (score_win - mu) / sigma\n\n        # Step 3: BARRA style + industry cross-sectional regression.\n        y = score_z\n\n        weights = None\n        if regression == "wls":\n            weights = g[weight_column].to_numpy(np.float64)\n        try:\n            coef, solver_diag = _solve_ols(\n                design, y, weights=weights,\n                condition_number_threshold=condition_number_threshold,\n                svd_rcond=svd_rcond,\n            )\n            fitted = design @ coef\n            residual = y - fitted\n        except (np.linalg.LinAlgError, ValueError, FloatingPointError):\n            coef = np.zeros(design.shape[1], dtype=np.float64)\n            fitted = np.zeros_like(y)\n            residual = y\n            solver_diag = {\n                "rank": 0,\n                "condition_number": np.inf,\n                "solver": "fallback_zero",\n                "used_svd_pinv": True,\n                "svd_rcond": float(svd_rcond),\n            }\n\n        cond_num = solver_diag["condition_number"]\n        r2 = _weighted_r_squared(y, residual, weights)\n\n        diag_record = {\n            "date": date, "n_total": n_total, "n_valid": n_valid,\n            "n_features": design.shape[1], "n_industries": n_industries,\n            "regression": regression,\n            "rank": solver_diag["rank"], "condition_number": cond_num,\n            "solver": solver_diag["solver"],\n            "used_svd_pinv": solver_diag["used_svd_pinv"],\n            "r_squared": r2,\n            "residual_mean": float(residual.mean()),\n            "residual_std": float(residual.std(ddof=0)),\n        }\n        for name, value in zip(feature_names, coef):\n            diag_record[f"coef_{name}"] = float(value)\n\n        daily_diag.append(diag_record)\n\n        g = g.assign(\n            score_winsorized=score_win,\n            score_z=score_z,\n            score_style_fitted=fitted,\n            score_residual=residual,\n        )\n        merged_insts = set(g["instrument"])\n        missing = group[~group["instrument"].isin(merged_insts)]\n        if len(missing) > 0:\n            missing = missing.assign(\n                score_winsorized=np.nan, score_z=np.nan,\n                score_style_fitted=np.nan, score_residual=np.nan,\n            )\n            g = pd.concat([g, missing], ignore_index=True)\n        results_list.append(g)\n\n    result_df = pd.concat(results_list, ignore_index=True)\n    result_df = result_df.sort_values(["date", "instrument"]).reset_index(drop=True)\n    diag_df = pd.DataFrame(daily_diag)\n\n    cols = ["date", "instrument", "score", "score_winsorized",\n            "score_z", "score_style_fitted", "score_residual"]\n    result_df = result_df[[c for c in cols if c in result_df.columns]]\n\n    return result_df, diag_df\n\n\ndef barra_ols_residuals(labels: pd.DataFrame,\n                        exposures: pd.DataFrame,\n                        config: Optional[Dict] = None) -> Tuple[pd.DataFrame, pd.DataFrame]:\n    """Compute official-style residual labels with explicit OLS/SVD policy.\n\n    ``labels`` must contain ``date``, ``instrument`` and ``score`` where score\n    is the raw future return.  Unlike score neutralization, this function does\n    not winsorize or z-score the target: it returns the raw-return residual\n    from a daily cross-sectional regression with an intercept and the supplied\n    BARRA style exposures.  When present, first-level industry codes are added\n    as categorical controls.  ``ret`` is deliberately not used as a target:\n    the caller supplies the correctly aligned future return, avoiding leakage\n    from an ambiguously timed exposure-table return column.\n    """\n    config = config or {}\n    regression = str(config.get("regression", "ols")).lower()\n    if regression not in {"ols", "wls"}:\n        raise ValueError("regression must be \'ols\' or \'wls\'")\n    add_intercept = config.get("add_intercept", True)\n    precondition = config.get("precondition", True)\n    include_industry = config.get("include_industry", True)\n    require_industry = config.get("require_industry", False)\n    industry_column = config.get("industry_column", INDUSTRY_COLUMN)\n    weight_column = config.get("weight_column", WEIGHT_COLUMN)\n    condition_number_threshold = config.get("condition_number_threshold", 1e8)\n    svd_rcond = config.get("svd_rcond", 1e-10)\n    min_observations = int(config.get("min_observations", 0))\n\n    scores = labels.copy()\n    exposures = exposures.copy()\n    required = {"date", "instrument", "score"}\n    missing = required.difference(scores.columns)\n    if missing:\n        raise ValueError(f"labels missing columns: {sorted(missing)}")\n    scores["date"] = pd.to_datetime(scores["date"]).dt.normalize()\n    exposures["date"] = pd.to_datetime(exposures["date"]).dt.normalize()\n    scores["instrument"] = scores["instrument"].astype("string").str.strip()\n    exposures["instrument"] = exposures["instrument"].astype("string").str.strip()\n    merged = scores.merge(exposures, on=["date", "instrument"], how="left")\n\n    available_styles = [c for c in STYLE_COLUMNS if c in merged.columns]\n    if not available_styles:\n        raise ValueError("No BARRA style columns available for OLS residuals")\n    if require_industry and industry_column not in merged.columns:\n        raise ValueError(f"required industry column is missing: {industry_column}")\n    use_industry = bool(include_industry and industry_column in merged.columns)\n    if regression == "wls" and weight_column not in merged.columns:\n        raise ValueError(f"WLS weight column is missing: {weight_column}")\n\n    result_groups = []\n    diagnostics = []\n    for date, group in merged.groupby("date", sort=True):\n        required_columns = ["score", *available_styles]\n        if use_industry:\n            required_columns.append(industry_column)\n        if regression == "wls":\n            required_columns.append(weight_column)\n        valid = group.dropna(subset=required_columns).copy()\n        valid = valid[np.isfinite(valid["score"].to_numpy(np.float64))]\n        valid = valid[\n            np.isfinite(valid[available_styles].to_numpy(np.float64)).all(axis=1)\n        ]\n        weights = None\n        if regression == "wls":\n            weights_array = valid[weight_column].to_numpy(np.float64)\n            valid = valid[np.isfinite(weights_array) & (weights_array > 0)]\n            weights = valid[weight_column].to_numpy(np.float64)\n        n_valid = len(valid)\n        design, feature_names, n_industries = _build_cross_section_design(\n            valid, available_styles,\n            add_intercept=add_intercept,\n            precondition=precondition,\n            include_industry=use_industry,\n            industry_column=industry_column,\n        ) if n_valid else (np.empty((0, 0)), [], 0)\n        required_observations = max(min_observations, design.shape[1] + 1)\n        base_diag = {\n            "date": date, "n_total": len(group), "n_valid": n_valid,\n            "n_features": design.shape[1], "n_industries": n_industries,\n            "regression": regression,\n        }\n        if n_valid < required_observations:\n            diagnostics.append({\n                **base_diag, "rank": 0, "condition_number": np.nan,\n                "solver": "insufficient_observations", "used_svd_pinv": False,\n                "r_squared": np.nan,\n            })\n            result_groups.append(group.assign(\n                score_fitted=np.nan, score_residual=np.nan,\n            ))\n            continue\n\n        y = valid["score"].to_numpy(np.float64)\n        try:\n            beta, solve_diag = _solve_ols(\n                design, y, weights=weights,\n                condition_number_threshold=condition_number_threshold,\n                svd_rcond=svd_rcond,\n            )\n            fitted = design @ beta\n            residual = y - fitted\n        except (np.linalg.LinAlgError, ValueError, FloatingPointError):\n            beta = np.zeros(design.shape[1], dtype=np.float64)\n            fitted = np.full(n_valid, np.nan)\n            residual = np.full(n_valid, np.nan)\n            solve_diag = {\n                "rank": 0, "condition_number": np.inf,\n                "solver": "fallback_nan", "used_svd_pinv": True,\n            }\n\n        r_squared = _weighted_r_squared(y, residual, weights)\n        diagnostic = {**base_diag, **solve_diag, "r_squared": r_squared}\n        if weights is not None:\n            diagnostic["weight_sum"] = float(weights.sum())\n            diagnostic["effective_n"] = float(\n                weights.sum() ** 2 / np.sum(weights ** 2)\n            )\n        for name, value in zip(feature_names, beta):\n            diagnostic[f"coef_{name}"] = float(value)\n        diagnostics.append(diagnostic)\n\n        valid = valid.assign(score_fitted=fitted, score_residual=residual)\n        missing_rows = group.loc[~group.index.isin(valid.index)].copy()\n        if len(missing_rows):\n            missing_rows = missing_rows.assign(score_fitted=np.nan, score_residual=np.nan)\n            valid = pd.concat([valid, missing_rows], axis=0)\n        result_groups.append(valid)\n\n    if not result_groups:\n        return pd.DataFrame(columns=["date", "instrument", "score",\n                                     "score_fitted", "score_residual"]), pd.DataFrame()\n    result = pd.concat(result_groups, ignore_index=True)\n    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)\n    result = result[["date", "instrument", "score", "score_fitted", "score_residual"]]\n    return result, pd.DataFrame(diagnostics)\n\n\ndef compute_comparison_metrics(scores_df: pd.DataFrame,\n                                exposures: pd.DataFrame,\n                                label_column: str = "label") -> Dict:\n    """比较中性化前后的 IC、ICIR、Sharpe。\n\n    Args:\n        scores_df: [date, instrument, score, label]\n        exposures: [date, instrument] + STYLE_COLUMNS\n        label_column: 标签列名，默认 "label"\n\n    Returns:\n        metrics dict with before/after comparisons\n    """\n    neutralized, diag = barra_neutralize_scores(\n        scores_df[["date", "instrument", "score"]], exposures,\n    )\n\n    neutralized["date"] = pd.to_datetime(neutralized["date"]).dt.normalize()\n    scores_df["date"] = pd.to_datetime(scores_df["date"]).dt.normalize()\n    merged = scores_df.merge(\n        neutralized[["date", "instrument", "score_residual"]],\n        on=["date", "instrument"], how="inner"\n    )\n    merged = merged.dropna(subset=["score_residual"])\n    # Use the provided label column name\n    if label_column in merged.columns:\n        merged = merged.dropna(subset=[label_column])\n\n    def _compute_ic_metrics(group):\n        """Per-date IC statistics."""\n        dates = sorted(group["date"].unique())\n        ics_raw, ics_res = [], []\n        for d in dates:\n            day = group[group["date"] == d]\n            if len(day) < 10:\n                continue\n            # IC of score_residual vs the label column\n            s = day[label_column].values\n            r = day["score_residual"].values\n            s_z = (s - s.mean()) / (s.std() + 1e-10)\n            r_z = (r - r.mean()) / (r.std() + 1e-10)\n            if np.std(s_z) > 1e-10 and np.std(r_z) > 1e-10:\n                ics_raw.append(np.corrcoef(s_z, r_z)[0, 1])\n                ics_res.append(np.corrcoef(r_z, s_z)[0, 1])\n\n        ics_raw = np.array(ics_raw)\n        ics_res = np.array(ics_res)\n        return {\n            "n_days": len(dates),\n            "ic_mean_raw": float(np.nanmean(ics_raw)),\n            "ic_std_raw": float(np.nanstd(ics_raw)),\n            "icir_raw": float(np.nanmean(ics_raw) / np.nanstd(ics_raw)) if np.nanstd(ics_raw) > 0 else 0.0,\n            "ic_mean_residual": float(np.nanmean(ics_res)),\n            "ic_std_residual": float(np.nanstd(ics_res)),\n            "icir_residual": float(np.nanmean(ics_res) / np.nanstd(ics_res)) if np.nanstd(ics_res) > 0 else 0.0,\n        }\n\n    ic_metrics = _compute_ic_metrics(merged)\n    avg_r2 = float(diag["r_squared"].mean()) if len(diag) > 0 else np.nan\n\n    return {\n        **ic_metrics,\n        "avg_style_r_squared": avg_r2,\n        "n_dates_processed": len(diag),\n        "exposure_coverage": float(diag["n_valid"].sum() / max(diag["n_total"].sum(), 1)),\n        "daily_diagnostics": diag,\n    }\n', 'ba_tail_loss': '"""Tail separation loss - separate top/bottom stocks."""\n\nimport torch\nimport torch.nn as nn\n\n\nclass TailSeparationLoss(nn.Module):\n    def __init__(self, tail_fraction: float = 0.10, margin: float = 0.50):\n        super().__init__()\n        self.tail_fraction = tail_fraction\n        self.margin = margin\n\n    def forward(self, pred: torch.Tensor, target: torch.Tensor,\n                mask: torch.Tensor = None) -> torch.Tensor:\n        if mask is not None:\n            pred = pred[mask.bool()]\n            target = target[mask.bool()]\n\n        N = pred.shape[0]\n        k = max(1, int(N * self.tail_fraction))\n\n        if k < 2:\n            return torch.tensor(0.0, device=pred.device)\n\n        _, top_indices = torch.topk(target, k, largest=True)\n        _, bottom_indices = torch.topk(target, k, largest=False)\n\n        top_mean = pred[top_indices].mean()\n        bottom_mean = pred[bottom_indices].mean()\n\n        loss = torch.relu(self.margin - top_mean + bottom_mean)\n        return loss', 'ba_temporal_encoder': '"""Cross-day temporal transformer encoder."""\n\nimport math\nimport torch\nimport torch.nn as nn\n\n\nclass TemporalEncoder(nn.Module):\n    def __init__(self, d_model: int, num_layers: int, num_heads: int,\n                 ff_mult: int = 4, dropout: float = 0.1,\n                 attention_dropout: float = 0.05, activation: str = "gelu",\n                 pre_norm: bool = True, max_lookback: int = 240,\n                 use_sdpa: bool = True):\n        super().__init__()\n        self.d_model = d_model\n        self.num_layers = num_layers\n        self.pre_norm = pre_norm\n\n        self.day_pos_embed = nn.Parameter(torch.randn(1, max_lookback, d_model) * 0.02)\n\n        encoder_layer = nn.TransformerEncoderLayer(\n            d_model=d_model, nhead=num_heads,\n            dim_feedforward=d_model * ff_mult,\n            dropout=dropout, activation=activation,\n            batch_first=True, norm_first=pre_norm,\n        )\n        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)\n\n    def forward(self, day_repr: torch.Tensor, day_mask: torch.Tensor = None) -> torch.Tensor:\n        # day_repr: [B, D, d_model]\n        B, D, E = day_repr.shape\n\n        pos = self.day_pos_embed[:, :D, :]\n        day_repr = day_repr + pos\n\n        if day_mask is not None:\n            day_mask_2d = day_mask.max(dim=-1)[0] if day_mask.dim() == 3 else day_mask\n            src_key_padding_mask = (day_mask_2d < 0.5)\n        else:\n            src_key_padding_mask = None\n\n        out = self.transformer(day_repr, src_key_padding_mask=src_key_padding_mask)\n\n        return out  # [B, D, d_model]\n\n    @property\n    def attention_store(self):\n        return None', 'ba_training_checkpoint': '"""Checkpoint saving and loading."""\n\nimport os\nimport torch\nfrom typing import Dict, Any, Optional\n\n\ndef save_checkpoint(model: torch.nn.Module, optimizer: torch.optim.Optimizer,\n                    scheduler: Any, epoch: int, step: int, config: dict,\n                    metrics: dict, path: str):\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    checkpoint = {\n        "epoch": epoch,\n        "step": step,\n        "model_state_dict": model.state_dict(),\n        "optimizer_state_dict": optimizer.state_dict() if optimizer else None,\n        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,\n        "config": {k: v for k, v in config.items() if not k.startswith("_")},\n        "preprocessor": config.get("_preprocessor"),\n        "metrics": metrics,\n    }\n    torch.save(checkpoint, path)\n\n\ndef load_checkpoint(path: str, model: torch.nn.Module,\n                    optimizer: Optional[torch.optim.Optimizer] = None,\n                    scheduler: Any = None,\n                    device: str = "cpu") -> Dict[str, Any]:\n    checkpoint = torch.load(path, map_location=device)\n    model.load_state_dict(checkpoint["model_state_dict"])\n    \n    if optimizer and checkpoint.get("optimizer_state_dict"):\n        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])\n    if scheduler and checkpoint.get("scheduler_state_dict"):\n        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])\n    \n    return checkpoint\n', 'ba_data_indexing': '"""Data indexing: date/instrument indices, bar positions."""\n\nimport os\nimport pickle\nfrom collections import defaultdict\nfrom typing import Dict, List, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.feather as feather\n\nfrom ba_preprocessing import canonicalize_local_frame\n\n\nclass DataIndex:\n    """Index the locally cached bar files.\n\n    The index cache is deliberately tied to the source-file manifest.  A cache\n    built before a data refresh must never be reused with a changed universe or\n    date range, since that can silently misalign labels and feature arrays.\n    """\n\n    CACHE_VERSION = 2\n\n    def __init__(self, data_dir: str):\n        self.data_dir = data_dir\n        self.files = sorted([f for f in os.listdir(data_dir) if f.endswith(".feather")])\n        self._cache_path = os.path.join(data_dir, "..", "cache", "data_index.pkl")\n        self.dates = []\n        self.instruments = []\n        self.date_to_idx = {}\n        self.idx_to_date = {}\n        self.instrument_to_idx = {}\n        self.idx_to_instrument = {}\n        self.date_instruments = {}\n        self.instrument_dates = {}\n        self.date_bars = {}\n\n    def _source_manifest(self):\n        """Small, deterministic fingerprint for the Feather source files."""\n        return [\n            (name, os.stat(os.path.join(self.data_dir, name)).st_size,\n             os.stat(os.path.join(self.data_dir, name)).st_mtime_ns)\n            for name in self.files\n        ]\n\n    def _build_from_files(self):\n        dates_set = set()\n        instruments_set = set()\n        date_instruments = defaultdict(set)\n        instrument_dates = defaultdict(set)\n        date_bars = {}\n\n        for f in self.files:\n            path = os.path.join(self.data_dir, f)\n            df = feather.read_feather(path, columns=["date", "instrument_id"])\n            dates = pd.to_datetime(df["date"], errors="raise").dt.date\n            instruments = df["instrument_id"].astype("int64")\n            dates_set.update(dates.unique())\n            instruments_set.update(instruments.unique())\n\n            # Iterate only over date/instrument groups, not every bar.  The\n            # raw data contains millions of bars but only ~1k instruments.\n            grouped = pd.DataFrame({"date": dates, "instrument": instruments})\n            for date, group in grouped.groupby("date", sort=False):\n                date_instruments[date].update(group["instrument"].unique())\n            for instrument, group in grouped.groupby("instrument", sort=False):\n                instrument_dates[int(instrument)].update(group["date"].unique())\n\n            # This field is informational; report the maximum intraday bars\n            # per instrument rather than the entire universe\'s row count.\n            daily = grouped.groupby(["date", "instrument"], sort=False).size()\n            for date, count in daily.groupby(level=0).max().items():\n                date_bars[date] = max(date_bars.get(date, 0), int(count))\n\n        self.dates = sorted(dates_set)\n        self.instruments = sorted(instruments_set)\n        self.date_to_idx = {d: i for i, d in enumerate(self.dates)}\n        self.idx_to_date = {v: k for k, v in self.date_to_idx.items()}\n        self.instrument_to_idx = {inst: i for i, inst in enumerate(self.instruments)}\n        self.idx_to_instrument = {v: k for k, v in self.instrument_to_idx.items()}\n        self.date_instruments = {d: sorted(date_instruments[d]) for d in self.dates}\n        self.instrument_dates = {inst: sorted(instrument_dates[inst]) for inst in self.instruments}\n        self.date_bars = {d: date_bars.get(d, 0) for d in self.dates}\n\n    def _save_cache(self):\n        os.makedirs(os.path.dirname(self._cache_path), exist_ok=True)\n        data = {\n            "cache_version": self.CACHE_VERSION,\n            "data_dir": os.path.abspath(self.data_dir),\n            "source_manifest": self._source_manifest(),\n            "dates": self.dates, "instruments": self.instruments,\n            "date_instruments": self.date_instruments,\n            "instrument_dates": self.instrument_dates,\n            "date_bars": self.date_bars,\n        }\n        with open(self._cache_path, "wb") as f:\n            pickle.dump(data, f)\n\n    def _load_cache(self) -> bool:\n        if not os.path.exists(self._cache_path):\n            return False\n        try:\n            with open(self._cache_path, "rb") as f:\n                data = pickle.load(f)\n            if (\n                data.get("cache_version") != self.CACHE_VERSION\n                or data.get("data_dir") != os.path.abspath(self.data_dir)\n                or data.get("source_manifest") != self._source_manifest()\n            ):\n                return False\n            self.dates = data["dates"]\n            self.instruments = data["instruments"]\n            self.date_instruments = data["date_instruments"]\n            self.instrument_dates = data["instrument_dates"]\n            self.date_bars = data["date_bars"]\n            self.date_to_idx = {d: i for i, d in enumerate(self.dates)}\n            self.idx_to_date = {v: k for k, v in self.date_to_idx.items()}\n            self.instrument_to_idx = {inst: i for i, inst in enumerate(self.instruments)}\n            self.idx_to_instrument = {v: k for k, v in self.instrument_to_idx.items()}\n            return True\n        except Exception:\n            return False\n\n    def build(self) -> dict:\n        if self._load_cache():\n            pass\n        else:\n            self._build_from_files()\n            self._save_cache()\n\n        if not self.dates:\n            raise RuntimeError(f"No dated bar records found in {self.data_dir}")\n\n        return {\n            "num_dates": len(self.dates),\n            "num_instruments": len(self.instruments),\n            "date_range": (str(self.dates[0]), str(self.dates[-1])),\n            "total_files": len(self.files),\n            "avg_bars_per_day": np.mean(list(self.date_bars.values())),\n        }\n\n    def get_trading_dates(self, start_date: str, end_date: str) -> List:\n        start = pd.Timestamp(start_date).date()\n        end = pd.Timestamp(end_date).date()\n        return [d for d in self.dates if start <= d <= end]\n\n\nclass DataLoader:\n    def __init__(self, data_dir: str, index: DataIndex):\n        self.data_dir = data_dir\n        self.index = index\n        self._file_cache = {}\n        \n    def _load_file(self, year_month: str) -> pd.DataFrame:\n        if year_month not in self._file_cache:\n            fname = f"{year_month}.0.feather"\n            path = os.path.join(self.data_dir, fname)\n            if os.path.exists(path):\n                self._file_cache[year_month] = feather.read_feather(path)\n            else:\n                self._file_cache[year_month] = None\n        return self._file_cache[year_month]\n    \n    def _get_year_month(self, date) -> str:\n        d = pd.Timestamp(date)\n        return f"{d.year}{d.month:02d}"\n    \n    def get_daily_data(self, date, instruments: Optional[List[int]] = None, \n                        fields: Optional[List[str]] = None) -> pd.DataFrame:\n        ym = self._get_year_month(date)\n        df = self._load_file(ym)\n        if df is None:\n            return pd.DataFrame()\n        \n        date_mask = df["date"].dt.date == date\n        df_day = df[date_mask].copy()\n        if instruments is not None:\n            df_day = df_day[df_day["instrument_id"].isin(instruments)]\n        \n        if fields is not None:\n            cols = ["date", "instrument_id"] + [f for f in fields if f in df_day.columns]\n            df_day = df_day[cols]\n        \n        return canonicalize_local_frame(\n            df_day.sort_values(["instrument_id", "date"]),\n            fields,\n        )\n    \n    def clear_cache(self):\n        self._file_cache.clear()\n', 'ba_data_dataset': '"""PyTorch Dataset for BigAlpha 30m bar data."""\n\nimport hashlib\nimport json\nimport os\nimport numpy as np\nimport torch\nimport pandas as pd\nimport pyarrow.feather as feather\nfrom torch.utils.data import Dataset\nfrom typing import Dict, List, Optional, Tuple\n\nfrom ba_data_indexing import DataIndex, DataLoader\nfrom ba_preprocessing import FieldScaler, LogTransform, clip_outliers, canonicalize_local_frame\nfrom ba_data_targets import compute_ma_targets, compute_future_return, compute_daily_ma_target\n\n\n# Module-level cache for BARRA exposure data.  Keying by source prevents a\n# second experiment (or a test fixture) from silently reusing another\n# directory\'s official factor table.\n_exposure_cache = {}\n\n\ndef _source_manifest(data_dir: str):\n    """Return the source-file fingerprint used to validate derived arrays."""\n    files = sorted(f for f in os.listdir(data_dir) if f.endswith(".feather"))\n    return [\n        [name, int(os.stat(os.path.join(data_dir, name)).st_size),\n         int(os.stat(os.path.join(data_dir, name)).st_mtime_ns)]\n        for name in files\n    ]\n\n\ndef _cache_metadata_path(cache_path: str) -> str:\n    return f"{cache_path}.meta.json"\n\n\ndef _load_validated_array(cache_path: Optional[str], metadata: dict):\n    """Load an array only when it matches its data/index/field contract."""\n    if not cache_path or not os.path.exists(cache_path):\n        return None\n    try:\n        with open(_cache_metadata_path(cache_path), "r", encoding="utf-8") as f:\n            cached_metadata = json.load(f)\n        if cached_metadata != metadata:\n            return None\n        array = np.load(cache_path, allow_pickle=False)\n        if list(array.shape) != metadata["shape"]:\n            return None\n        return array\n    except (OSError, ValueError, json.JSONDecodeError):\n        return None\n\n\ndef _save_validated_array(cache_path: Optional[str], array: np.ndarray, metadata: dict) -> None:\n    if not cache_path:\n        return\n    cache_dir = os.path.dirname(cache_path)\n    if cache_dir:\n        os.makedirs(cache_dir, exist_ok=True)\n    np.save(cache_path, array)\n    with open(_cache_metadata_path(cache_path), "w", encoding="utf-8") as f:\n        json.dump(metadata, f, sort_keys=True)\n\n\ndef _field_cache_key(fields: List[str]) -> str:\n    return hashlib.sha256("\\0".join(fields).encode("utf-8")).hexdigest()[:12]\n\n\ndef get_exposures(data_dir: str, force_reload: bool = False,\n                  exposure_path: Optional[str] = None):\n    """Load official BARRA exposures, cached by directory/path."""\n    global _exposure_cache\n    configured = exposure_path or os.environ.get("BIGALPHA_BARRA_EXPOSURE_PATH")\n    cache_key = os.path.abspath(configured) if configured else os.path.abspath(data_dir)\n    if cache_key in _exposure_cache and not force_reload:\n        return _exposure_cache[cache_key]\n    try:\n        from ba_style_neutralize import load_exposures\n        _exposure_cache[cache_key] = load_exposures(\n            data_dir, exposure_path=exposure_path,\n        )\n    except Exception as exc:\n        _exposure_cache[cache_key] = None\n        if configured:\n            raise RuntimeError(\n                f"Unable to load configured BARRA exposure file {configured}: {exc}"\n            ) from exc\n    return _exposure_cache[cache_key]\n\n\ndef _proxy_exposure_config(neutralization_config: Optional[Dict]) -> Dict:\n    return (neutralization_config or {}).get("proxy_exposure", {}) or {}\n\n\ndef _effective_neutralization_config(config: Optional[Dict],\n                                     exposures: Optional[pd.DataFrame]) -> Dict:\n    """Return the regression config that is valid for the supplied exposure table."""\n    cfg = dict(config or {})\n    cfg.pop("proxy_exposure", None)\n    if exposures is not None and exposures.get("exposure_source", pd.Series(dtype=str)).eq(\n        "proxy_barra_like"\n    ).any():\n        industry_column = cfg.get("industry_column", "industry_level1_code")\n        if industry_column not in exposures.columns:\n            cfg["include_industry"] = False\n            cfg["require_industry"] = False\n    return cfg\n\n\ndef _load_proxy_exposures_if_enabled(index, data_dir: str,\n                                     neutralization_config: Optional[Dict]):\n    proxy_cfg = _proxy_exposure_config(neutralization_config)\n    if not (\n        proxy_cfg.get("enabled", False)\n        and proxy_cfg.get("fallback_when_official_missing", True)\n    ):\n        return None\n    from ba_data_proxy_exposures import build_proxy_exposures\n    return build_proxy_exposures(data_dir, index, proxy_cfg)\n\n\ndef compute_residual_labels(index, data_dir, close_matrix, label_field="close",\n                             prediction_horizon=1, exposure_path: Optional[str] = None,\n                             neutralization_config: Optional[Dict] = None):\n    """Compute residual return labels using BARRA neutralization.\n\n    Uses the official exposure table if available.  When configured, missing\n    official exposures fall back to proxy BARRA-like exposures built from local\n    bars; otherwise labels fall back to raw returns.\n    Returns: residual_labels [n_inst, n_dates], r2_stats dict\n    """\n    # Keep the no-argument call compatible with small downstream fixtures that\n    # monkeypatch get_exposures(data_dir), while allowing production runs to\n    # point at the official export explicitly.\n    exposures = (\n        get_exposures(data_dir)\n        if exposure_path is None\n        else get_exposures(data_dir, exposure_path=exposure_path)\n    )\n    source = "official_barra"\n    if exposures is None:\n        exposures = _load_proxy_exposures_if_enabled(\n            index, data_dir, neutralization_config,\n        )\n        if exposures is not None:\n            source = "proxy_barra_like"\n            print("[dataset] Official BARRA exposures unavailable; using proxy "\n                  "BARRA-like exposures from local bar data")\n\n    if exposures is None:\n        return compute_future_return(close_matrix, prediction_horizon), {"available": False}\n\n    # Build raw return labels first\n    raw_labels = compute_future_return(close_matrix, prediction_horizon)\n\n    # Build scores DataFrame in platform format\n    import pandas as pd\n    from ba_style_neutralize import barra_ols_residuals\n\n    n_inst, n_dates = raw_labels.shape\n    records = []\n    for di in range(n_dates):\n        for ii in range(n_inst):\n            v = raw_labels[ii, di]\n            if np.isfinite(v):\n                records.append({\n                    "date": index.dates[di],\n                    "instrument": str(index.instruments[ii]),\n                    "score": float(v),\n                })\n\n    scores_df = pd.DataFrame(records)\n    effective_config = _effective_neutralization_config(\n        neutralization_config, exposures,\n    )\n    try:\n        result_df, diag_df = barra_ols_residuals(\n            scores_df, exposures, config=effective_config,\n        )\n        # Map back to matrix\n        residual_labels = np.full_like(raw_labels, np.nan, dtype=np.float32)\n        date_to_idx = {d: i for i, d in enumerate(index.dates)}\n        inst_to_idx = {str(i): ii for ii, i in enumerate(index.instruments)}\n        for _, row in result_df.iterrows():\n            # ``barra_neutralize_scores`` returns normalized pandas\n            # Timestamps while the data index deliberately stores ``date``\n            # objects.  They compare unequal as dictionary keys.\n            di = date_to_idx.get(pd.Timestamp(row["date"]).date())\n            ii = inst_to_idx.get(row["instrument"])\n            if di is not None and ii is not None:\n                residual_labels[ii, di] = float(row.get("score_residual", np.nan)) if pd.notna(row.get("score_residual")) else np.nan\n\n        n_finite = int(np.isfinite(residual_labels).sum())\n        if n_finite == 0:\n            print("[dataset] Residual labels unavailable after neutralization; "\n                  "falling back to raw returns")\n            return raw_labels, {"available": False, "error": "no_finite_residual_labels"}\n\n        avg_r2 = float(diag_df["r_squared"].mean()) if len(diag_df) > 0 else 0.0\n        svd_dates = int(diag_df.get("used_svd_pinv", pd.Series(dtype=bool)).fillna(False).sum())\n        insufficient_dates = int(\n            (diag_df.get("solver", pd.Series(dtype=str)) == "insufficient_observations").sum()\n        )\n        raw_finite = max(int(np.isfinite(raw_labels).sum()), 1)\n        stats = {\n            "available": True,\n            "avg_r2": avg_r2,\n            "n_dates": len(diag_df),\n            "svd_fallback_dates": svd_dates,\n            "insufficient_dates": insufficient_dates,\n            "label_coverage": n_finite / raw_finite,\n            "regression": effective_config.get("regression", "ols"),\n            "include_industry": effective_config.get("include_industry", True),\n            "exposure_source": source,\n            "exposure_is_proxy": source == "proxy_barra_like",\n        }\n        print(f"[dataset] Residual labels computed: avg R²={avg_r2:.4f}, "\n              f"{len(diag_df)} dates processed, {n_finite} labels, "\n              f"SVD fallback on {svd_dates} dates")\n        return residual_labels, stats\n    except Exception as e:\n        if exposure_path or os.environ.get("BIGALPHA_BARRA_EXPOSURE_PATH"):\n            raise RuntimeError(\n                "Configured BARRA exposures could not be used for OLS residual labels"\n            ) from e\n        print(f"[dataset] Residual labels failed ({e}), falling back to raw returns")\n        return raw_labels, {"available": False, "error": str(e)}\n\n\ndef build_bar_array_cache(data_dir: str, index: DataIndex, selected_fields: List[str],\n                           cache_path: str = None) -> np.ndarray:\n    """Build and cache a [num_instruments, num_dates, 8, num_fields] bar array for O(1) lookup."""\n    n_inst = len(index.instruments)\n    n_dates = len(index.dates)\n    n_fields = len(selected_fields)\n    metadata = {\n        "version": 3,\n        "kind": "bar_array",\n        "data_dir": os.path.abspath(data_dir),\n        "source_manifest": _source_manifest(data_dir),\n        "selected_fields": list(selected_fields),\n        "instruments": [int(instrument) for instrument in index.instruments],\n        "dates": [str(d) for d in index.dates],\n        "shape": [n_inst, n_dates, 8, n_fields],\n    }\n    cached = _load_validated_array(cache_path, metadata)\n    if cached is not None:\n        return cached\n\n    bar_array = np.zeros((n_inst, n_dates, 8, n_fields), dtype=np.float32)\n    inst_to_idx = {inst: i for i, inst in enumerate(index.instruments)}\n    date_to_idx = {d: i for i, d in enumerate(index.dates)}\n\n    files = sorted([f for f in os.listdir(data_dir) if f.endswith(".feather")])\n    for fname in files:\n        df = feather.read_feather(os.path.join(data_dir, fname),\n                                   columns=["date", "instrument_id"] + selected_fields)\n        df = canonicalize_local_frame(df, selected_fields)\n        # Preserve chronological intraday order and write every bar in a\n        # vectorised assignment.  ``iterrows`` made a full cache rebuild take\n        # hours on the multi-year universe.\n        df = df.sort_values(["instrument_id", "date"])\n        df["_ii"] = df["instrument_id"].map(inst_to_idx)\n        df["_di"] = df["date"].dt.date.map(date_to_idx)\n        df = df.dropna(subset=["_ii", "_di"])\n        df["_slot"] = df.groupby(["_ii", "_di"], sort=False).cumcount()\n        df = df[df["_slot"] < bar_array.shape[2]]\n        if not df.empty:\n            ii = df["_ii"].to_numpy(dtype=np.intp)\n            di = df["_di"].to_numpy(dtype=np.intp)\n            slot = df["_slot"].to_numpy(dtype=np.intp)\n            values = df[selected_fields].to_numpy(dtype=np.float32, copy=False)\n            bar_array[ii, di, slot] = values\n\n    _save_validated_array(cache_path, bar_array, metadata)\n    \n    return bar_array\n\n\ndef build_close_matrix(data_dir: str, index: DataIndex, label_field: str = "close",\n                        cache_path: str = None) -> np.ndarray:\n    """Build close price matrix [num_instruments x num_dates] efficiently."""\n    n_inst = len(index.instruments)\n    n_dates = len(index.dates)\n    metadata = {\n        "version": 3,\n        "kind": "close_matrix",\n        "data_dir": os.path.abspath(data_dir),\n        "source_manifest": _source_manifest(data_dir),\n        "label_field": label_field,\n        "instruments": [int(instrument) for instrument in index.instruments],\n        "dates": [str(d) for d in index.dates],\n        "shape": [n_inst, n_dates],\n    }\n    cached = _load_validated_array(cache_path, metadata)\n    if cached is not None:\n        return cached\n\n    close_matrix = np.full((n_inst, n_dates), np.nan, dtype=np.float32)\n    \n    inst_to_idx = {inst: i for i, inst in enumerate(index.instruments)}\n    date_to_idx = {d: i for i, d in enumerate(index.dates)}\n\n    files = sorted([f for f in os.listdir(data_dir) if f.endswith(".feather")])\n    for fname in files:\n        df = feather.read_feather(os.path.join(data_dir, fname),\n                                  columns=["date", "instrument_id", label_field])\n        df = canonicalize_local_frame(df, [label_field])\n        # The final intraday bar is the daily close.  Stable chronological\n        # sorting makes repeated vectorized assignment retain that row.\n        df = df.sort_values(["instrument_id", "date"], kind="stable")\n        ii = df["instrument_id"].map(inst_to_idx)\n        di = df["date"].dt.date.map(date_to_idx)\n        valid = ii.notna() & di.notna()\n        if valid.any():\n            close_matrix[\n                ii[valid].to_numpy(dtype=np.intp),\n                di[valid].to_numpy(dtype=np.intp),\n            ] = df.loc[valid, label_field].to_numpy(dtype=np.float32, copy=False)\n\n    _save_validated_array(cache_path, close_matrix, metadata)\n\n    return close_matrix\n\n\nclass FastBar30mDataset(Dataset):\n    """Fast bar dataset using preloaded data chunks."""\n    \n    def __init__(self, index: DataIndex, dates: List, selected_fields: List[str],\n                 lookback_days: int = 60, min_history_days: int = 20):\n        self.index = index\n        self.dates = sorted(dates)\n        self.selected_fields = selected_fields\n        self.lookback_days = lookback_days\n        self.min_history_days = min_history_days\n        \n        selected_dates = set(self.dates)\n\n        # Build positions once.  Repeated ``list.index`` calls made dataset\n        # construction quadratic in each instrument\'s trading history.\n        self.samples = []\n        for instrument, inst_dates in index.instrument_dates.items():\n            for d_pos, date in enumerate(inst_dates):\n                if date in selected_dates and d_pos >= min_history_days:\n                    self.samples.append((date, instrument, d_pos))\n    \n    def __len__(self):\n        return len(self.samples)\n    \n    def __getitem__(self, idx):\n        sample_date, instrument, d_pos = self.samples[idx]\n        inst_dates = self.index.instrument_dates[instrument]\n        \n        # The signal is generated after the final bar of sample_date.  The\n        # current day is therefore valid input and must be included; slicing\n        # only up to d_pos silently discarded the most recent trading day.\n        end = d_pos + 1\n        start = max(0, end - self.lookback_days)\n        lookback_dates = inst_dates[start:end]\n        \n        bars = np.zeros((self.lookback_days, 8, len(self.selected_fields)), dtype=np.float32)\n        day_mask = np.zeros((self.lookback_days, 8), dtype=np.float32)\n        \n        offset = self.lookback_days - len(lookback_dates)\n        for i, d in enumerate(lookback_dates):\n            li = offset + i\n            day_data = self._get_day_bars(d, instrument)\n            if len(day_data) > 0:\n                n = min(len(day_data), 8)\n                bars[li, :n] = day_data[:n]\n                day_mask[li, :n] = 1.0\n        \n        return {\n            "bars": torch.tensor(bars, dtype=torch.float32),\n            "day_mask": torch.tensor(day_mask, dtype=torch.float32),\n            "date": str(sample_date),\n            "instrument": instrument,\n        }\n\n    def _get_day_bars(self, date, instrument):\n        return np.zeros((0, len(self.selected_fields)), dtype=np.float32)\n\n\nclass FastSSFTDataset(FastBar30mDataset):\n    """Fast supervised fine-tuning dataset."""\n    \n    def __init__(self, index: DataIndex, data_dir: str, dates: List,\n                 selected_fields: List[str], scaler: FieldScaler,\n                 lookback_days: int = 60, min_history_days: int = 20,\n                 label_field: str = "close", prediction_horizon: int = 1,\n                 ma_windows: List[int] = None, clip_low: float = 0.001,\n                 clip_high: float = 0.999, max_label_date=None,\n                 require_future_label: bool = True,\n                 ma_target_type: str = "absolute",\n                 include_next_day_target: bool = False,\n                 barra_exposure_path: Optional[str] = None,\n                 barra_neutralization_config: Optional[Dict] = None,\n                 compute_supervised_labels: bool = True):\n        super().__init__(index, dates, selected_fields, lookback_days, min_history_days)\n        \n        self.data_dir = data_dir\n        self.scaler = scaler\n        self.label_field = label_field\n        self.prediction_horizon = prediction_horizon\n        self.ma_windows = ma_windows or []\n        self.max_label_date = max_label_date\n        self.num_fields = len(selected_fields)\n        self.require_future_label = require_future_label\n        self.ma_target_type = ma_target_type\n        self.include_next_day_target = include_next_day_target\n        self.barra_exposure_path = barra_exposure_path\n        self.barra_neutralization_config = barra_neutralization_config or {}\n        self.compute_supervised_labels = compute_supervised_labels\n\n        # Use the global trading-date axis for the horizon.  An\n        # instrument-specific "next available" date would turn a suspension\n        # into an unintended multi-day return and previously let it become a\n        # zero label later in __getitem__.\n        all_date_to_idx = {d: i for i, d in enumerate(index.dates)}\n        self.samples = [\n            (d, inst, d_pos) for d, inst, d_pos in self.samples\n            if (\n                not require_future_label\n                or all_date_to_idx[d] + prediction_horizon < len(index.dates)\n            )\n            and (\n                max_label_date is None\n                or not require_future_label\n                or index.dates[all_date_to_idx[d] + prediction_horizon] <= max_label_date\n            )\n        ]\n        \n        # Build or load fast bar array cache\n        cache_dir = os.path.join(os.path.dirname(data_dir), "cache")\n        # The field signature prevents accidental reuse when two experiments\n        # select the same number of fields in a different order.\n        bar_cache_path = os.path.join(\n            cache_dir, f"bar_array_v3_{_field_cache_key(selected_fields)}.npy")\n        close_cache_path = os.path.join(cache_dir, f"close_matrix_v3_{label_field}.npy")\n        \n        print("Loading bar array cache...")\n        self.bar_array = build_bar_array_cache(data_dir, index, selected_fields, bar_cache_path)\n        \n        print("Loading close matrix...")\n        self.close_matrix = build_close_matrix(data_dir, index, label_field, close_cache_path)\n        \n        self.all_date_to_idx = all_date_to_idx\n        self.all_inst_to_idx = {inst: i for i, inst in enumerate(index.instruments)}\n        \n        self.return_labels = compute_future_return(self.close_matrix, prediction_horizon)\n\n        # Pretraining only needs MA/next-day reconstruction targets.  Avoid\n        # loading the potentially large official BARRA table and regressing\n        # every date when no supervised return label will be consumed.\n        if compute_supervised_labels:\n            self.residual_labels, self.residual_r2_stats = compute_residual_labels(\n                index, data_dir, self.close_matrix, label_field, prediction_horizon,\n                exposure_path=barra_exposure_path,\n                neutralization_config=self.barra_neutralization_config,\n            )\n        else:\n            self.residual_labels = None\n            self.residual_r2_stats = {"available": False, "skipped": True}\n\n        if require_future_label:\n            label_matrix = (\n                self.residual_labels\n                if self.residual_r2_stats.get("available", False)\n                else self.return_labels\n            )\n            # Do not convert missing target values into zero returns.  This\n            # covers suspended instruments and missing BARRA exposures alike.\n            self.samples = [\n                (d, inst, d_pos) for d, inst, d_pos in self.samples\n                if np.isfinite(label_matrix[\n                    self.all_inst_to_idx[inst], self.all_date_to_idx[d]\n                ])\n            ]\n\n        if self.ma_windows:\n            self.ma_targets = np.zeros(\n                (len(index.instruments), len(index.dates), len(self.ma_windows)),\n                dtype=np.float32)\n            for w_idx, w in enumerate(self.ma_windows):\n                self.ma_targets[:, :, w_idx] = compute_daily_ma_target(\n                    self.close_matrix, w,\n                    relative=(self.ma_target_type == "relative"),\n                )\n    \n    def _get_day_bars(self, date, instrument):\n        di = self.all_date_to_idx.get(date)\n        ii = self.all_inst_to_idx.get(instrument)\n        if di is None or ii is None:\n            return np.zeros((0, self.num_fields), dtype=np.float32)\n        day_bars = self.bar_array[ii, di]  # [8, num_fields]\n        # Return only non-zero rows (valid bars)\n        valid = (day_bars != 0).any(axis=1)\n        day_bars = day_bars[valid]\n        if len(day_bars) and self.scaler is not None:\n            day_bars = self.scaler.transform(day_bars, self.selected_fields)\n        return day_bars\n    \n    def __getitem__(self, idx):\n        sample = super().__getitem__(idx)\n        sample_date = sample["date"]\n        instrument = sample["instrument"]\n        \n        di = self.all_date_to_idx.get(pd.Timestamp(sample_date).date())\n        ii = self.all_inst_to_idx.get(instrument)\n        \n        label = 0.0 if not self.require_future_label else np.nan\n        if self.require_future_label and di is not None and ii is not None:\n            # Use residual labels if available, otherwise raw returns\n            if self.residual_r2_stats.get("available", False):\n                label = self.residual_labels[ii, di]\n            else:\n                label = self.return_labels[ii, di]\n        if self.require_future_label and not np.isfinite(label):\n            raise RuntimeError(\n                f"Missing supervised label for instrument={instrument}, date={sample_date}; "\n                "sample filtering and target matrix are inconsistent."\n            )\n        \n        sample["label"] = torch.tensor(label, dtype=torch.float32)\n        \n        if self.ma_windows and ii is not None and di is not None:\n            ma = self.ma_targets[ii, di]\n            sample["ma_targets"] = torch.tensor(ma, dtype=torch.float32)\n\n        if self.include_next_day_target:\n            next_target = np.zeros(self.num_fields, dtype=np.float32)\n            next_mask = np.zeros(self.num_fields, dtype=np.float32)\n            next_di = None if di is None else di + 1\n            if (\n                next_di is not None and next_di < len(self.index.dates)\n                and (self.max_label_date is None\n                     or self.index.dates[next_di] <= self.max_label_date)\n            ):\n                next_bars = self._get_day_bars(self.index.dates[next_di], instrument)\n                if len(next_bars):\n                    finite = np.isfinite(next_bars)\n                    counts = finite.sum(axis=0)\n                    valid = counts > 0\n                    next_target[valid] = np.where(\n                        valid, np.nansum(next_bars, axis=0) / np.maximum(counts, 1), 0.0,\n                    )[valid]\n                    next_mask[valid] = 1.0\n            sample["next_day_targets"] = torch.tensor(next_target, dtype=torch.float32)\n            sample["next_day_mask"] = torch.tensor(next_mask, dtype=torch.float32)\n\n        return sample\n\n\nclass FastPretrainDataset(FastSSFTDataset):\n    """Fast pre-training dataset."""\n    \n    def __init__(self, index: DataIndex, data_dir: str, dates: List,\n                 selected_fields: List[str], scaler: FieldScaler,\n                 lookback_days: int = 60, min_history_days: int = 20,\n                 label_field: str = "close",\n                 ma_windows: List[int] = None,\n                 mask_ratio: float = 0.20, block_mask_prob: float = 0.70,\n                 max_label_date=None, ma_target_type: str = "relative",\n                 include_next_day_target: bool = True,\n                 barra_exposure_path: Optional[str] = None,\n                 barra_neutralization_config: Optional[Dict] = None):\n        super().__init__(index, data_dir, dates, selected_fields, scaler,\n                         lookback_days, min_history_days, label_field, 1,\n                         ma_windows, max_label_date=max_label_date,\n                         require_future_label=False,\n                         ma_target_type=ma_target_type,\n                         include_next_day_target=include_next_day_target,\n                         barra_exposure_path=barra_exposure_path,\n                         barra_neutralization_config=barra_neutralization_config,\n                         compute_supervised_labels=False)\n        self.mask_ratio = mask_ratio\n        self.block_mask_prob = block_mask_prob\n\n\n# Backwards compatibility aliases\nBar30mDataset = FastBar30mDataset\nSSFTDataset = FastSSFTDataset\nPretrainDataset = FastPretrainDataset\n', 'ba_data_proxy_exposures': '"""Build BARRA-like proxy exposures from local 30-minute bar data.\n\nThese exposures are a fallback for experiments when the official\n``bigalpha_2026_exposure`` table is unavailable.  They intentionally cover\nonly factors that can be inferred from price and liquidity history; fundamental\nstyle columns are neutral constants and must not be presented as official\nBARRA data.\n"""\n\nimport hashlib\nimport json\nimport os\nfrom typing import Dict, Optional\n\nimport numpy as np\nimport pandas as pd\nimport pyarrow.feather as feather\n\nfrom ba_style_neutralize import STYLE_COLUMNS\nfrom ba_preprocessing import canonicalize_local_frame\n\n\n_proxy_exposure_cache = {}\n\n\ndef _source_manifest(data_dir: str):\n    files = sorted(f for f in os.listdir(data_dir) if f.endswith(".feather"))\n    return [\n        [name, int(os.stat(os.path.join(data_dir, name)).st_size),\n         int(os.stat(os.path.join(data_dir, name)).st_mtime_ns)]\n        for name in files\n    ]\n\n\ndef _proxy_config(config: Optional[Dict]) -> Dict:\n    cfg = dict(config or {})\n    return {\n        "beta_window_days": int(cfg.get("beta_window_days", 60)),\n        "beta_min_periods": int(cfg.get("beta_min_periods", 20)),\n        "residual_vol_window_days": int(cfg.get("residual_vol_window_days", 60)),\n        "residual_vol_min_periods": int(cfg.get("residual_vol_min_periods", 20)),\n        "liquidity_window_days": int(cfg.get("liquidity_window_days", 20)),\n        "size_window_days": int(cfg.get("size_window_days", 60)),\n        "momentum_lookback_days": int(cfg.get("momentum_lookback_days", 120)),\n        "momentum_skip_days": int(cfg.get("momentum_skip_days", 20)),\n        "market_return_weight": str(cfg.get("market_return_weight", "amount")),\n        "standardize": bool(cfg.get("standardize", True)),\n    }\n\n\ndef _cache_path(data_dir: str, metadata: Dict, configured_path: Optional[str]) -> str:\n    if configured_path:\n        return configured_path\n    payload = json.dumps(metadata, sort_keys=True, default=str).encode("utf-8")\n    key = hashlib.sha256(payload).hexdigest()[:12]\n    cache_dir = os.path.abspath(os.path.join(data_dir, "..", "cache"))\n    return os.path.join(cache_dir, f"proxy_barra_like_exposure_v1_{key}.feather")\n\n\ndef _cache_meta_path(path: str) -> str:\n    return f"{path}.meta.json"\n\n\ndef _load_cached(path: str, metadata: Dict) -> Optional[pd.DataFrame]:\n    if not os.path.exists(path):\n        return None\n    try:\n        with open(_cache_meta_path(path), "r", encoding="utf-8") as f:\n            cached_metadata = json.load(f)\n        if cached_metadata != metadata:\n            return None\n        return feather.read_feather(path)\n    except (OSError, ValueError, json.JSONDecodeError):\n        return None\n\n\ndef _save_cached(path: str, frame: pd.DataFrame, metadata: Dict) -> None:\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    feather.write_feather(frame, path)\n    with open(_cache_meta_path(path), "w", encoding="utf-8") as f:\n        json.dump(metadata, f, sort_keys=True)\n\n\ndef _build_daily_matrices(data_dir: str, index):\n    n_inst = len(index.instruments)\n    n_dates = len(index.dates)\n    close = np.full((n_inst, n_dates), np.nan, dtype=np.float64)\n    amount = np.full((n_inst, n_dates), np.nan, dtype=np.float64)\n    volume = np.full((n_inst, n_dates), np.nan, dtype=np.float64)\n\n    inst_to_idx = {inst: i for i, inst in enumerate(index.instruments)}\n    date_to_idx = {d: i for i, d in enumerate(index.dates)}\n\n    for fname in sorted(f for f in os.listdir(data_dir) if f.endswith(".feather")):\n        path = os.path.join(data_dir, fname)\n        df = feather.read_feather(\n            path, columns=["date", "instrument_id", "close", "amount", "volume"],\n        )\n        df = canonicalize_local_frame(df, ["close", "amount", "volume"])\n        df = df.sort_values(["instrument_id", "date"], kind="stable")\n        df["_ii"] = df["instrument_id"].map(inst_to_idx)\n        df["_di"] = df["date"].dt.date.map(date_to_idx)\n        df = df.dropna(subset=["_ii", "_di"])\n        if df.empty:\n            continue\n\n        grouped = df.groupby(["_ii", "_di"], sort=False)\n        daily = grouped.agg(\n            close=("close", "last"),\n            amount=("amount", "sum"),\n            volume=("volume", "sum"),\n        ).reset_index()\n        ii = daily["_ii"].to_numpy(dtype=np.intp)\n        di = daily["_di"].to_numpy(dtype=np.intp)\n        close[ii, di] = daily["close"].to_numpy(dtype=np.float64)\n        amount[ii, di] = daily["amount"].to_numpy(dtype=np.float64)\n        volume[ii, di] = daily["volume"].to_numpy(dtype=np.float64)\n\n    amount[~np.isfinite(amount) | (amount <= 0)] = np.nan\n    volume[~np.isfinite(volume) | (volume <= 0)] = np.nan\n    return close, amount, volume\n\n\ndef _daily_returns(close: np.ndarray) -> np.ndarray:\n    out = np.full_like(close, np.nan, dtype=np.float64)\n    prev = close[:, :-1]\n    curr = close[:, 1:]\n    valid = np.isfinite(prev) & np.isfinite(curr) & (prev > 0)\n    out[:, 1:] = np.where(valid, curr / prev - 1.0, np.nan)\n    return out\n\n\ndef _weighted_market_return(ret: np.ndarray, amount: np.ndarray,\n                            mode: str = "amount") -> np.ndarray:\n    market = np.full(ret.shape[1], np.nan, dtype=np.float64)\n    for di in range(ret.shape[1]):\n        y = ret[:, di]\n        valid = np.isfinite(y)\n        if not valid.any():\n            continue\n        weights = None\n        if mode == "amount":\n            w = amount[:, di]\n            w_valid = valid & np.isfinite(w) & (w > 0)\n            if w_valid.any():\n                weights = np.zeros_like(y, dtype=np.float64)\n                weights[w_valid] = w[w_valid]\n        if weights is None or weights.sum() <= 0:\n            market[di] = float(np.mean(y[valid]))\n        else:\n            market[di] = float(np.sum(y * weights) / np.sum(weights))\n    return market\n\n\ndef _rolling_mean(matrix: np.ndarray, window: int, min_periods: int = 1) -> np.ndarray:\n    out = np.full_like(matrix, np.nan, dtype=np.float64)\n    for di in range(matrix.shape[1]):\n        start = max(0, di - window + 1)\n        block = matrix[:, start:di + 1]\n        finite = np.isfinite(block)\n        count = finite.sum(axis=1)\n        total = np.where(finite, block, 0.0).sum(axis=1)\n        valid = count >= min_periods\n        out[valid, di] = total[valid] / count[valid]\n    return out\n\n\ndef _rolling_std(matrix: np.ndarray, window: int, min_periods: int) -> np.ndarray:\n    out = np.full_like(matrix, np.nan, dtype=np.float64)\n    for di in range(matrix.shape[1]):\n        start = max(0, di - window + 1)\n        block = matrix[:, start:di + 1]\n        finite = np.isfinite(block)\n        count = finite.sum(axis=1)\n        total = np.where(finite, block, 0.0).sum(axis=1)\n        mean = np.divide(total, count, out=np.zeros_like(total), where=count > 0)\n        centered = np.where(finite, block - mean[:, None], 0.0)\n        var = np.divide(\n            (centered ** 2).sum(axis=1), count,\n            out=np.full_like(total, np.nan), where=count > 0,\n        )\n        valid = count >= min_periods\n        out[valid, di] = np.sqrt(np.maximum(var[valid], 0.0))\n    return out\n\n\ndef _rolling_beta(ret: np.ndarray, market_ret: np.ndarray,\n                  window: int, min_periods: int) -> np.ndarray:\n    beta = np.full_like(ret, np.nan, dtype=np.float64)\n    for di in range(ret.shape[1]):\n        start = max(0, di - window + 1)\n        x = market_ret[start:di + 1]\n        y = ret[:, start:di + 1]\n        finite = np.isfinite(y) & np.isfinite(x)[None, :]\n        count = finite.sum(axis=1)\n        x_block = np.broadcast_to(x, y.shape)\n        x_sum = np.where(finite, x_block, 0.0).sum(axis=1)\n        y_sum = np.where(finite, y, 0.0).sum(axis=1)\n        x_mean = np.divide(x_sum, count, out=np.zeros_like(x_sum), where=count > 0)\n        y_mean = np.divide(y_sum, count, out=np.zeros_like(y_sum), where=count > 0)\n        x_centered = np.where(finite, x_block - x_mean[:, None], 0.0)\n        y_centered = np.where(finite, y - y_mean[:, None], 0.0)\n        cov = np.divide(\n            (x_centered * y_centered).sum(axis=1), count,\n            out=np.full_like(x_sum, np.nan), where=count > 0,\n        )\n        var = np.divide(\n            (x_centered ** 2).sum(axis=1), count,\n            out=np.full_like(x_sum, np.nan), where=count > 0,\n        )\n        valid = (count >= min_periods) & np.isfinite(var) & (var > 1e-12)\n        beta[valid, di] = cov[valid] / var[valid]\n    return beta\n\n\ndef _momentum(close: np.ndarray, lookback: int, skip: int) -> np.ndarray:\n    out = np.full_like(close, np.nan, dtype=np.float64)\n    if lookback <= skip:\n        return out\n    for di in range(lookback, close.shape[1]):\n        recent = close[:, di - skip]\n        past = close[:, di - lookback]\n        valid = np.isfinite(recent) & np.isfinite(past) & (past > 0)\n        out[valid, di] = recent[valid] / past[valid] - 1.0\n    return out\n\n\ndef _cross_section_zscore(matrix: np.ndarray, fill_value: float = 0.0) -> np.ndarray:\n    out = np.full_like(matrix, fill_value, dtype=np.float64)\n    for di in range(matrix.shape[1]):\n        x = matrix[:, di]\n        valid = np.isfinite(x)\n        if not valid.any():\n            continue\n        mu = float(np.mean(x[valid]))\n        sigma = float(np.std(x[valid]))\n        if sigma < 1e-12:\n            out[valid, di] = 0.0\n        else:\n            out[valid, di] = (x[valid] - mu) / sigma\n    return out\n\n\ndef _size_nonlinear(size: np.ndarray) -> np.ndarray:\n    out = np.zeros_like(size, dtype=np.float64)\n    for di in range(size.shape[1]):\n        x = size[:, di]\n        valid = np.isfinite(x)\n        if valid.sum() < 3:\n            continue\n        x_valid = x[valid]\n        y = x_valid ** 3\n        design = np.column_stack([np.ones(len(x_valid)), x_valid])\n        coef = np.linalg.pinv(design) @ y\n        out[valid, di] = y - design @ coef\n    return _cross_section_zscore(out)\n\n\ndef _date_weights(liquidity_base: np.ndarray) -> np.ndarray:\n    weights = np.zeros_like(liquidity_base, dtype=np.float64)\n    for di in range(liquidity_base.shape[1]):\n        w = liquidity_base[:, di]\n        valid = np.isfinite(w) & (w > 0)\n        if valid.any():\n            weights[valid, di] = w[valid] / np.sum(w[valid])\n    return weights\n\n\ndef build_proxy_exposures(data_dir: str, index, config: Optional[Dict] = None) -> pd.DataFrame:\n    """Construct and cache BARRA-like exposures from local bar data."""\n    cfg = _proxy_config(config)\n    configured_cache_path = (config or {}).get("cache_path") or None\n    metadata = {\n        "version": 1,\n        "kind": "proxy_barra_like_exposure",\n        "data_dir": os.path.abspath(data_dir),\n        "source_manifest": _source_manifest(data_dir),\n        "instruments": [int(instrument) for instrument in index.instruments],\n        "dates": [str(d) for d in index.dates],\n        "config": cfg,\n    }\n    path = _cache_path(data_dir, metadata, configured_cache_path)\n    cache_key = os.path.abspath(path)\n    if cache_key in _proxy_exposure_cache:\n        return _proxy_exposure_cache[cache_key]\n    cached = _load_cached(path, metadata)\n    if cached is not None:\n        cached["date"] = pd.to_datetime(cached["date"]).dt.normalize()\n        cached["instrument"] = cached["instrument"].astype("string").str.strip()\n        _proxy_exposure_cache[cache_key] = cached\n        return cached\n\n    print(f"[proxy_exposures] Building proxy BARRA-like exposures: {path}")\n    close, amount, volume = _build_daily_matrices(data_dir, index)\n    ret = _daily_returns(close)\n    market_ret = _weighted_market_return(\n        ret, amount, mode=cfg["market_return_weight"],\n    )\n    beta = _rolling_beta(\n        ret, market_ret, cfg["beta_window_days"], cfg["beta_min_periods"],\n    )\n    residual = ret - beta * market_ret[None, :]\n    resvol = _rolling_std(\n        residual, cfg["residual_vol_window_days"], cfg["residual_vol_min_periods"],\n    )\n    momentum = _momentum(\n        close, cfg["momentum_lookback_days"], cfg["momentum_skip_days"],\n    )\n\n    amount_liq = _rolling_mean(amount, cfg["liquidity_window_days"], min_periods=1)\n    volume_liq = _rolling_mean(volume, cfg["liquidity_window_days"], min_periods=1)\n    liquidity_base = np.where(np.isfinite(amount_liq), amount_liq, volume_liq)\n    liquidity = np.log1p(liquidity_base)\n    size_base = _rolling_mean(amount, cfg["size_window_days"], min_periods=1)\n    size = np.log1p(size_base)\n    siznl = _size_nonlinear(_cross_section_zscore(size))\n\n    style_matrices = {\n        "SIZE": size,\n        "BETA": beta,\n        "MOMENTUM": momentum,\n        "RESVOL": resvol,\n        "LIQUIDTY": liquidity,\n        "BTOP": np.zeros_like(size),\n        "EARNYILD": np.zeros_like(size),\n        "GROWTH": np.zeros_like(size),\n        "LEVERAGE": np.zeros_like(size),\n        "SIZENL": siznl,\n    }\n    if cfg["standardize"]:\n        for column in ["SIZE", "BETA", "MOMENTUM", "RESVOL", "LIQUIDTY"]:\n            style_matrices[column] = _cross_section_zscore(style_matrices[column])\n\n    weights = _date_weights(liquidity_base)\n    float_market_cap = np.where(np.isfinite(size_base), size_base, np.nan)\n\n    records = []\n    for di, date in enumerate(index.dates):\n        date_value = pd.Timestamp(date).normalize()\n        for ii, instrument in enumerate(index.instruments):\n            row = {\n                "date": date_value,\n                "instrument": str(instrument),\n                "float_market_cap": float(float_market_cap[ii, di])\n                if np.isfinite(float_market_cap[ii, di]) else np.nan,\n                "weights": float(weights[ii, di]),\n                "ret": float(ret[ii, di]) if np.isfinite(ret[ii, di]) else np.nan,\n                "exposure_source": "proxy_barra_like",\n            }\n            for column in STYLE_COLUMNS:\n                value = style_matrices[column][ii, di]\n                row[column] = float(value) if np.isfinite(value) else 0.0\n            records.append(row)\n\n    frame = pd.DataFrame(records)\n    frame = frame[[\n        "date", "instrument", *STYLE_COLUMNS,\n        "float_market_cap", "weights", "ret", "exposure_source",\n    ]]\n    _save_cached(path, frame, metadata)\n    _proxy_exposure_cache[cache_key] = frame\n    return frame\n', 'ba_sspt30m': '"""Main SSPT-30m model combining all components."""\n\nimport torch\nimport torch.nn as nn\nimport torch.distributed as dist\nimport torch.nn.functional as F\nfrom typing import Optional, Dict, List\n\ntry:\n    from torch.distributed.nn.functional import all_gather as differentiable_all_gather\nexcept (ImportError, AttributeError):\n    # Public inference is single-process.  Older cloud PyTorch releases do\n    # not expose the differentiable distributed helper, so retain an import-\n    # safe local fallback while leaving DDP behavior unchanged where supported.\n    def differentiable_all_gather(tensor):\n        return (tensor,)\n\nfrom ba_bar_encoder import BarEncoder\nfrom ba_intraday_encoder import GRUIntradayEncoder, IntradayEncoder\nfrom ba_temporal_encoder import TemporalEncoder\nfrom ba_market_latent import MarketLatentModule\nfrom ba_score_head import ScoreHead\nfrom ba_pretrain_heads import MAPredictionHead, MaskedBarHead, NextDayHead\n\n\nclass SSPT30mModel(nn.Module):\n    def __init__(self, config: dict):\n        super().__init__()\n        cfg = config.get("model", config)\n        self.d_model = cfg["d_model"]\n        self.num_fields = cfg.get("num_fields", 11)\n        self.bars_per_day = cfg.get("bars_per_day", 8)\n\n        self.bar_encoder = BarEncoder(self.num_fields, self.d_model,\n                                       cfg.get("dropout", 0.1))\n\n        self.intraday_encoder_type = str(\n            cfg.get("intraday_encoder_type", "transformer")\n        ).lower()\n        if self.intraday_encoder_type == "transformer":\n            self.intraday_encoder = IntradayEncoder(\n                d_model=self.d_model,\n                num_layers=cfg["intraday_layers"],\n                num_heads=cfg["num_heads"],\n                ff_mult=cfg.get("ff_mult", 4),\n                dropout=cfg.get("dropout", 0.1),\n                attention_dropout=cfg.get("attention_dropout", 0.05),\n                activation=cfg.get("activation", "gelu"),\n                pre_norm=cfg.get("pre_norm", True),\n                bars_per_day=self.bars_per_day,\n            )\n        elif self.intraday_encoder_type == "gru":\n            self.intraday_encoder = GRUIntradayEncoder(\n                d_model=self.d_model,\n                num_layers=cfg["intraday_layers"],\n                dropout=cfg.get("dropout", 0.1),\n                bars_per_day=self.bars_per_day,\n                bidirectional=cfg.get("intraday_gru_bidirectional", True),\n                hidden_size=cfg.get("intraday_gru_hidden_size"),\n            )\n        else:\n            raise ValueError(\n                "model.intraday_encoder_type must be \'transformer\' or \'gru\'"\n            )\n\n        self.temporal_encoder = TemporalEncoder(\n            d_model=self.d_model,\n            num_layers=cfg["temporal_layers"],\n            num_heads=cfg["num_heads"],\n            ff_mult=cfg.get("ff_mult", 4),\n            dropout=cfg.get("dropout", 0.1),\n            attention_dropout=cfg.get("attention_dropout", 0.05),\n            activation=cfg.get("activation", "gelu"),\n            pre_norm=cfg.get("pre_norm", True),\n            use_sdpa=cfg.get("use_sdpa", True),\n        )\n\n        self.score_head = ScoreHead(self.d_model,\n                                     use_nonlinear=cfg.get("use_nonlinear_score", True),\n                                     dropout=cfg.get("dropout", 0.1))\n\n        self.use_cross_section = cfg.get("use_cross_section_module", False)\n        if self.use_cross_section:\n            self.market_latent = MarketLatentModule(\n                d_model=self.d_model,\n                num_latents=cfg.get("num_market_latents", 16),\n                num_layers=cfg.get("cross_section_layers", 2),\n                dropout=cfg.get("dropout", 0.1),\n                num_heads=cfg["num_heads"],\n                ff_mult=cfg.get("cross_section_ff_mult", cfg.get("ff_mult", 4)),\n                activation=cfg.get("activation", "gelu"),\n                pre_norm=cfg.get("pre_norm", True),\n            )\n\n        self.ma_head: Optional[MAPredictionHead] = None\n        self.masked_bar_head: Optional[MaskedBarHead] = None\n        self.next_day_head: Optional[NextDayHead] = None\n\n    def add_pretrain_heads(self, pretrain_config: dict):\n        if pretrain_config.get("use_ma_prediction", True):\n            num_windows = len(pretrain_config.get("ma_windows", [3, 5, 10, 20]))\n            self.ma_head = MAPredictionHead(self.d_model, num_windows)\n\n        if pretrain_config.get("use_masked_bar", True):\n            self.masked_bar_head = MaskedBarHead(\n                self.d_model, self.bars_per_day, self.num_fields,\n            )\n\n        if pretrain_config.get("use_next_day_prediction", True):\n            self.next_day_head = NextDayHead(self.d_model, self.num_fields, self.bars_per_day)\n\n    def forward_encoder(self, bars: torch.Tensor, day_mask: torch.Tensor = None,\n                         stock_mask: torch.Tensor = None) -> torch.Tensor:\n        # bars: [B, D, P, F]\n        # day_mask: [B, D, P]\n        B, D, P, F = bars.shape\n\n        bar_emb = self.bar_encoder(bars)  # [B, D, P, d_model]\n        day_repr = self.intraday_encoder(bar_emb, day_mask)  # [B, D, d_model]\n\n        if day_mask is not None:\n            day_valid = (day_mask.max(dim=-1)[0] > 0.5).float()  # [B, D]\n        else:\n            day_valid = None\n\n        temporal_out = self.temporal_encoder(day_repr, day_valid)  # [B, D, d_model]\n\n        if temporal_out.dim() == 3:\n            if day_valid is not None:\n                valid_times = day_valid.unsqueeze(-1)  # [B, D, 1]\n                masked_out = temporal_out * valid_times\n                stock_repr = masked_out.sum(dim=1) / valid_times.sum(dim=1).clamp(min=1)  # [B, d_model]\n            else:\n                stock_repr = temporal_out.mean(dim=1)  # [B, d_model]\n        else:\n            stock_repr = temporal_out\n\n        return stock_repr, temporal_out\n\n    def forward(self, bars: torch.Tensor, day_mask: torch.Tensor = None,\n                stock_mask: torch.Tensor = None, return_all: bool = False,\n                distributed_cross_section: bool = True,\n                gather_complete_date: bool = False):\n        stock_repr, temporal_out = self.forward_encoder(bars, day_mask)\n        if gather_complete_date:\n            stock_repr, stock_mask = self._gather_variable_cross_section(\n                stock_repr, stock_mask,\n            )\n        score, stock_repr = self.score_from_stock_repr(\n            stock_repr,\n            stock_mask=stock_mask,\n            distributed_cross_section=distributed_cross_section,\n            return_representation=True,\n        )\n\n        if return_all:\n            return {\n                "score": score,\n                "stock_repr": stock_repr,\n                "temporal_out": temporal_out,\n            }\n        return score\n\n    @staticmethod\n    def _gather_variable_cross_section(stock_repr: torch.Tensor,\n                                       stock_mask: torch.Tensor = None):\n        """Differentiably gather unequal local stock shards without repeats.\n\n        Every rank pads only for the collective, and the padded rows are\n        removed before Market Latent.  This replaces the legacy equal-length\n        repeat padding contract while preserving gradients to local encoders.\n        """\n        if not (dist.is_available() and dist.is_initialized()):\n            return stock_repr, stock_mask\n        world_size = dist.get_world_size()\n        local_n = torch.tensor([stock_repr.shape[0]], device=stock_repr.device, dtype=torch.long)\n        counts = [torch.zeros_like(local_n) for _ in range(world_size)]\n        dist.all_gather(counts, local_n)\n        lengths = [int(value.item()) for value in counts]\n        max_n = max(lengths)\n        if max_n == 0:\n            return stock_repr.new_empty((0, stock_repr.shape[-1])), stock_mask\n        padded = F.pad(stock_repr, (0, 0, 0, max_n - stock_repr.shape[0]))\n        gathered = differentiable_all_gather(padded.contiguous())\n        complete_repr = torch.cat(\n            [part[:length] for part, length in zip(gathered, lengths)], dim=0,\n        )\n        if stock_mask is None:\n            return complete_repr, None\n        padded_mask = F.pad(stock_mask, (0, max_n - stock_mask.shape[0]), value=0)\n        masks = [torch.empty_like(padded_mask) for _ in range(world_size)]\n        dist.all_gather(masks, padded_mask.contiguous())\n        complete_mask = torch.cat(\n            [part[:length] for part, length in zip(masks, lengths)], dim=0,\n        )\n        return complete_repr, complete_mask\n\n    def score_from_stock_repr(self, stock_repr: torch.Tensor,\n                              stock_mask: torch.Tensor = None,\n                              distributed_cross_section: bool = True,\n                              return_representation: bool = False):\n        """Apply the complete-date cross-section stage and score head.\n\n        Exposing this boundary permits chunked *local* encoding while keeping\n        Market Latent on the complete signal-date universe.\n        """\n        final_repr = stock_repr\n        if self.use_cross_section:\n            final_repr = self.market_latent(\n                final_repr, stock_mask,\n                distributed_gather=distributed_cross_section,\n            )\n        score = self.score_head(final_repr)\n        if return_representation:\n            return score, final_repr\n        return score\n\n    def predict_score(self, bars: torch.Tensor, day_mask: torch.Tensor = None) -> torch.Tensor:\n        return self.forward(bars, day_mask, return_all=False)\n\n    def forward_pretrain(self, bars: torch.Tensor, day_mask: torch.Tensor = None) -> Dict[str, torch.Tensor]:\n        stock_repr, temporal_out = self.forward_encoder(bars, day_mask)\n        # stock_repr: [B, d_model], temporal_out: [B, D, d_model]\n\n        outputs = {}\n\n        if self.ma_head is not None:\n            outputs["ma_pred"] = self.ma_head(stock_repr)\n\n        if self.masked_bar_head is not None:\n            outputs["bar_recon"] = self.masked_bar_head(temporal_out, day_mask)\n\n        if self.next_day_head is not None:\n            outputs["next_day_pred"] = self.next_day_head(stock_repr)\n\n        return outputs\n\n    @property\n    def device(self):\n        return next(self.parameters()).device\n', 'ba_date_level': '"""Shared complete-date forward path for training, validation and inference.\n\nThe cross-sectional module is mathematically defined on one whole signal-day.\nThis helper keeps local temporal encoding chunkable while postponing Market\nLatent and the score head until every stock of the date is present.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Optional\n\nimport torch\nimport torch.nn as nn\nimport torch.distributed as dist\nimport torch.nn.functional as F\n\n\ndef raw_model(model: nn.Module) -> nn.Module:\n    """Return the underlying module without importing DDP at module import time."""\n    return getattr(model, "module", model)\n\n\ndef predict_one_date(\n    model: nn.Module,\n    bars: torch.Tensor,\n    day_mask: Optional[torch.Tensor] = None,\n    *,\n    local_chunk_size: int = 0,\n    distributed_cross_section: bool = False,\n    distributed_shard: bool = False,\n) -> torch.Tensor:\n    """Score one and only one full stock cross-section.\n\n    ``bars`` is ``[n_stocks, lookback_days, bars_per_day, fields]``.  A\n    positive ``local_chunk_size`` splits only the stock-local bar/intraday/\n    temporal encoders.  The concatenated representations still enter one\n    Market Latent call, so chunking cannot change the market universe.\n\n    With ``distributed_shard=True``, each rank supplies a distinct stock shard.\n    The model differentiably gathers ragged local representations, removes\n    collective-only padding, then runs one complete Market Latent on every\n    rank.  No stock is repeated.\n    """\n    if bars.ndim != 4:\n        raise ValueError(f"expected bars [N,D,P,F], got {tuple(bars.shape)}")\n    n_stocks = int(bars.shape[0])\n    if n_stocks == 0:\n        return bars.new_empty((0,))\n    if local_chunk_size < 0:\n        raise ValueError("local_chunk_size must be non-negative")\n\n    base = raw_model(model)\n    is_wrapped = base is not model\n    use_chunks = 0 < local_chunk_size < n_stocks\n    if distributed_shard and not (dist.is_available() and dist.is_initialized()):\n        raise RuntimeError("distributed_shard=True requires an initialized process group")\n    if use_chunks and is_wrapped:\n        raise RuntimeError(\n            "date-level local chunks with DDP are not supported: use replicated "\n            "full-date DDP (local_chunk_size=0) or a single process."\n        )\n\n    if not use_chunks:\n        # Calling the wrapper is important for normal DDP reducer hooks.\n        return model(\n            bars,\n            day_mask,\n            distributed_cross_section=False if distributed_shard else distributed_cross_section,\n            gather_complete_date=distributed_shard,\n        )\n\n    representations = []\n    for offset in range(0, n_stocks, local_chunk_size):\n        upper = min(offset + local_chunk_size, n_stocks)\n        chunk_mask = None if day_mask is None else day_mask[offset:upper]\n        stock_repr, _ = base.forward_encoder(bars[offset:upper], chunk_mask)\n        representations.append(stock_repr)\n    complete_repr = torch.cat(representations, dim=0)\n    return base.score_from_stock_repr(\n        complete_repr,\n        distributed_cross_section=distributed_cross_section,\n    )\n\n\ndef gather_date_tensor(value: torch.Tensor) -> torch.Tensor:\n    """All-gather unequal first-dimension date shards without duplicate pads."""\n    if not (dist.is_available() and dist.is_initialized()):\n        return value\n    world_size = dist.get_world_size()\n    local_n = torch.tensor([value.shape[0]], device=value.device, dtype=torch.long)\n    counts = [torch.zeros_like(local_n) for _ in range(world_size)]\n    dist.all_gather(counts, local_n)\n    lengths = [int(item.item()) for item in counts]\n    max_n = max(lengths)\n    padded = F.pad(value, (0, max_n - value.shape[0]))\n    gathered = [torch.empty_like(padded) for _ in range(world_size)]\n    dist.all_gather(gathered, padded)\n    return torch.cat([item[:length] for item, length in zip(gathered, lengths)], dim=0)\n\n\ndef gather_date_objects(values: list) -> list:\n    """Gather metadata in the same rank-major order as ``gather_date_tensor``."""\n    if not (dist.is_available() and dist.is_initialized()):\n        return list(values)\n    gathered = [None for _ in range(dist.get_world_size())]\n    dist.all_gather_object(gathered, list(values))\n    return [item for rank_values in gathered for item in rank_values]\n', 'ba_cross_section_metrics': '"""Local daily-cross-section evaluation used for model selection.\n\nThese are transparent proxies for the public four-part evaluation.  In\nparticular, ``stress_proxy`` is intentionally not presented as the hidden\nofficial Stress formula; it rewards the worst observed monthly/20-day IC.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\n\ndef _safe_corr(left: pd.Series, right: pd.Series) -> float:\n    if len(left) < 3:\n        return float("nan")\n    x = left.to_numpy(dtype=float)\n    y = right.to_numpy(dtype=float)\n    if not (np.isfinite(x).all() and np.isfinite(y).all()):\n        return float("nan")\n    if np.std(x) <= 1e-12 or np.std(y) <= 1e-12:\n        return float("nan")\n    return float(np.corrcoef(x, y)[0, 1])\n\n\ndef evaluate_daily_cross_section(\n    frame: pd.DataFrame,\n    *,\n    score_column: str = "score",\n    label_column: str = "label",\n    groups: int = 10,\n    annualization_days: int = 252,\n) -> tuple[dict[str, float], pd.DataFrame]:\n    """Return IC/ICIR/long-short/stress proxies and one row per signal date."""\n    required = {"date", score_column, label_column}\n    missing = required.difference(frame.columns)\n    if missing:\n        raise KeyError(f"evaluation frame missing columns: {sorted(missing)}")\n    work = frame[["date", score_column, label_column]].copy()\n    work["date"] = pd.to_datetime(work["date"], errors="raise").dt.normalize()\n    work = work.replace([np.inf, -np.inf], np.nan).dropna()\n    rows = []\n    for date, day in work.groupby("date", sort=True):\n        if len(day) < max(10, groups):\n            continue\n        ic = _safe_corr(day[score_column], day[label_column])\n        rank_ic = _safe_corr(day[score_column].rank(method="average"), day[label_column].rank(method="average"))\n        ranks = day[score_column].rank(method="first")\n        bucket = pd.qcut(ranks, q=groups, labels=False, duplicates="drop")\n        grouped = day.assign(_bucket=bucket).groupby("_bucket", observed=True)[label_column].mean()\n        spread = float(grouped.iloc[-1] - grouped.iloc[0]) if len(grouped) >= 2 else float("nan")\n        rows.append({\n            "date": date,\n            "n_stocks": int(len(day)),\n            "daily_ic": ic,\n            "daily_rank_ic": rank_ic,\n            "long_short": spread,\n        })\n    daily = pd.DataFrame(rows)\n    if daily.empty:\n        return {\n            "ic_mean": float("nan"), "ic_std": float("nan"), "ic_ir": float("nan"),\n            "ic_positive_ratio": float("nan"), "rank_ic_mean": float("nan"),\n            "long_short_mean": float("nan"), "long_short_sharpe": float("nan"),\n            "stress_proxy": float("nan"), "worst_month_ic": float("nan"),\n            "worst_20d_ic": float("nan"),\n        }, daily\n    ic = daily["daily_ic"].dropna()\n    spread = daily["long_short"].dropna()\n    ic_mean = float(ic.mean()) if len(ic) else float("nan")\n    ic_std = float(ic.std(ddof=1)) if len(ic) > 1 else float("nan")\n    ic_ir = float(ic_mean / ic_std * np.sqrt(annualization_days)) if ic_std and ic_std > 1e-12 else float("nan")\n    spread_std = float(spread.std(ddof=1)) if len(spread) > 1 else float("nan")\n    ls_sharpe = float(spread.mean() / spread_std * np.sqrt(annualization_days)) if spread_std and spread_std > 1e-12 else float("nan")\n    month_ic = daily.set_index("date")["daily_ic"].resample("ME").mean().dropna()\n    rolling_ic = daily.set_index("date")["daily_ic"].rolling(20, min_periods=5).mean().dropna()\n    worst_month = float(month_ic.min()) if len(month_ic) else float("nan")\n    worst_20d = float(rolling_ic.min()) if len(rolling_ic) else float("nan")\n    stress = float(np.nanmean([worst_month, worst_20d]))\n    return {\n        "ic_mean": ic_mean,\n        "ic_std": ic_std,\n        "ic_ir": ic_ir,\n        "ic_positive_ratio": float((ic > 0).mean()) if len(ic) else float("nan"),\n        "rank_ic_mean": float(daily["daily_rank_ic"].mean()),\n        "long_short_mean": float(spread.mean()) if len(spread) else float("nan"),\n        "long_short_sharpe": ls_sharpe,\n        "stress_proxy": stress,\n        "worst_month_ic": worst_month,\n        "worst_20d_ic": worst_20d,\n    }, daily\n', 'checkpoint': '"""Split ASCII checkpoint helpers for submission_v6."""\nfrom __future__ import annotations\n\nimport base64\nimport gc\nimport io\nfrom pathlib import Path\nfrom typing import Any, Dict\nimport zlib\n\nimport torch\n\nTEXT_CHECKPOINT_HEADER = "BIGALPHA_TEXT_PT_V2:zlib+b64"\nLEGACY_TEXT_CHECKPOINT_HEADER = "BIGALPHA_TEXT_PT_V1:zlib+b85"\n\n\ndef _torch_load_compat(source, *, map_location: str):\n    try:\n        return torch.load(source, map_location=map_location, weights_only=False)\n    except TypeError:\n        return torch.load(source, map_location=map_location)\n\n\ndef _load_serialized_checkpoint(path: Path, *, map_location: str):\n    with path.open("rb") as handle:\n        header = handle.readline().decode("ascii", errors="ignore").strip()\n        payload = handle.read()\n    if header not in {TEXT_CHECKPOINT_HEADER, LEGACY_TEXT_CHECKPOINT_HEADER}:\n        return _torch_load_compat(path, map_location=map_location)\n    try:\n        compact = payload.replace(b"\\n", b"").replace(b"\\r", b"")\n        decoded = (\n            base64.b64decode(compact)\n            if header == TEXT_CHECKPOINT_HEADER\n            else base64.b85decode(compact)\n        )\n        raw = zlib.decompress(decoded)\n    except Exception as exc:\n        raise ValueError(f"Unable to decode text checkpoint {path}") from exc\n    return _torch_load_compat(io.BytesIO(raw), map_location=map_location)\n\n\ndef iter_weight_shards(checkpoint: Dict[str, Any], map_location: str = "cpu"):\n    """Yield one state-dict shard at a time without retaining decoded payloads."""\n    checkpoint_path = Path(checkpoint.get("_checkpoint_path", "model_meta.pt"))\n    for name in checkpoint.get("weight_shards", []):\n        shard_path = checkpoint_path.parent / str(name)\n        if not shard_path.is_file():\n            raise FileNotFoundError(f"Missing V6 model shard: {shard_path.name}")\n        shard = _load_serialized_checkpoint(shard_path, map_location=map_location)\n        part = shard.get("state_dict", shard.get("model_state_dict"))\n        if not isinstance(part, dict):\n            raise KeyError(f"Invalid state_dict in V6 shard: {shard_path.name}")\n        yield shard_path.name, part\n        del shard, part\n        gc.collect()\n\n\ndef load_checkpoint(\n    path: str | Path = "model_meta.pt",\n    map_location: str = "cpu",\n    load_shards: bool = False,\n) -> Dict[str, Any]:\n    """Load lightweight metadata by default; full assembly is opt-in.\n\n    The official worker can have a tight RAM limit.  Materialising all decoded\n    text shards before model construction caused a roughly 2 GB startup peak.\n    Inference streams shards directly into the model instead.\n    """\n    path = Path(path)\n    checkpoint = _load_serialized_checkpoint(path, map_location=map_location)\n    checkpoint["_checkpoint_path"] = str(path.resolve())\n    shard_names = checkpoint.get("weight_shards")\n    if shard_names and load_shards:\n        state_dict = {}\n        for shard_name, part in iter_weight_shards(checkpoint, map_location=map_location):\n            duplicate = set(state_dict).intersection(part)\n            if duplicate:\n                raise KeyError(f"Duplicate V6 state keys in {shard_name}: {sorted(duplicate)[:3]}")\n            state_dict.update(part)\n        checkpoint["state_dict"] = state_dict\n        checkpoint["model_state_dict"] = state_dict\n    elif "state_dict" not in checkpoint and "model_state_dict" in checkpoint:\n        checkpoint["state_dict"] = checkpoint["model_state_dict"]\n    elif "model_state_dict" not in checkpoint and "state_dict" in checkpoint:\n        checkpoint["model_state_dict"] = checkpoint["state_dict"]\n    required = ["model_config", "feature_manifest", "preprocessor"]\n    if not shard_names:\n        required.append("state_dict")\n    missing = [key for key in required if key not in checkpoint]\n    if missing:\n        raise KeyError(f"Checkpoint missing required keys: {missing}")\n    return checkpoint\n', 'data_pipeline': '"""Canonical data and window construction shared by training and inference."""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Sequence\n\nimport numpy as np\nimport pandas as pd\n\nfrom ba_preprocessing import FieldScaler\n\n\nPRICE_SCALE = 100.0\nOHLC_COLS = ["open", "high", "low", "close"]\nSCALE_FIELDS = [\n    "open", "high", "low", "close", "amount",\n    "ask_price1", "ask_price2", "ask_price3",\n    "bid_price1", "bid_price2", "bid_price3",\n]\nBOOK_PREFIXES = (\n    "ask_price", "bid_price",\n    "ask_volume", "bid_volume",\n    "ask_num_orders", "bid_num_orders",\n)\n\n\ndef to_canonical(df: pd.DataFrame, *, source: str, feature_cols: list[str]) -> pd.DataFrame:\n    if source not in {"local", "cloud"}:\n        raise ValueError(f"Unknown source: {source}")\n\n    out = df.copy()\n    out["date"] = pd.to_datetime(out["date"], errors="raise")\n\n    if source == "local":\n        if "instrument_id" not in out.columns:\n            raise KeyError("local data missing instrument_id")\n        for col in OHLC_COLS:\n            if col in out.columns:\n                out.loc[out[col] == -1, col] = np.nan\n        for col in SCALE_FIELDS:\n            if col in out.columns:\n                out[col] = out[col].astype("float64") / PRICE_SCALE\n        out["key"] = out["instrument_id"].astype(str)\n    else:\n        if "instrument" not in out.columns:\n            raise KeyError("cloud data missing instrument")\n        drop_cols = [\n            col for col in out.columns\n            if any(col.startswith(prefix) and col[-1:] in {"4", "5"} for prefix in BOOK_PREFIXES)\n        ]\n        out = out.drop(columns=drop_cols, errors="ignore")\n        out["key"] = out["instrument"].astype(str)\n\n    required = {"date", "key", *feature_cols}\n    missing = required.difference(out.columns)\n    if missing:\n        raise KeyError(f"Canonical data missing fields: {sorted(missing)}")\n\n    out = out[["date", "key", *feature_cols]]\n    return out.sort_values(["key", "date"]).reset_index(drop=True)\n\n\ndef make_scaler(preprocessor: dict) -> FieldScaler:\n    scaler = FieldScaler()\n    scaler.load_dict(preprocessor)\n    return scaler\n\n\ndef buffer_start_date(start_date, lookback_days: int, safety_margin: int = 20) -> pd.Timestamp:\n    if lookback_days > 240:\n        raise ValueError(f"lookback_days exceeds 240: {lookback_days}")\n    buffer_days = int(math.ceil(lookback_days * 1.8) + safety_margin)\n    return pd.Timestamp(start_date).normalize() - pd.Timedelta(days=buffer_days)\n\n\ndef prepare_day_window_source(\n    canonical: pd.DataFrame,\n    *,\n    feature_cols: Sequence[str],\n    scaler: FieldScaler,\n) -> dict[str, tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:\n    """Transform a query block once and index each instrument by trading day.\n\n    The returned arrays contain no pandas objects.  This is deliberately a\n    block-local cache: callers release it after a small group of signal dates,\n    avoiding both repeated scaler work and a full-interval window list.\n    """\n    fields = list(feature_cols)\n    histories: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]] = {}\n    for key, inst_df in canonical.groupby("key", sort=False):\n        ordered = inst_df.sort_values("date")\n        day_values = ordered["date"].dt.normalize().to_numpy(dtype="datetime64[ns]")\n        unique_days, starts = np.unique(day_values, return_index=True)\n        ends = np.append(starts[1:], len(ordered)).astype(np.int64, copy=False)\n        values = scaler.transform(ordered[fields].to_numpy(dtype=np.float32, copy=True), fields)\n        histories[str(key)] = (unique_days, starts.astype(np.int64, copy=False), ends, values)\n    return histories\n\n\ndef build_day_windows(\n    histories: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]],\n    *,\n    signal_day,\n    instruments: Sequence[str],\n    feature_cols: Sequence[str],\n    lookback_days: int,\n    bars_per_day: int,\n) -> tuple[np.ndarray, np.ndarray, list[str]]:\n    """Build one complete signal-date cross-section without retaining other windows.\n\n    The MarketLatent module is a cross-sectional operation, so splitting a day\n    into arbitrary model batches changes its output.  This function therefore\n    returns every available constituent for *one* day together.  Its arrays are\n    discarded immediately after that day is scored; peak memory is independent\n    of the length of the evaluation interval.\n\n    A constituent with bars on ``signal_day`` is kept even when it has less\n    than ``min_history_days`` of prior data.  Left padding is the same padding\n    used for ordinary short histories and avoids needlessly violating the\n    platform\'s daily coverage rule for recent listings.\n    """\n    if not 1 <= lookback_days <= 240:\n        raise ValueError(f"lookback_days must be in [1, 240], got {lookback_days}")\n\n    signal_day = pd.Timestamp(signal_day).normalize()\n    fields = list(feature_cols)\n    signal_value = signal_day.to_datetime64()\n\n    # Preallocate the maximum possible cross-section, then slice it.  Unlike a\n    # Python list of tensors this has small, predictable overhead.\n    max_n = len(instruments)\n    bars_out = np.zeros((max_n, lookback_days, bars_per_day, len(fields)), dtype=np.float32)\n    masks_out = np.zeros((max_n, lookback_days, bars_per_day), dtype=np.float32)\n    kept: list[str] = []\n\n    for instrument in instruments:\n        key = str(instrument)\n        history_data = histories.get(key)\n        if history_data is None:\n            continue\n        dates, starts, ends, values = history_data\n        position = int(np.searchsorted(dates, signal_value))\n        if position >= len(dates) or dates[position] != signal_value:\n            continue\n        first = max(0, position + 1 - lookback_days)\n        history_count = position + 1 - first\n        offset = lookback_days - history_count\n        row = len(kept)\n        for day_offset, day_index in enumerate(range(first, position + 1)):\n            day_rows = values[starts[day_index]:ends[day_index]]\n            count = min(len(day_rows), bars_per_day)\n            if count:\n                bars_out[row, offset + day_offset, :count] = day_rows[:count]\n                masks_out[row, offset + day_offset, :count] = 1.0\n        kept.append(key)\n\n    return bars_out[:len(kept)], masks_out[:len(kept)], kept\n', 'validation': '"""Submission output validation."""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\n\ndef validate_scores(\n    df: pd.DataFrame,\n    *,\n    start_date=None,\n    end_date=None,\n    expected_dates=None,\n    expected_instruments_by_date=None,\n    min_coverage: float = 0.60,\n) -> dict:\n    """Validate both structural rules and official daily coverage rules.\n\n    ``expected_dates`` must be the official trading calendar, not calendar days.\n    ``expected_instruments_by_date`` must be the historical CSI-1000 membership\n    for the same dates.  Supplying either makes omissions fail locally instead\n    of first being discovered by the platform evaluator.\n    """\n    expected = ["date", "instrument", "score"]\n    if list(df.columns) != expected:\n        raise ValueError(f"Output columns must be exactly {expected}, got {list(df.columns)}")\n    if df.empty:\n        raise ValueError("Output score DataFrame is empty")\n\n    out = df.copy()\n    out["date"] = pd.to_datetime(out["date"]).dt.normalize()\n    if out["date"].isna().any():\n        raise ValueError("date contains NaN")\n    if out["instrument"].isna().any():\n        raise ValueError("instrument contains NaN")\n\n    scores = pd.to_numeric(out["score"], errors="coerce")\n    if scores.isna().any():\n        raise ValueError("score contains NaN or non-numeric value")\n    if not np.isfinite(scores.to_numpy(np.float64)).all():\n        raise ValueError("score contains Inf or -Inf")\n    if out.duplicated(["date", "instrument"]).any():\n        raise ValueError("Duplicate date/instrument rows found")\n\n    daily_count = out.groupby("date")["instrument"].nunique()\n    daily_std = out.groupby("date")["score"].std()\n    if (daily_std.fillna(0.0) <= 1e-12).any():\n        bad = daily_std[daily_std.fillna(0.0) <= 1e-12].index.tolist()\n        raise ValueError(f"Constant score cross-section: {bad[:10]}")\n\n    if start_date is not None and out["date"].min() < pd.Timestamp(start_date).normalize():\n        raise ValueError("Output contains date before start_date")\n    if end_date is not None and out["date"].max() > pd.Timestamp(end_date).normalize():\n        raise ValueError("Output contains date after end_date")\n\n    expected_day_set = None\n    if expected_dates is not None:\n        expected_day_set = {pd.Timestamp(day).normalize() for day in expected_dates}\n        actual_day_set = set(out["date"].unique())\n        missing_days = sorted(expected_day_set.difference(actual_day_set))\n        if missing_days:\n            raise ValueError(f"Output misses official trading days: {missing_days[:10]}")\n        unexpected_days = sorted(actual_day_set.difference(expected_day_set))\n        if unexpected_days:\n            raise ValueError(f"Output contains dates outside official trading calendar: {unexpected_days[:10]}")\n\n    coverage_by_day = None\n    if expected_instruments_by_date is not None:\n        if not 0.0 < float(min_coverage) <= 1.0:\n            raise ValueError("min_coverage must be in (0, 1]")\n        normalized_universe = {\n            pd.Timestamp(day).normalize(): {str(instrument) for instrument in instruments}\n            for day, instruments in expected_instruments_by_date.items()\n        }\n        if expected_day_set is not None and set(normalized_universe) != expected_day_set:\n            raise ValueError("expected_dates and expected_instruments_by_date disagree")\n\n        coverage_by_day = {}\n        for day, expected_instruments in normalized_universe.items():\n            if not expected_instruments:\n                raise ValueError(f"Official constituent universe is empty on {day.date()}")\n            predicted = set(out.loc[out["date"] == day, "instrument"].astype(str))\n            outside = predicted.difference(expected_instruments)\n            if outside:\n                raise ValueError(\n                    f"Output includes non-constituents on {day.date()}: {sorted(outside)[:10]}"\n                )\n            coverage = len(predicted.intersection(expected_instruments)) / len(expected_instruments)\n            coverage_by_day[day] = coverage\n            if coverage < float(min_coverage):\n                raise ValueError(\n                    f"Daily constituent coverage below {min_coverage:.0%} on {day.date()}: "\n                    f"{len(predicted.intersection(expected_instruments))}/{len(expected_instruments)} "\n                    f"({coverage:.1%})"\n                )\n\n    report = {\n        "rows": len(out),\n        "dates": out["date"].nunique(),\n        "min_stocks_per_day": int(daily_count.min()),\n        "median_stocks_per_day": float(daily_count.median()),\n        "max_stocks_per_day": int(daily_count.max()),\n        "median_daily_score_std": float(daily_std.median()),\n    }\n    if coverage_by_day is not None:\n        report["min_constituent_coverage"] = float(min(coverage_by_day.values()))\n        report["median_constituent_coverage"] = float(np.median(list(coverage_by_day.values())))\n    return report\n', 'model': '"""V6 5-minute submission model entry point."""\nimport gc\n\nfrom ba_sspt30m import SSPT30mModel\n\nMODEL_NAME = "SSPT5m_GRU_Temporal9_MarketLatent16_V6"\n\n\ndef build_model(model_config: dict) -> SSPT30mModel:\n    return SSPT30mModel({"model": dict(model_config)})\n\n\ndef create_model_from_checkpoint(checkpoint: dict, *, load_weights: bool = False) -> SSPT30mModel:\n    model_config = dict(checkpoint["model_config"])\n    fields = checkpoint.get("feature_manifest", {}).get("frequencies", {}).get("bar5m")\n    if fields is not None:\n        model_config["num_fields"] = len(fields)\n    if checkpoint.get("lookback_days") and checkpoint.get("seq_len"):\n        model_config["bars_per_day"] = int(checkpoint["seq_len"]) // int(checkpoint["lookback_days"])\n    model = build_model(model_config)\n    if load_weights:\n        if "state_dict" in checkpoint:\n            model.load_state_dict(checkpoint["state_dict"], strict=True)\n        elif checkpoint.get("weight_shards"):\n            from checkpoint import iter_weight_shards\n            expected = set(model.state_dict())\n            loaded = set()\n            for shard_name, part in iter_weight_shards(checkpoint, map_location="cpu"):\n                duplicate = loaded.intersection(part)\n                if duplicate:\n                    raise KeyError(f"Duplicate keys in {shard_name}: {sorted(duplicate)[:3]}")\n                incompatible = model.load_state_dict(part, strict=False)\n                if incompatible.unexpected_keys:\n                    raise KeyError(\n                        f"Unexpected keys in {shard_name}: {incompatible.unexpected_keys[:3]}"\n                    )\n                loaded.update(part)\n                del part\n                gc.collect()\n            missing = expected.difference(loaded)\n            unexpected = loaded.difference(expected)\n            if missing or unexpected:\n                raise KeyError(\n                    f"Incomplete V6 sharded checkpoint: missing={sorted(missing)[:3]}, "\n                    f"unexpected={sorted(unexpected)[:3]}"\n                )\n        else:\n            raise KeyError("checkpoint contains neither state_dict nor weight_shards")\n    return model\n\n\ndef count_trainable_parameters(model) -> int:\n    return sum(p.numel() for p in model.parameters() if p.requires_grad)\n', 'inference': '"""Cloud inference entry point used by ``predict.ipynb``.\n\nThe platform passes datasource *table names*, including hidden-table names, in\n``datasources``.  A table name is deliberately queried through ``dai.query``;\nit is never interpreted as a local file path.  Local files/DataFrames remain\nsupported only for offline smoke tests.\n"""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Mapping\nfrom pathlib import Path\nfrom typing import Any, Iterable, Sequence\nimport re\nimport zlib\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom checkpoint import load_checkpoint\nfrom data_pipeline import (\n    buffer_start_date,\n    build_day_windows,\n    make_scaler,\n    prepare_day_window_source,\n    to_canonical,\n)\nfrom model import build_model\nfrom validation import validate_scores\n\n\nUNIVERSE_TABLE = "bigalpha_2026_instruments"\n# Keep DAI query granularity aligned with the already accepted v3 package.\nDEFAULT_QUERY_CHUNK_DAYS = 1\nDEFAULT_INSTRUMENT_QUERY_CHUNK_SIZE = 128\nBAR5M_DATASOURCE_KEYS = ("bar5m", "bar_5m", "e2e_bar5m", "stock_bar5m")\n_TABLE_IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*(?:\\.[A-Za-z_][A-Za-z0-9_]*)?$")\n_INSTRUMENT_IDENTIFIER = re.compile(r"^[A-Za-z0-9_.-]+$")\n\n\ndef _select_device() -> torch.device:\n    return torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\ndef _date_text(value) -> str:\n    return pd.Timestamp(value).strftime("%Y-%m-%d")\n\n\ndef _next_date_text(value) -> str:\n    return (pd.Timestamp(value).normalize() + pd.Timedelta(days=1)).strftime("%Y-%m-%d")\n\n\ndef _bar_filter_end_text(value) -> str:\n    """Return the exclusive day boundary required by intraday DAI tables.\n\n    ``bigalpha_2026_stock_bar5m.date`` contains timestamps such as\n    ``2023-12-04 10:00:00``.  Passing ``2023-12-04`` as a DAI filter upper\n    bound denotes midnight and can therefore omit every current-day bar.  The\n    next midnight includes the full signal day while adding no next-day\n    intraday observations.\n    """\n    return _next_date_text(value)\n\n\ndef _validate_table_name(table: str) -> str:\n    """Accept platform table identifiers but reject SQL fragments."""\n    name = str(table).strip()\n    if not _TABLE_IDENTIFIER.fullmatch(name):\n        raise ValueError(f"Unsafe datasource table identifier: {table!r}")\n    return name\n\n\ndef _instrument_predicate(instruments: Sequence[str] | None) -> str:\n    if instruments is None:\n        return ""\n    values = []\n    for value in instruments:\n        instrument = str(value).strip()\n        if not _INSTRUMENT_IDENTIFIER.fullmatch(instrument):\n            raise ValueError(f"Unsafe instrument identifier: {value!r}")\n        values.append(f"\'{instrument}\'")\n    if not values:\n        return " AND 1 = 0"\n    return f" AND instrument IN ({\', \'.join(values)})"\n\n\ndef _pick_bar5m_source(datasources: Mapping[str, Any]) -> Any:\n    """Select the platform\'s 5-minute source without assuming one key spelling."""\n    for key in BAR5M_DATASOURCE_KEYS:\n        if key in datasources and datasources[key] is not None:\n            return datasources[key]\n    if len(datasources) == 1:\n        return next(iter(datasources.values()))\n    expected = ", ".join(repr(key) for key in BAR5M_DATASOURCE_KEYS)\n    raise KeyError(f"datasources must provide one of: {expected}")\n\n\ndef _dai_query(sql: str):\n    """Use the same SQL WHERE style as the accepted v3 package."""\n    import dai\n\n    try:\n        return dai.query(sql, compression=True)\n    except TypeError:\n        return dai.query(sql)\n\n\ndef _available_columns(table: str, *, start_date, end_date) -> set[str] | None:\n    """Probe schema once so an optional raw field cannot abort inference."""\n    try:\n        result = _dai_query(\n            f"SELECT * FROM {_validate_table_name(table)} "\n            f"WHERE date >= \'{_date_text(start_date)}\' "\n            f"AND date < \'{_bar_filter_end_text(end_date)}\' "\n            "LIMIT 1"\n        ).df()\n    except Exception:\n        return None\n    frame = result if isinstance(result, pd.DataFrame) else pd.DataFrame(result)\n    return set(frame.columns)\n\n\ndef _query_dataframe(table: str, *, start_date, end_date, columns: Sequence[str],\n                     instruments: Sequence[str] | None = None,\n                     bind_instruments: bool = True) -> pd.DataFrame:\n    """Read official cloud data by table name via the BigQuant DAI API."""\n    table = _validate_table_name(table)\n    selected = ", ".join(columns)\n    instrument_clause = _instrument_predicate(instruments) if bind_instruments else ""\n    result = _dai_query(\n        f"SELECT {selected} FROM {table} "\n        f"WHERE date >= \'{_date_text(start_date)}\' "\n        f"AND date < \'{_bar_filter_end_text(end_date)}\' "\n        f"{instrument_clause} "\n        "ORDER BY instrument, date"\n    ).df()\n    frame = result if isinstance(result, pd.DataFrame) else pd.DataFrame(result)\n    if instruments is not None and not bind_instruments and "instrument" in frame.columns:\n        allowed = {str(value) for value in instruments}\n        frame = frame.loc[frame["instrument"].astype(str).isin(allowed)].copy()\n    return frame\n\n\ndef _query_universe(*, start_date, end_date) -> pd.DataFrame:\n    """Get the official historical CSI-1000 constituent set and trading days."""\n    result = _dai_query(\n        f"SELECT date, instrument FROM {UNIVERSE_TABLE} "\n        f"WHERE date >= \'{_date_text(start_date)}\' "\n        f"AND date < \'{_next_date_text(end_date)}\' "\n        "ORDER BY date, instrument"\n    ).df()\n    out = result if isinstance(result, pd.DataFrame) else pd.DataFrame(result)\n    if out.empty:\n        raise ValueError("Official constituent query returned no rows for the evaluation interval")\n    if not {"date", "instrument"}.issubset(out.columns):\n        raise KeyError("Official constituent query must return date and instrument")\n    out = out[["date", "instrument"]].copy()\n    out["date"] = pd.to_datetime(out["date"], errors="raise").dt.normalize()\n    out["instrument"] = out["instrument"].astype(str)\n    return out.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"])\n\n\ndef _read_offline_datasource(source: Any, *, columns: Sequence[str]) -> pd.DataFrame:\n    """Read explicit local test sources only; cloud table strings use DAI instead."""\n    if isinstance(source, pd.DataFrame):\n        return source.copy()\n    if isinstance(source, (str, Path)):\n        path = Path(source)\n        if path.suffix == ".feather":\n            import pyarrow.feather as feather\n            return feather.read_feather(path, columns=list(columns))\n        if path.suffix == ".parquet":\n            return pd.read_parquet(path, columns=list(columns))\n        if path.suffix == ".csv":\n            return pd.read_csv(path, usecols=list(columns))\n        raise TypeError("Only explicit .feather/.parquet/.csv paths are offline datasources")\n    if hasattr(source, "read"):\n        try:\n            value = source.read(columns=list(columns))\n        except TypeError:\n            value = source.read()\n        return value if isinstance(value, pd.DataFrame) else pd.DataFrame(value)\n    if hasattr(source, "to_df"):\n        value = source.to_df()\n        return value if isinstance(value, pd.DataFrame) else pd.DataFrame(value)\n    raise TypeError(f"Unsupported datasource type: {type(source)!r}")\n\n\ndef _chunked(values: Sequence[pd.Timestamp], size: int) -> Iterable[Sequence[pd.Timestamp]]:\n    for offset in range(0, len(values), size):\n        yield values[offset:offset + size]\n\n\ndef _filter_raw_frame(raw: pd.DataFrame, *, start_date, end_date,\n                      instruments: Sequence[str]) -> pd.DataFrame:\n    if "date" not in raw.columns or "instrument" not in raw.columns:\n        raise KeyError("bar data must contain date and instrument")\n    dates = pd.to_datetime(raw["date"], errors="raise")\n    allowed = {str(value) for value in instruments}\n    return raw.loc[\n        (dates >= pd.Timestamp(start_date))\n        & (dates < pd.Timestamp(end_date).normalize() + pd.Timedelta(days=1))\n        & raw["instrument"].astype(str).isin(allowed)\n    ].copy()\n\n\ndef _load_model(checkpoint: dict, *, bars_per_day: int) -> tuple[torch.nn.Module, torch.device]:\n    model_config = dict(checkpoint["model_config"])\n    model_config["num_fields"] = len(checkpoint["feature_manifest"]["frequencies"]["bar5m"])\n    model_config["bars_per_day"] = bars_per_day\n    try:\n        device = _select_device()\n        model = build_model(model_config).to(device)\n    except RuntimeError:\n        # Local test images can contain a CUDA runtime that predates the GPU.\n        device = torch.device("cpu")\n        model = build_model(model_config).to(device)\n    model.load_state_dict(checkpoint["state_dict"], strict=True)\n    model.eval()\n    return model, device\n\n\ndef _predict_day(\n    model: torch.nn.Module,\n    bars: np.ndarray,\n    masks: np.ndarray,\n    *,\n    device: torch.device,\n    batch_size: int,\n) -> np.ndarray:\n    """Predict one full date without turning its encoder into one huge batch.\n\n    The intraday and temporal encoders operate independently for each stock,\n    whereas MarketLatent must receive the complete date-level cross-section.\n    Encoding stock chunks first and applying ``score_from_stock_repr`` once\n    therefore gives exactly the intended MarketLatent contract, but avoids the\n    [stocks, heads, days, days] attention allocation that caused cloud kernels\n    to be killed on CPU-only workers.\n    """\n    if len(bars) == 0:\n        return np.empty(0, dtype=np.float32)\n\n    if getattr(model, "use_cross_section", False):\n        representations = []\n        for offset in range(0, len(bars), batch_size):\n            encoded, _ = model.forward_encoder(\n                torch.from_numpy(bars[offset:offset + batch_size]).to(device),\n                torch.from_numpy(masks[offset:offset + batch_size]).to(device),\n            )\n            representations.append(encoded)\n        stock_repr = torch.cat(representations, dim=0)\n        score = model.score_from_stock_repr(\n            stock_repr,\n            distributed_cross_section=False,\n        )\n        return score.detach().cpu().numpy()\n\n    score_batches = []\n    for offset in range(0, len(bars), batch_size):\n        score_batches.append(model.predict_score(\n            torch.from_numpy(bars[offset:offset + batch_size]).to(device),\n            torch.from_numpy(masks[offset:offset + batch_size]).to(device),\n        ).detach().cpu().numpy())\n    return np.concatenate(score_batches)\n\n\ndef _resolve_checkpoint_path(checkpoint_path) -> Path:\n    """Resolve an uploaded weight without depending on a fake module path."""\n    requested = Path(checkpoint_path)\n    candidates = [requested]\n    if not requested.is_absolute():\n        candidates.append(Path.cwd() / requested)\n    # Embedded notebook modules have synthetic ``__file__`` values.  Keeping\n    # this final candidate preserves normal local-module execution as well.\n    candidates.append(Path(__file__).resolve().parent / requested.name)\n    for candidate in candidates:\n        if candidate.is_file():\n            return candidate\n    raise FileNotFoundError(\n        f"Could not locate uploaded checkpoint {requested.name!r}; "\n        f"checked: {[str(path) for path in candidates]}"\n    )\n\n\ndef _fallback_scores(signal_day, instruments: Sequence[str]) -> np.ndarray:\n    """Deterministic non-constant fallback for days with unavailable bar rows."""\n    if not instruments:\n        return np.empty(0, dtype=np.float64)\n    day_key = _date_text(signal_day)\n    values = np.asarray(\n        [zlib.crc32(f"{day_key}:{str(instrument)}".encode("utf-8")) for instrument in instruments],\n        dtype=np.float64,\n    )\n    values = (values - values.mean()) / (values.std() + 1e-12)\n    return values * 1e-6\n\n\ndef run_cloud_inference(\n    *,\n    datasources,\n    start_date,\n    end_date,\n    checkpoint_path="model.pt",\n    batch_size=2048,\n    query_chunk_days: int = DEFAULT_QUERY_CHUNK_DAYS,\n    instrument_query_chunk_size: int = DEFAULT_INSTRUMENT_QUERY_CHUNK_SIZE,\n) -> pd.DataFrame:\n    """Score a cloud interval with bounded memory and date-complete output.\n\n    Data are queried in small blocks of *trading days*.  Within a block windows\n    are built and released one signal day at a time.  Most importantly, all\n    available constituents of each day are inferred in one call, preserving the\n    MarketLatent cross-section rather than accidentally treating arbitrary\n    inference mini-batches as separate markets.\n    """\n    if batch_size <= 0:\n        raise ValueError("batch_size must be positive")\n    if query_chunk_days <= 0:\n        raise ValueError("query_chunk_days must be positive")\n    if instrument_query_chunk_size <= 0:\n        raise ValueError("instrument_query_chunk_size must be positive")\n    if not isinstance(datasources, Mapping):\n        raise TypeError("datasources must be a mapping containing \'bar5m\'")\n\n    source = _pick_bar5m_source(datasources)\n\n    ckpt_path = _resolve_checkpoint_path(checkpoint_path)\n    checkpoint = load_checkpoint(ckpt_path, map_location="cpu")\n    feature_cols = list(checkpoint["feature_manifest"]["frequencies"]["bar5m"])\n    lookback_days = int(checkpoint["lookback_days"])\n    bars_per_day = int(checkpoint["seq_len"] // lookback_days)\n    if not 1 <= lookback_days <= 240:\n        raise ValueError(f"checkpoint lookback_days is out of bounds: {lookback_days}")\n\n    # A platform datasource is always a table-name string.  Query it via DAI\n    # and use the official constituent table as both the trading calendar and\n    # the denominator of the 60% daily-coverage rule.\n    # A cloud table can be schema-qualified (and consequently contain a dot),\n    # so only the three explicit local-file extensions opt into offline mode.\n    # All other datasource strings are opaque DAI table identifiers supplied by\n    # the platform, including hidden validation-table names.\n    source_suffix = Path(str(source)).suffix.lower() if isinstance(source, (str, Path)) else ""\n    is_offline_file = source_suffix in {".feather", ".parquet", ".csv"}\n    cloud_table = source if isinstance(source, str) and not is_offline_file else None\n    if cloud_table is not None:\n        universe = _query_universe(start_date=start_date, end_date=end_date)\n        available_columns = _available_columns(\n            cloud_table, start_date=start_date, end_date=end_date,\n        )\n        query_feature_cols = (\n            feature_cols if available_columns is None\n            else [field for field in feature_cols if field in available_columns]\n        )\n        if not query_feature_cols:\n            raise KeyError("None of the checkpoint\'s bar5m raw fields are available in the cloud table")\n        offline_raw = None\n    else:\n        needed = ["date", "instrument", *feature_cols]\n        offline_raw = _read_offline_datasource(source, columns=needed)\n        query_feature_cols = feature_cols\n        # Offline smoke tests do not have DAI.  The source\'s observed dates and\n        # instruments are the only available expected universe in that mode.\n        date_values = pd.to_datetime(offline_raw["date"], errors="raise").dt.normalize()\n        mask = (date_values >= pd.Timestamp(start_date).normalize()) & (\n            date_values <= pd.Timestamp(end_date).normalize())\n        universe = offline_raw.loc[mask, ["date", "instrument"]].copy()\n        universe["date"] = pd.to_datetime(universe["date"]).dt.normalize()\n        universe["instrument"] = universe["instrument"].astype(str)\n        universe = universe.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"])\n        if universe.empty:\n            raise ValueError("offline datasource contains no rows in the requested interval")\n\n    universe_by_day = {\n        day: group["instrument"].astype(str).tolist()\n        for day, group in universe.groupby("date", sort=True)\n    }\n    evaluation_days = sorted(universe_by_day)\n    scaler = make_scaler(checkpoint["preprocessor"])\n    model, device = _load_model(checkpoint, bars_per_day=bars_per_day)\n    rows: list[dict[str, Any]] = []\n\n    with torch.inference_mode():\n        for day_block in _chunked(evaluation_days, query_chunk_days):\n            block_start, block_end = day_block[0], day_block[-1]\n            block_instruments = sorted({\n                instrument for day in day_block for instrument in universe_by_day[day]\n            })\n            history_start = buffer_start_date(block_start, lookback_days)\n            if cloud_table is not None:\n                histories = {}\n                did_unfiltered_retry = False\n                for instrument_block in _chunked(block_instruments, instrument_query_chunk_size):\n                    raw = _query_dataframe(\n                        cloud_table,\n                        start_date=history_start,\n                        end_date=block_end,\n                        columns=["date", "instrument", *query_feature_cols],\n                        instruments=instrument_block,\n                    )\n                    if raw.empty:\n                        continue\n                    # A schema-compatible table can omit an optional raw\n                    # field.  Missing values are handled by the training-set\n                    # scaler exactly as in the normal preprocessing contract.\n                    for field in feature_cols:\n                        if field not in raw.columns:\n                            raw[field] = np.nan\n                    canonical = to_canonical(raw, source="cloud", feature_cols=feature_cols)\n                    histories.update(prepare_day_window_source(\n                        canonical, feature_cols=feature_cols, scaler=scaler,\n                    ))\n                    del canonical, raw\n                if not histories:\n                    raw = _query_dataframe(\n                        cloud_table,\n                        start_date=history_start,\n                        end_date=block_end,\n                        columns=["date", "instrument", *query_feature_cols],\n                        instruments=block_instruments,\n                        bind_instruments=False,\n                    )\n                    did_unfiltered_retry = True\n                    if not raw.empty:\n                        for field in feature_cols:\n                            if field not in raw.columns:\n                                raw[field] = np.nan\n                        canonical = to_canonical(raw, source="cloud", feature_cols=feature_cols)\n                        histories.update(prepare_day_window_source(\n                            canonical, feature_cols=feature_cols, scaler=scaler,\n                        ))\n                        del canonical, raw\n            else:\n                raw = _filter_raw_frame(\n                    offline_raw,\n                    start_date=history_start,\n                    end_date=block_end,\n                    instruments=block_instruments,\n                )\n                if raw.empty:\n                    raise ValueError(f"bar query returned no rows for block beginning {block_start.date()}")\n                canonical = to_canonical(raw, source="cloud", feature_cols=feature_cols)\n                histories = prepare_day_window_source(\n                    canonical, feature_cols=feature_cols, scaler=scaler,\n                )\n                del canonical, raw\n            if not histories and cloud_table is None:\n                raise ValueError(f"bar query returned no rows for block beginning {block_start.date()}")\n            for signal_day in day_block:\n                expected_instruments = universe_by_day[signal_day]\n                bars, masks, instruments = build_day_windows(\n                    histories,\n                    signal_day=signal_day,\n                    instruments=expected_instruments,\n                    feature_cols=feature_cols,\n                    lookback_days=lookback_days,\n                    bars_per_day=bars_per_day,\n                )\n                if not instruments and cloud_table is not None and not did_unfiltered_retry:\n                    raw = _query_dataframe(\n                        cloud_table,\n                        start_date=history_start,\n                        end_date=block_end,\n                        columns=["date", "instrument", *query_feature_cols],\n                        instruments=block_instruments,\n                        bind_instruments=False,\n                    )\n                    did_unfiltered_retry = True\n                    if not raw.empty:\n                        histories.clear()\n                        for field in feature_cols:\n                            if field not in raw.columns:\n                                raw[field] = np.nan\n                        canonical = to_canonical(raw, source="cloud", feature_cols=feature_cols)\n                        histories.update(prepare_day_window_source(\n                            canonical, feature_cols=feature_cols, scaler=scaler,\n                        ))\n                        del canonical, raw\n                        bars, masks, instruments = build_day_windows(\n                            histories,\n                            signal_day=signal_day,\n                            instruments=expected_instruments,\n                            feature_cols=feature_cols,\n                            lookback_days=lookback_days,\n                            bars_per_day=bars_per_day,\n                        )\n                if not instruments:\n                    day_text = signal_day.strftime("%Y-%m-%d")\n                    scores = np.empty(0, dtype=np.float32)\n                    rows.extend(\n                        {"date": day_text, "instrument": instrument, "score": float(score)}\n                        for instrument, score in zip(\n                            expected_instruments,\n                            _fallback_scores(signal_day, expected_instruments),\n                        )\n                    )\n                    continue\n\n                scores = np.nan_to_num(\n                    _predict_day(model, bars, masks, device=device, batch_size=batch_size),\n                    nan=0.0, posinf=0.0, neginf=0.0,\n                )\n                day_text = signal_day.strftime("%Y-%m-%d")\n                rows.extend({"date": day_text, "instrument": instrument, "score": float(score)}\n                            for instrument, score in zip(instruments, scores))\n            # Avoid retaining even the prior block\'s transformed history while\n            # the next block is being fetched from DAI.\n            del histories, bars, masks, instruments, scores\n\n    output = pd.DataFrame(rows, columns=["date", "instrument", "score"])\n    if not output.empty:\n        output["date"] = pd.to_datetime(output["date"], errors="raise").dt.normalize()\n        output["instrument"] = output["instrument"].astype(str)\n    # The official constituent table, rather than an observed-bar subset, is\n    # the required output denominator.  A short halt or missing intraday bar\n    # must not silently delete a stock from an otherwise valid cross-section.\n    # Zero is the neutral fallback because score heads are trained on centered\n    # residual-return targets; normal model scores remain untouched.\n    output = universe.merge(output, how="left", on=["date", "instrument"])\n    output["score"] = pd.to_numeric(output["score"], errors="coerce")\n    output["score"] = np.nan_to_num(output["score"].to_numpy(dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)\n    output = output.sort_values(["date", "instrument"]).reset_index(drop=True)\n    validate_scores(\n        output,\n        start_date=start_date,\n        end_date=end_date,\n        expected_dates=evaluation_days,\n        expected_instruments_by_date=universe_by_day,\n        min_coverage=0.60,\n    )\n    return output\n\n\n# V6 cloud overrides -------------------------------------------------------\n# Stream model shards directly into a CPU model, then move the completed\n# module to the selected device.  This avoids holding a full decoded state\n# dict and a second full model copy at the same time.\ndef _load_model(checkpoint: dict, *, bars_per_day: int):\n    import gc\n    from model import create_model_from_checkpoint\n\n    model_config = dict(checkpoint["model_config"])\n    model_config["num_fields"] = len(\n        checkpoint["feature_manifest"]["frequencies"]["bar5m"]\n    )\n    model_config["bars_per_day"] = bars_per_day\n    lightweight = dict(checkpoint)\n    lightweight["model_config"] = model_config\n    model = create_model_from_checkpoint(lightweight, load_weights=True)\n    checkpoint.pop("state_dict", None)\n    checkpoint.pop("model_state_dict", None)\n    gc.collect()\n    device = _select_device()\n    try:\n        model = model.to(device)\n    except RuntimeError:\n        device = torch.device("cpu")\n        model = model.to(device)\n    model.eval()\n    return model, device\n\n\ndef _forward_encoder_v6_bounded(\n    model: torch.nn.Module,\n    bars: np.ndarray,\n    masks: np.ndarray,\n    *,\n    device: torch.device,\n    intraday_chunk_size: int = 32,\n) -> torch.Tensor:\n    """Exact V6 encoder with bounded BarEncoder/GRU working memory.\n\n    Intraday encoding is independent for every stock-day.  Flattening that\n    axis and processing small chunks is numerically equivalent to the original\n    forward_encoder, while avoiding a [stocks, days, bars, d_model] activation\n    that killed CPU workers even at 32 stocks.\n    """\n    x = torch.from_numpy(bars).to(device)\n    mask = torch.from_numpy(masks).to(device)\n    batch, days, periods, fields = x.shape\n    if getattr(model, "input_encoder_type", "legacy") != "legacy":\n        return model.forward_encoder(x, mask)[0]\n\n    flat_x = x.reshape(batch * days, periods, fields)\n    flat_mask = mask.reshape(batch * days, periods)\n    day_parts = []\n    for offset in range(0, flat_x.shape[0], intraday_chunk_size):\n        stop = offset + intraday_chunk_size\n        local_x = flat_x[offset:stop].unsqueeze(1)\n        local_mask = flat_mask[offset:stop].unsqueeze(1)\n        embedded = model.bar_encoder(local_x)\n        day_parts.append(model.intraday_encoder(embedded, local_mask)[:, 0])\n    day_repr = torch.cat(day_parts, dim=0).reshape(batch, days, model.d_model)\n    day_valid = (mask.max(dim=-1).values > 0.5).to(day_repr.dtype)\n    temporal = model.temporal_encoder(day_repr, day_valid)\n    valid = day_valid.unsqueeze(-1)\n    return (temporal * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)\n\n\ndef _predict_day(\n    model: torch.nn.Module,\n    bars: np.ndarray,\n    masks: np.ndarray,\n    *,\n    device: torch.device,\n    batch_size: int,\n) -> np.ndarray:\n    if len(bars) == 0:\n        return np.empty(0, dtype=np.float32)\n    # GPU evaluation can use the configured stock batch.  CPU GRU/Transformer\n    # kernels retain large native workspaces, so cap that fallback to a stable\n    # bound rather than letting the worker be killed.\n    effective_batch_size = min(batch_size, 8) if device.type == "cpu" else batch_size\n    if getattr(model, "use_cross_section", False):\n        representations = []\n        for offset in range(0, len(bars), effective_batch_size):\n            encoded = _forward_encoder_v6_bounded(\n                model,\n                bars[offset:offset + effective_batch_size],\n                masks[offset:offset + effective_batch_size],\n                device=device,\n            )\n            representations.append(encoded.detach().cpu())\n        stock_repr = torch.cat(representations, dim=0).to(device)\n        score = model.score_from_stock_repr(\n            stock_repr,\n            distributed_cross_section=False,\n        )\n        return score.detach().cpu().numpy()\n    scores = []\n    for offset in range(0, len(bars), effective_batch_size):\n        encoded = _forward_encoder_v6_bounded(\n            model,\n            bars[offset:offset + effective_batch_size],\n            masks[offset:offset + effective_batch_size],\n            device=device,\n        )\n        scores.append(model.score_head(encoded).detach().cpu().numpy())\n    return np.concatenate(scores)\n', 'ba_trainer': '"""Main trainer for SSPT-30m."""\n\nimport os\nimport time\nimport math\nfrom typing import Dict, Optional, Tuple\nfrom contextlib import nullcontext\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.distributed as dist\nfrom torch.utils.data import DataLoader\nfrom torch.cuda.amp import GradScaler\nfrom torch.amp import autocast\n\nfrom ba_logging import setup_logging, MetricsLogger, CSVLogger\nfrom ba_environment import save_environment_info\nfrom ba_parameters import save_parameter_report\nfrom ba_regression import RegressionLoss\nfrom ba_ic_loss import ICLoss, compute_daily_ic\nfrom ba_pairwise_rank import PairwiseRankingLoss\nfrom ba_tail_loss import TailSeparationLoss\nfrom ba_pretrain_losses import compute_pretrain_losses\nfrom ba_style_neutralize import barra_neutralize_scores, load_exposures as _load_exposures, STYLE_COLUMNS\nfrom ba_cross_section_metrics import evaluate_daily_cross_section\nfrom ba_data_dataset import _effective_neutralization_config\nfrom ba_data_proxy_exposures import build_proxy_exposures\nfrom ba_training_checkpoint import save_checkpoint, load_checkpoint\nfrom ba_date_level import predict_one_date, gather_date_objects, gather_date_tensor\nfrom ba_reproducibility import set_seed\nfrom ba_scheduler import get_cosine_schedule_with_warmup\n\n\nclass Trainer:\n    def __init__(self, model: nn.Module, config: dict, device: torch.device,\n                 train_loader: DataLoader, valid_loader: DataLoader = None,\n                 stage: str = "finetune", is_ddp: bool = False, rank: int = 0):\n        self.model = model\n        self.config = config\n        self.device = device\n        self.train_loader = train_loader\n        self.valid_loader = valid_loader\n        self.stage = stage\n        self.is_ddp = is_ddp\n        self.rank = rank\n        self._raw_model = model.module if hasattr(model, \'module\') else model\n\n        cfg = config["training"]\n        self.epochs = cfg["epochs"]\n        self.batch_size = cfg["batch_size"]\n        self.grad_accum = cfg.get("gradient_accumulation_steps", 1)\n        self.max_grad_norm = cfg.get("max_grad_norm", 1.0)\n        self.log_interval = cfg.get("log_interval", 50)\n        self.eval_interval = cfg.get("eval_interval", 500)\n        self.save_interval = cfg.get("save_interval", 2000)\n        self.eval_every_epochs = max(1, int(cfg.get("eval_every_epochs", 1)))\n        self.precision = cfg.get("precision", "fp32")\n        self.output_dir = cfg.get("output_dir", "outputs")\n        self.run_name = cfg.get("run_name", "run")\n\n        if self.output_dir.startswith("outputs"):\n            base = config.get("_project_root", ".")\n            self.output_dir = os.path.join(base, self.output_dir, self.run_name)\n        os.makedirs(self.output_dir, exist_ok=True)\n\n        self.logger = setup_logging() if rank == 0 else None\n        self.metrics_logger = MetricsLogger(self.output_dir) if rank == 0 else None\n        self.csv_logger = CSVLogger(self.output_dir) if rank == 0 else None\n\n        self.optimizer = torch.optim.AdamW(\n            model.parameters(),\n            lr=cfg.get("lr", 1e-4),\n            weight_decay=cfg.get("weight_decay", 0.01),\n        )\n\n        total_steps = len(train_loader) * self.epochs // self.grad_accum\n        warmup = cfg.get("warmup_epochs", 5) * len(train_loader) // self.grad_accum\n        self.scheduler = get_cosine_schedule_with_warmup(self.optimizer, warmup, total_steps)\n\n        self.scaler = GradScaler() if self.precision == "fp16" else None\n\n        self.reg_loss = RegressionLoss(config["loss"].get("regression_type", "huber"))\n        self.ic_loss = ICLoss()\n        self.rank_loss = PairwiseRankingLoss(\n            config["loss"].get("pair_sample_size", 32768))\n        self.tail_loss = TailSeparationLoss(\n            config["loss"].get("tail_fraction", 0.10),\n            config["loss"].get("tail_margin", 0.50))\n\n        set_seed(cfg.get("seed", 42))\n        save_environment_info(self.output_dir)\n\n        self.global_step = 0\n        self.current_epoch = 0\n        self.best_metric = -float("inf")\n        self.best_by_metric = {\n            "ic": -float("inf"), "icir": -float("inf"),\n            "sharpe": -float("inf"), "stress": -float("inf"),\n            "composite": -float("inf"),\n        }\n        self.selection_history = []\n        self._pretrain_mode = (stage == "pretrain")\n        self.date_level_training = bool(cfg.get("date_level_training", False))\n        self.date_level_validation = bool(\n            cfg.get("date_level_validation", self.date_level_training)\n        )\n        self.local_encoder_chunk_size = int(cfg.get("local_encoder_chunk_size", 0))\n        self.date_level_ddp_mode = str(\n            cfg.get("date_level_ddp_mode", "replicate")\n        ).lower()\n        if self.date_level_ddp_mode not in {"replicate", "shard"}:\n            raise ValueError("training.date_level_ddp_mode must be \'replicate\' or \'shard\'")\n        self._style_exposures_checked = False\n        self._style_exposures = None\n        self._style_neutralization_config = None\n        # Cached (date, instrument) -> exposure row for the differentiable\n        # second-stage risk target.  The pandas implementation remains the\n        # authoritative label/validation contract; this tensor projection is\n        # deliberately a numerically stable training surrogate.\n        self._risk_exposure_map = None\n\n    def _get_style_exposures(self):\n        """Load the exact configured exposure table once per trainer."""\n        if not self._style_exposures_checked:\n            data_cfg = self.config["data"]\n            exposure_path = data_cfg.get("barra_exposure_path") or None\n            self._style_exposures = _load_exposures(\n                data_cfg["data_dir"], exposure_path=exposure_path,\n            )\n            if self._style_exposures is None:\n                proxy_cfg = (\n                    data_cfg.get("barra_neutralization", {})\n                    .get("proxy_exposure", {})\n                )\n                if (\n                    proxy_cfg.get("enabled", False)\n                    and proxy_cfg.get("fallback_when_official_missing", True)\n                ):\n                    dataset = getattr(self.train_loader, "dataset", None)\n                    if dataset is None and self.valid_loader is not None:\n                        dataset = getattr(self.valid_loader, "dataset", None)\n                    index = getattr(dataset, "index", None)\n                    if index is not None:\n                        self._style_exposures = build_proxy_exposures(\n                            data_cfg["data_dir"], index, proxy_cfg,\n                        )\n                        if self.logger:\n                            self.logger.info(\n                                "Using proxy BARRA-like exposures for validation neutralization"\n                            )\n            self._style_neutralization_config = _effective_neutralization_config(\n                data_cfg.get("barra_neutralization", {}),\n                self._style_exposures,\n            )\n            self._style_exposures_checked = True\n        return self._style_exposures\n\n    def _get_style_neutralization_config(self):\n        self._get_style_exposures()\n        return self._style_neutralization_config or {}\n\n    def _prepare_risk_exposure_map(self):\n        if self._risk_exposure_map is not None:\n            return self._risk_exposure_map\n        exposures = self._get_style_exposures()\n        if exposures is None:\n            self._risk_exposure_map = {}\n            return self._risk_exposure_map\n        import pandas as pd\n        frame = exposures.copy()\n        frame["date"] = pd.to_datetime(frame["date"]).dt.normalize()\n        frame["instrument"] = frame["instrument"].astype("string").str.strip()\n        columns = [c for c in STYLE_COLUMNS if c in frame.columns]\n        industry = self._get_style_neutralization_config().get(\n            "industry_column", "industry_level1_code")\n        if not columns:\n            self._risk_exposure_map = {}\n            return self._risk_exposure_map\n        result = {}\n        keep = ["date", "instrument", *columns]\n        if industry in frame.columns:\n            keep.append(industry)\n        weight_column = self._get_style_neutralization_config().get(\n            "weight_column", "weights")\n        if weight_column in frame.columns:\n            keep.append(weight_column)\n        for row in frame[keep].itertuples(index=False, name=None):\n            date, instrument, *values = row\n            result[(pd.Timestamp(date), str(instrument))] = {\n                "styles": np.asarray(values[:len(columns)], dtype=np.float32),\n                "industry": (str(values[len(columns)])\n                             if industry in keep and values[len(columns)] is not None\n                             else ""),\n                "weight": (float(values[-1]) if weight_column in keep\n                           and np.isfinite(values[-1]) and values[-1] > 0 else 1.0),\n            }\n        self._risk_exposure_map = (result, columns, industry)\n        return self._risk_exposure_map\n\n    def _compute_risk_target_loss(self, scores, labels, dates, instruments):\n        """Differentiable approximation of the BARRA score post-processing.\n\n        For each date, build the same intercept/style/industry design used by\n        ``barra_neutralize_scores`` and project the model scores onto its\n        risk-factor span with a torch SVD pseudoinverse.  The loss compares\n        the resulting residual score to the already neutralized return label,\n        so gradients directly discourage style/industry exposure in stage 2.\n        """\n        loss_cfg = self.config.get("loss", {})\n        if not loss_cfg.get("use_risk_neutralized_target", False) or not dates:\n            return scores.sum() * 0.0\n        prepared = self._prepare_risk_exposure_map()\n        if not prepared or not prepared[0]:\n            return scores.sum() * 0.0\n        exposure_map, style_columns, industry_column = prepared\n        import pandas as pd\n        groups = {}\n        for i, date in enumerate(dates):\n            groups.setdefault(pd.Timestamp(date).normalize(), []).append(i)\n        residual_losses = []\n        cfg = self._get_style_neutralization_config()\n        include_industry = bool(cfg.get("include_industry", True))\n        precondition = bool(cfg.get("precondition", True))\n        use_wls = str(cfg.get("regression", "ols")).lower() == "wls"\n        for date, indices in groups.items():\n            rows = [exposure_map.get((date, str(instruments[i])))\n                    for i in indices]\n            valid = [j for j, row in enumerate(rows)\n                     if row is not None and np.isfinite(row["styles"]).all()]\n            if len(valid) < 3:\n                continue\n            x_styles = np.stack([rows[j]["styles"] for j in valid]).astype(np.float32)\n            if precondition and x_styles.shape[1]:\n                mean = x_styles.mean(axis=0, keepdims=True)\n                std = x_styles.std(axis=0, keepdims=True)\n                x_styles = (x_styles - mean) / np.maximum(std, 1e-6)\n            blocks = [np.ones((len(valid), 1), dtype=np.float32)]\n            if x_styles.shape[1]:\n                blocks.append(x_styles)\n            if include_industry:\n                labels_ind = [rows[j]["industry"] for j in valid]\n                categories = sorted({x for x in labels_ind if x})\n                if len(categories) > 1:\n                    blocks.append(np.asarray(\n                        [[float(x == cat) for cat in categories[1:]]\n                         for x in labels_ind], dtype=np.float32))\n            design = torch.as_tensor(np.concatenate(blocks, axis=1),\n                                     device=scores.device, dtype=torch.float32)\n            idx = torch.as_tensor([indices[j] for j in valid], device=scores.device)\n            target_scores = scores.index_select(0, idx).float()\n            target_labels = labels.index_select(0, idx).float()\n            if use_wls:\n                weights = torch.as_tensor([rows[j]["weight"] for j in valid],\n                                          device=scores.device, dtype=torch.float32)\n                sqrt_w = torch.sqrt(weights / weights.mean().clamp_min(1e-6))\n                beta = torch.linalg.pinv(design * sqrt_w[:, None]) @ (target_scores * sqrt_w)\n            else:\n                beta = torch.linalg.pinv(design) @ target_scores\n            residual = target_scores - design @ beta\n            residual_losses.append(self.reg_loss(residual, target_labels))\n        if not residual_losses:\n            return scores.sum() * 0.0\n        return torch.stack(residual_losses).mean()\n\n    def _to_device(self, batch: Dict) -> Dict:\n        result = {}\n        for k, v in batch.items():\n            if isinstance(v, torch.Tensor):\n                result[k] = v.to(self.device)\n            else:\n                result[k] = v\n        return result\n\n    def _autocast_context(self):\n        if self.precision == "bf16" and self.device.type == "cuda":\n            return autocast(device_type="cuda", dtype=torch.bfloat16)\n        if self.precision == "fp16" and self.device.type == "cuda":\n            return autocast(device_type="cuda", dtype=torch.float16)\n        return nullcontext()\n\n    def _predict_finetune_batch(self, batch: Dict, *, validation: bool = False) -> torch.Tensor:\n        """Use the date-level contract when configured, otherwise legacy batching."""\n        use_date_level = self.date_level_validation if validation else self.date_level_training\n        if not use_date_level:\n            model = self._raw_model if validation else self.model\n            return model(\n                batch["bars"], batch.get("day_mask"),\n                distributed_cross_section=False if validation else True,\n            )\n        # Validation runs only on rank 0.  Training in date-level DDP mode\n        # replicates the complete date to every rank, so gathering would\n        # duplicate all stocks; DDP itself handles gradient synchronization.\n        model = self._raw_model if validation else self.model\n        distributed_shard = bool(\n            not validation and self.is_ddp and self.date_level_ddp_mode == "shard"\n        )\n        return predict_one_date(\n            model,\n            batch["bars"],\n            batch.get("day_mask"),\n            local_chunk_size=self.local_encoder_chunk_size,\n            distributed_cross_section=False,\n            distributed_shard=distributed_shard,\n        )\n\n    def _complete_date_supervision(self, batch: Dict, *, validation: bool = False):\n        """Gather labels/metadata to match DDP-sharded complete-date scores."""\n        if (\n            validation or not self.date_level_training or not self.is_ddp\n            or self.date_level_ddp_mode != "shard"\n        ):\n            return batch["labels"], batch.get("dates"), batch.get("instruments")\n        return (\n            gather_date_tensor(batch["labels"]),\n            gather_date_objects(batch.get("dates", [])),\n            gather_date_objects(batch.get("instruments", [])),\n        )\n\n    def _compute_finetune_losses(self, scores: torch.Tensor, labels: torch.Tensor,\n                                  dates: list = None,\n                                  instruments: list = None) -> Tuple[Dict[str, float], torch.Tensor]:\n        loss_cfg = self.config["loss"]\n        losses = {}\n        total = 0.0\n\n        # Huber (reduced weight: raw return prediction is less relevant\n        # since platform evaluates on style-neutralized residuals)\n        reg_loss = self.reg_loss(scores, labels)\n        weight = loss_cfg.get("reg_loss_weight", 0.1)\n        losses["reg_loss"] = reg_loss.item()\n        total += weight * reg_loss\n\n        # Robust IC: compute IC after per-batch winsorize+zscore (mimics platform)\n        if loss_cfg.get("use_ic_loss", True) and dates:\n            unique_dates = sorted(set(dates))\n            date_to_id = {d: i for i, d in enumerate(unique_dates)}\n            date_groups = torch.tensor([date_to_id.get(d, 0) for d in dates],\n                                        device=self.device)\n            ic_l = self.ic_loss(scores, labels, date_groups)\n            weight = loss_cfg.get("ic_loss_weight", 1.0)\n            losses["ic_loss"] = ic_l.item()\n            total += weight * ic_l\n\n        # Diagnostic IC after the same official style+industry post-processing\n        # used for label construction and validation.\n        if loss_cfg.get("use_style_robust_ic", True) and dates:\n            sr_ic = self._compute_style_robust_ic(\n                scores, labels, dates, instruments,\n            )\n            # The official BARRA routine is pandas/numpy based and therefore\n            # intentionally detached from autograd.  Treat it as a diagnostic\n            # until a differentiable torch projection with the official\n            # exposures is available; adding it to ``total`` produced a\n            # constant loss and falsely suggested style robustness was being\n            # optimized.\n            losses["style_robust_ic"] = sr_ic.item()\n\n        if loss_cfg.get("use_pairwise_rank", True):\n            rank_l = self._grouped_loss(self.rank_loss, scores, labels, dates)\n            weight = loss_cfg.get("rank_loss_weight", 0.3)\n            losses["rank_loss"] = rank_l.item()\n            total += weight * rank_l\n\n        if loss_cfg.get("use_tail_loss", True):\n            tail_l = self._grouped_loss(self.tail_loss, scores, labels, dates)\n            weight = loss_cfg.get("tail_loss_weight", 0.1)\n            losses["tail_loss"] = tail_l.item()\n            total += weight * tail_l\n\n        if loss_cfg.get("use_risk_neutralized_target", False) and dates:\n            risk_l = self._compute_risk_target_loss(scores, labels, dates, instruments)\n            weight = loss_cfg.get("risk_neutralized_target_weight", 0.5)\n            losses["risk_neutralized_target"] = risk_l.item()\n            total += weight * risk_l\n\n        losses["total_loss"] = total.item()\n        return losses, total\n\n    def _grouped_loss(self, loss_fn, scores: torch.Tensor, labels: torch.Tensor,\n                      dates: list = None) -> torch.Tensor:\n        """Apply a ranking/tail loss independently within each trading day."""\n        if not dates:\n            return loss_fn(scores, labels)\n        groups = {}\n        for i, date in enumerate(dates):\n            groups.setdefault(str(date), []).append(i)\n        values = []\n        for indices in groups.values():\n            if len(indices) < 2:\n                continue\n            index = torch.tensor(indices, device=scores.device, dtype=torch.long)\n            values.append(loss_fn(scores.index_select(0, index),\n                                  labels.index_select(0, index)))\n        if not values:\n            return scores.sum() * 0.0\n        return torch.stack(values).mean()\n\n    def _compute_style_robust_ic(self, scores, labels, dates, instruments=None):\n        """IC after BARRA style neutralization using official exposures."""\n        try:\n            import pandas as pd\n            exposures = self._get_style_exposures()\n            if exposures is None:\n                return torch.tensor(0.0, device=self.device)\n\n            records = []\n            for i in range(len(scores)):\n                records.append({\n                    "date": dates[i] if isinstance(dates[i], str) else str(dates[i]),\n                    "instrument": str(instruments[i]) if instruments else str(i),\n                    "score": float(scores[i].detach().cpu()),\n                })\n            scores_df = pd.DataFrame(records)\n            result_df, _ = barra_neutralize_scores(\n                scores_df, exposures,\n                config=self._get_style_neutralization_config(),\n            )\n\n            residual_map = {}\n            for _, row in result_df.iterrows():\n                key = (pd.Timestamp(row["date"]).normalize(), str(row["instrument"]))\n                v = row.get("score_residual", np.nan)\n                if pd.notna(v):\n                    residual_map[key] = float(v)\n\n            residual_t = torch.zeros_like(scores)\n            for i in range(len(scores)):\n                instrument = str(instruments[i]) if instruments else str(i)\n                key = (pd.Timestamp(dates[i]).normalize(), instrument)\n                if key in residual_map:\n                    residual_t[i] = residual_map[key]\n\n            mask = (residual_t != 0) | (labels != 0)\n            if mask.sum() < 10:\n                return torch.tensor(0.0, device=self.device)\n            s = residual_t[mask]\n            l = labels[mask]\n            sc = s - s.mean()\n            lc = l - l.mean()\n            sv = (sc * sc).mean()\n            lv = (lc * lc).mean()\n            if sv > 1e-12 and lv > 1e-12:\n                return -(sc * lc).mean() / (torch.sqrt(sv) * torch.sqrt(lv))\n            return torch.tensor(0.0, device=self.device)\n        except Exception:\n            return torch.tensor(0.0, device=self.device)\n\n    def _compute_platform_style_ic(self, scores, labels, dates, instruments):\n        """Validation: compute IC through full BARRA neutralization pipeline.\n\n        This is the actual platform evaluation metric: IC on residuals after\n        winsorize → z-score → BARRA style regression.\n        """\n        try:\n            import pandas as pd\n            exposures = self._get_style_exposures()\n            if exposures is None:\n                return None\n\n            records = []\n            for i in range(len(scores)):\n                records.append({\n                    "date": dates[i] if isinstance(dates[i], str) else str(dates[i]),\n                    "instrument": str(instruments[i]),\n                    "score": float(scores[i].item()),\n                    "label": float(labels[i].item()),\n                })\n            scores_df = pd.DataFrame(records)\n            scores_df["date"] = pd.to_datetime(scores_df["date"]).dt.normalize()\n            scores_df["instrument"] = scores_df["instrument"].astype("string").str.strip()\n            result_df, diag = barra_neutralize_scores(\n                scores_df.rename(columns={"score": "score"})[["date", "instrument", "score"]],\n                exposures,\n                config=self._get_style_neutralization_config(),\n            )\n            merged = scores_df.merge(\n                result_df[["date", "instrument", "score_residual"]],\n                on=["date", "instrument"], how="inner")\n            merged = merged.dropna(subset=["label", "score_residual"])\n\n            ics = []\n            for date, group in merged.groupby("date"):\n                if len(group) < 10:\n                    continue\n                s = group["score_residual"].values\n                l = group["label"].values\n                if np.std(s) > 1e-10 and np.std(l) > 1e-10:\n                    ics.append(np.corrcoef(s, l)[0, 1])\n\n            return float(np.mean(ics)) if ics else None\n        except Exception:\n            return None\n\n    def _postprocess_validation_scores(self, scores, labels, dates, instruments):\n        """Return labels with platform-style residual scores when available."""\n        import pandas as pd\n\n        frame = pd.DataFrame({\n            "date": pd.to_datetime(list(dates)).normalize(),\n            "instrument": [str(value) for value in instruments],\n            # NumPy has no bfloat16 dtype.  Validation intentionally runs\n            # under the same BF16 autocast policy as training, so cast only\n            # at this CPU reporting boundary (not inside the loss path).\n            "score": scores.detach().float().cpu().numpy(),\n            "label": labels.detach().float().cpu().numpy(),\n        })\n        try:\n            exposures = self._get_style_exposures()\n            if exposures is None:\n                return frame, False\n            residuals, _ = barra_neutralize_scores(\n                frame[["date", "instrument", "score"]], exposures,\n                config=self._get_style_neutralization_config(),\n            )\n            merged = frame.merge(\n                residuals[["date", "instrument", "score_residual"]],\n                on=["date", "instrument"], how="inner",\n            ).dropna(subset=["score_residual", "label"])\n            if merged.empty:\n                return frame, False\n            merged = merged.drop(columns=["score"]).rename(columns={"score_residual": "score"})\n            return merged[["date", "instrument", "score", "label"]], True\n        except Exception:\n            return frame, False\n\n    def _selection_metrics(self, metrics: Dict[str, float]) -> Dict[str, float]:\n        """Rank current checkpoint against prior epochs on four local proxies."""\n        platform_ic = metrics.get("valid_platform_ic", float("nan"))\n        values = {\n            "ic": platform_ic if np.isfinite(platform_ic) else metrics.get("valid_ic", float("nan")),\n            "icir": metrics.get("valid_ic_ir", float("nan")),\n            "sharpe": metrics.get("valid_long_short_sharpe", float("nan")),\n            "stress": metrics.get("valid_stress_proxy", float("nan")),\n        }\n        self.selection_history.append(values)\n        ranks = []\n        for name, value in values.items():\n            history = [row[name] for row in self.selection_history if np.isfinite(row[name])]\n            if np.isfinite(value) and history:\n                ranks.append(float(np.mean(np.asarray(history) <= value)))\n        result = dict(values)\n        result["composite"] = float(np.mean(ranks)) if ranks else -float("inf")\n        return result\n\n    def train_epoch(self):\n        self.model.train()\n        epoch_sampler = getattr(self.train_loader, "sampler", None)\n        batch_sampler = getattr(self.train_loader, "batch_sampler", None)\n        if hasattr(epoch_sampler, "set_epoch"):\n            epoch_sampler.set_epoch(self.current_epoch)\n        if hasattr(batch_sampler, "set_epoch"):\n            batch_sampler.set_epoch(self.current_epoch)\n        total_loss = 0.0\n        n_batches = 0\n        start_time = time.time()\n        lr = self.optimizer.param_groups[0]["lr"]\n\n        for batch_idx, batch in enumerate(self.train_loader):\n            batch = self._to_device(batch)\n            n_batches += 1\n\n            with self._autocast_context():\n                if self._pretrain_mode:\n                    # The encoder must see the corrupted input; feeding the\n                    # original bars turns masked reconstruction into an\n                    # identity-style objective.  The previous extra encoder\n                    # pass was unused and doubled pretraining compute.\n                    pretrain_outputs = self._raw_model.forward_pretrain(\n                        batch.get("masked_bars", batch["bars"]),\n                        batch.get("day_mask"),\n                    )\n                    loss_dict, loss = compute_pretrain_losses(\n                        pretrain_outputs, batch, self.config["pretrain"])\n                else:\n                    scores = self._predict_finetune_batch(batch)\n                    labels, dates, instruments = self._complete_date_supervision(batch)\n                    loss_dict, loss = self._compute_finetune_losses(\n                        scores, labels, dates, instruments,\n                    )\n\n            if self.scaler:\n                self.scaler.scale(loss).backward()\n            else:\n                loss.backward()\n\n            if (batch_idx + 1) % self.grad_accum == 0:\n                if self.scaler:\n                    self.scaler.unscale_(self.optimizer)\n                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.max_grad_norm)\n                if self.scaler:\n                    self.scaler.step(self.optimizer)\n                    self.scaler.update()\n                else:\n                    self.optimizer.step()\n                self.scheduler.step()\n                self.optimizer.zero_grad()\n                self.global_step += 1\n\n            total_loss += loss_dict.get("total_loss", loss.item())\n            lr = self.optimizer.param_groups[0]["lr"]\n\n            if batch_idx % self.log_interval == 0 and self.logger is not None:\n                log_msg = f"Epoch {self.current_epoch} Step {self.global_step} "\n                for k, v in loss_dict.items():\n                    log_msg += f"{k}={v:.6f} "\n                log_msg += f"lr={lr:.2e}"\n                self.logger.info(log_msg)\n                self.metrics_logger.log({"lr": lr, **loss_dict}, self.global_step)\n\n        avg_loss = total_loss / max(n_batches, 1)\n        elapsed = time.time() - start_time\n        if self.logger:\n            self.logger.info(f"Epoch {self.current_epoch} avg_loss={avg_loss:.6f} time={elapsed:.1f}s lr={lr:.2e}")\n\n        return avg_loss\n\n    def validate(self) -> Dict[str, float]:\n        if self.valid_loader is None:\n            return {}\n\n        self.model.eval()\n        all_scores = []\n        all_labels = []\n        all_dates = []\n        all_instruments = []\n\n        with torch.no_grad():\n            for batch in self.valid_loader:\n                batch = self._to_device(batch)\n                with self._autocast_context():\n                    scores = self._predict_finetune_batch(batch, validation=True)\n                all_scores.append(scores.cpu())\n                all_labels.append(batch["labels"].cpu())\n                all_dates.extend(batch.get("dates", []))\n                all_instruments.extend(batch.get("instruments", []))\n\n        all_scores = torch.cat(all_scores)\n        all_labels = torch.cat(all_labels)\n\n        # Raw IC\n        # Python hashes are process-randomized and modulo bucketing can merge\n        # unrelated dates.  Use an exact deterministic date mapping.\n        unique_dates = sorted(set(all_dates))\n        date_to_id = {date: idx for idx, date in enumerate(unique_dates)}\n        date_groups = torch.tensor(\n            [date_to_id[date] for date in all_dates], dtype=torch.long,\n        )\n        ic = compute_daily_ic(all_scores, all_labels, date_groups)\n\n        # Platform-style score post-processing is applied once to the full\n        # validation prediction table, then the complete daily cross-section\n        # evaluator derives IC/IR/long-short/stress proxies from the same rows.\n        evaluation_frame, used_platform_postprocess = self._postprocess_validation_scores(\n            all_scores, all_labels, all_dates, all_instruments,\n        )\n        local_metrics, daily_metrics = evaluate_daily_cross_section(evaluation_frame)\n        if self.rank == 0 and self.output_dir and not daily_metrics.empty:\n            daily_metrics.to_csv(\n                os.path.join(self.output_dir, "validation_daily_metrics.csv"), index=False,\n            )\n\n        metrics = {\n            "valid_ic": ic,\n            "valid_platform_ic": local_metrics["ic_mean"] if used_platform_postprocess else float("nan"),\n            "valid_mse": ((all_scores - all_labels) ** 2).mean().item(),\n            "valid_ic_ir": local_metrics["ic_ir"],\n            "valid_rank_ic": local_metrics["rank_ic_mean"],\n            "valid_ic_positive_ratio": local_metrics["ic_positive_ratio"],\n            "valid_long_short_mean": local_metrics["long_short_mean"],\n            "valid_long_short_sharpe": local_metrics["long_short_sharpe"],\n            "valid_stress_proxy": local_metrics["stress_proxy"],\n            "valid_worst_month_ic": local_metrics["worst_month_ic"],\n            "valid_worst_20d_ic": local_metrics["worst_20d_ic"],\n        }\n        if self.logger:\n            platform_text = (\n                "unavailable" if not used_platform_postprocess\n                else f"{metrics[\'valid_platform_ic\']:.6f}"\n            )\n            self.logger.info(\n                f"Valid IC={ic:.6f} PlatformIC={platform_text} "\n                f"ICIR={metrics[\'valid_ic_ir\']:.4f} "\n                f"LSSharpe={metrics[\'valid_long_short_sharpe\']:.4f} "\n                f"Stress={metrics[\'valid_stress_proxy\']:.4f} "\n                f"MSE={metrics[\'valid_mse\']:.6f}"\n            )\n        return metrics\n\n    def train(self):\n        base_model = self.model.module if hasattr(self.model, \'module\') else self.model\n        if self.rank == 0:\n            save_parameter_report(base_model, self.output_dir)\n\n        for epoch in range(self.epochs):\n            self.current_epoch = epoch\n            avg_loss = self.train_epoch()\n\n            # Keep non-zero DDP ranks from entering the next epoch while rank\n            # 0 validates/checkpoints. Market Latent uses cross-rank all-gather,\n            # so mismatched epoch phases would deadlock NCCL.\n            if self.is_ddp and dist.is_initialized():\n                dist.barrier()\n\n            if self.rank == 0:\n                # Persist a recoverable state before rank-0-only validation.\n                # A validation failure must not discard a completed epoch.\n                save_checkpoint(\n                    base_model, self.optimizer, self.scheduler,\n                    epoch, self.global_step, self.config, {},\n                    os.path.join(self.output_dir, "latest_train_state.pt"),\n                )\n\n            if self.rank == 0 and (\n                epoch % self.eval_every_epochs == 0 or epoch == self.epochs - 1\n            ):\n                metrics = self.validate()\n                metrics["train_loss"] = avg_loss\n                selection = self._selection_metrics(metrics)\n                metrics["valid_local_composite"] = selection["composite"]\n                self.csv_logger.log(metrics, epoch)\n\n                checkpoint_names = {\n                    "ic": "best_ic.pt",\n                    "icir": "best_icir.pt",\n                    "sharpe": "best_sharpe.pt",\n                    "stress": "best_stress.pt",\n                    "composite": "best_composite.pt",\n                }\n                for name, value in selection.items():\n                    if not np.isfinite(value) or value <= self.best_by_metric[name]:\n                        continue\n                    self.best_by_metric[name] = value\n                    save_checkpoint(\n                        base_model, self.optimizer, self.scheduler,\n                        epoch, self.global_step, self.config,\n                        metrics,\n                        os.path.join(self.output_dir, checkpoint_names[name]))\n                    self.logger.info(\n                        f"Saved {checkpoint_names[name]} at epoch {epoch} "\n                        f"({name}={value:.6f})"\n                    )\n\n                # Compatibility alias for existing export/submit tooling.\n                if selection["composite"] > self.best_metric:\n                    self.best_metric = selection["composite"]\n                    save_checkpoint(\n                        base_model, self.optimizer, self.scheduler,\n                        epoch, self.global_step, self.config,\n                        metrics,\n                        os.path.join(self.output_dir, "best_model.pt"))\n\n            if self.rank == 0 and (epoch % 10 == 0 or epoch == self.epochs - 1):\n                save_checkpoint(\n                    base_model, self.optimizer, self.scheduler,\n                    epoch, self.global_step, self.config,\n                    {},\n                    os.path.join(self.output_dir, f"checkpoint_epoch{epoch}.pt"))\n\n            if self.is_ddp and dist.is_initialized():\n                dist.barrier()\n\n        if self.rank == 0:\n            save_checkpoint(\n                base_model, self.optimizer, self.scheduler,\n                self.epochs - 1, self.global_step, self.config,\n                {},\n                os.path.join(self.output_dir, "last_model.pt"))\n            self.csv_logger.close()\n            self.metrics_logger.close()\n            self.logger.info(f"Training complete. Outputs saved to {self.output_dir}")\n', 'train': '#!/usr/bin/env python\n"""Main training entry point for BigAlpha SSPT-30m. Supports single-GPU and multi-GPU (DDP)."""\n\nimport os\nimport sys\nimport argparse\nfrom datetime import timedelta\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader\nfrom torch.utils.data.distributed import DistributedSampler\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n\nfrom ba_config import load_config, save_resolved_config\nfrom ba_logging import setup_logging\nfrom ba_data_schema import DEFAULT_SELECTED_FIELDS\nfrom ba_data_indexing import DataIndex, DataLoader as DataLoader_\nfrom ba_preprocessing import FieldScaler, LogTransform\nfrom ba_data_dataset import SSFTDataset, PretrainDataset\nfrom ba_data_collate import collate_fn_sft, collate_fn_pretrain\nfrom ba_sspt30m import SSPT30mModel\nfrom ba_trainer import Trainer\nfrom ba_reproducibility import set_seed\nfrom ba_data_samplers import DateBatchSampler\n\n\ndef fit_scaler_representatively(loader, index, train_dates, selected_fields, scaler, config):\n    """Fit fixed preprocessing statistics across the whole training horizon.\n\n    The former first-200-days/first-50-identifiers loop biased statistics\n    toward early market regimes and code-sorted stocks.  This deterministic\n    stratified sample spans the date axis and randomly samples each selected\n    day\'s available universe using the training seed.\n    """\n    if not train_dates:\n        raise RuntimeError("cannot fit scaler without training dates")\n    fields_cfg = config.get("fields", {}).get("preprocessing", {})\n    n_dates = min(len(train_dates), int(fields_cfg.get("scaler_fit_dates", 48)))\n    n_instruments = int(fields_cfg.get("scaler_fit_instruments_per_date", 64))\n    max_rows = int(fields_cfg.get("scaler_fit_max_rows", 2_000_000))\n    positions = np.unique(np.linspace(0, len(train_dates) - 1, num=max(1, n_dates), dtype=int))\n    rng = np.random.default_rng(int(config["training"].get("seed", 42)))\n    samples = []\n    total_rows = 0\n    for position in positions:\n        date = train_dates[int(position)]\n        universe = np.asarray(index.date_instruments.get(date, []))\n        if universe.size == 0:\n            continue\n        take = min(n_instruments, universe.size)\n        instruments = rng.choice(universe, size=take, replace=False)\n        day_data = loader.get_daily_data(date, instruments.tolist(), selected_fields)\n        if day_data.empty:\n            continue\n        values = day_data[selected_fields].to_numpy(dtype=np.float32, copy=True)\n        samples.append(values)\n        total_rows += len(values)\n        if total_rows >= max_rows:\n            break\n    if not samples:\n        raise RuntimeError("representative scaler fitting found no valid bar rows")\n    stacked = np.concatenate(samples, axis=0)[:max_rows]\n    scaler.fit(stacked, selected_fields)\n    return {"sampled_dates": int(len(positions)), "sampled_rows": int(len(stacked))}\n\n\ndef setup_ddp():\n    if "RANK" in os.environ and "WORLD_SIZE" in os.environ:\n        rank = int(os.environ["RANK"])\n        world_size = int(os.environ["WORLD_SIZE"])\n        local_rank = int(os.environ["LOCAL_RANK"])\n        dist.init_process_group(backend="nccl", timeout=timedelta(hours=2))\n        torch.cuda.set_device(local_rank)\n        return rank, world_size, local_rank, True\n    return 0, 1, 0, False\n\n\ndef cleanup_ddp():\n    if dist.is_initialized():\n        dist.destroy_process_group()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Train SSPT-30m model")\n    parser.add_argument("--config", type=str, required=True)\n    args = parser.parse_args()\n\n    rank, world_size, local_rank, is_ddp = setup_ddp()\n    config = load_config(args.config)\n\n    device = torch.device(f"cuda:{local_rank}")\n    memory_fraction = config["training"].get("max_memory_fraction")\n    if memory_fraction is not None and device.type == "cuda":\n        torch.cuda.set_per_process_memory_fraction(float(memory_fraction), device=local_rank)\n    torch.set_float32_matmul_precision("high")\n    torch.backends.cuda.enable_flash_sdp(True)\n\n    if rank == 0:\n        logger = setup_logging()\n        logger.info(f"DDP: {is_ddp}, world_size={world_size}, device={device}")\n        logger.info(f"Config: {args.config}")\n\n    set_seed(config["training"]["seed"] + rank)\n\n    # Build data index (rank 0 builds + caches, others load from cache)\n    data_dir = config["data"]["data_dir"]\n    stage = config.get("stage", "finetune")\n\n    if rank == 0:\n        index = DataIndex(data_dir)\n        index_info = index.build()\n        logger.info(f"Data index: {index_info}")\n    if is_ddp:\n        dist.barrier()\n    if rank != 0:\n        index = DataIndex(data_dir)\n        index_info = index.build()  # loads from cache\n        if rank == 1:\n            pass  # suppress log from non-rank-0\n\n    loader = DataLoader_(data_dir, index)\n    selected_fields = DEFAULT_SELECTED_FIELDS\n    lookback = config["data"]["lookback_days"]\n    min_hist = config["data"]["min_history_days"]\n\n    # Fit scaler on training dates (only rank 0 fits, then broadcast)\n    train_dates = index.get_trading_dates(\n        config["split"]["train_start"], config["split"]["train_end"])\n\n    fields_config = config.get("fields", {})\n    log_fields = fields_config.get("log_transform", [])\n    preprocessing_config = fields_config.get("preprocessing", {})\n    clip_quantiles = None\n    if preprocessing_config.get("clip_outliers", False):\n        clip_quantiles = (\n            float(preprocessing_config.get("clip_quantile_low", 0.001)),\n            float(preprocessing_config.get("clip_quantile_high", 0.999)),\n        )\n    scaler = FieldScaler(\n        scaler_type=preprocessing_config.get("scaler", "standard"),\n        min_std=float(preprocessing_config.get("min_std_threshold", 1e-8)),\n        log_fields=log_fields,\n        clip_quantiles=clip_quantiles,\n    )\n    scaler_path = os.path.join(config.get("_project_root", "."),\n                               "artifacts/preprocessing/scaler.json")\n\n    if rank == 0:\n        load_cached = False\n        if os.path.exists(scaler_path):\n            scaler.load(scaler_path)\n            load_cached = (\n                scaler.representation == "canonical_yuan"\n                and scaler.log_fields == set(log_fields)\n                and tuple(scaler.clip_quantiles or ()) == tuple(clip_quantiles or ())\n            )\n        if load_cached:\n            logger.info("Canonical scaler loaded from cache")\n        else:\n            if os.path.exists(scaler_path):\n                logger.info("Existing scaler metadata is stale; refitting on canonical training data")\n            logger.info(f"Fitting scaler on training data...")\n            fit_info = fit_scaler_representatively(\n                loader, index, train_dates, selected_fields, scaler, config,\n            )\n            scaler.save(scaler_path)\n            logger.info("Scaler fitted and saved: %s", fit_info)\n\n    if is_ddp:\n        dist.barrier()\n        if rank != 0:\n            scaler.load(scaler_path)\n\n    # Carry the exact training-only preprocessing statistics into every\n    # checkpoint; cloud inference must never refit or guess these values.\n    config["_preprocessor"] = scaler.to_dict()\n    config["_preprocessor"]["field_order"] = list(selected_fields)\n\n    # Build dataset\n    batch_size = config["training"]["batch_size"]\n    barra_config = config["data"].get("barra_neutralization", {})\n\n    if stage == "pretrain":\n        dataset = PretrainDataset(\n            index, data_dir, train_dates, selected_fields,\n            scaler, lookback, min_hist,\n            ma_windows=config["pretrain"].get("ma_windows", []),\n            mask_ratio=config["pretrain"].get("mask_ratio", 0.20),\n            max_label_date=train_dates[-1] if train_dates else None,\n            ma_target_type=config["pretrain"].get("ma_target_type", "relative"),\n            include_next_day_target=config["pretrain"].get("use_next_day_prediction", True),\n            barra_exposure_path=config["data"].get("barra_exposure_path") or None,\n            barra_neutralization_config=barra_config,\n        )\n        collate_fn = collate_fn_pretrain\n    else:\n        dataset = SSFTDataset(\n            index, data_dir, train_dates, selected_fields,\n            scaler, lookback, min_hist,\n            # Fine-tuning never consumes MA reconstruction targets.  Building\n            # them for every stock/date is needless CPU and memory pressure.\n            ma_windows=config.get("finetune", {}).get("ma_windows", []),\n            max_label_date=train_dates[-1] if train_dates else None,\n            ma_target_type=config["pretrain"].get("ma_target_type", "relative"),\n            barra_exposure_path=config["data"].get("barra_exposure_path") or None,\n            barra_neutralization_config=barra_config,\n        )\n        collate_fn = collate_fn_sft\n\n    if len(dataset) == 0:\n        raise RuntimeError(\n            "No training samples remain after history/future-label filtering; "\n            "check data dates, split boundaries, and min_history_days."\n        )\n    if stage != "pretrain":\n        # Persist the exact label source/availability in checkpoints so a\n        # later export cannot silently be mistaken for a raw-return run.\n        exposure_source = dataset.residual_r2_stats.get("exposure_source")\n        if dataset.residual_r2_stats.get("available") and exposure_source == "proxy_barra_like":\n            label_method = "proxy_barra_like_ols_residual"\n        elif dataset.residual_r2_stats.get("available"):\n            label_method = "official_barra_ols_residual"\n        else:\n            label_method = "raw_return_fallback"\n        config["label_contract"] = {\n            "method": label_method,\n            "exposure_path": config["data"].get("barra_exposure_path") or os.environ.get("BIGALPHA_BARRA_EXPOSURE_PATH"),\n            **dataset.residual_r2_stats,\n        }\n    if rank == 0:\n        logger.info(f"Train samples: {len(dataset)}")\n\n    date_level_training = bool(config["training"].get("date_level_training", False))\n    date_level_validation = bool(\n        config["training"].get("date_level_validation", date_level_training)\n    )\n    if date_level_training and stage == "pretrain":\n        raise ValueError("date_level_training is a supervised fine-tuning contract, not pretraining")\n    date_level_ddp_mode = str(\n        config["training"].get("date_level_ddp_mode", "replicate")\n    ).lower()\n    if date_level_ddp_mode not in {"replicate", "shard"}:\n        raise ValueError("training.date_level_ddp_mode must be \'replicate\' or \'shard\'")\n    group_by_date = config["training"].get("group_by_date", False) or date_level_training\n    pad_date_batches = config["training"].get("pad_date_batches", False)\n    batch_sampler = (\n        DateBatchSampler(\n            dataset, batch_size,\n            seed=config["training"].get("seed", 42),\n            # Sharded date-level mode uses disjoint local stock shards and\n            # model-side ragged all-gather; replicate remains an audit\n            # baseline. Neither mode repeats tail stocks.\n            world_size=(world_size if (date_level_training and date_level_ddp_mode == "shard")\n                        else (1 if date_level_training else (world_size if is_ddp else 1))),\n            rank=(rank if (date_level_training and date_level_ddp_mode == "shard")\n                  else (0 if date_level_training else rank)),\n            drop_last=False if date_level_training else not pad_date_batches,\n            pad_to_global_batch=False if date_level_training else pad_date_batches,\n            full_date=date_level_training,\n            shard_full_date=(date_level_training and date_level_ddp_mode == "shard"),\n        )\n        if group_by_date else None\n    )\n    loader_kwargs = {\n        "num_workers": config["training"].get("num_workers", 4),\n        "pin_memory": config["training"].get("pin_memory", True),\n        "persistent_workers": config["training"].get("persistent_workers", True),\n        "collate_fn": collate_fn,\n    }\n    if batch_sampler is not None:\n        # batch_sampler is mutually exclusive with batch_size/shuffle/sampler/\n        # drop_last in torch DataLoader; do not pass those default-looking\n        # arguments explicitly.\n        train_loader = DataLoader(dataset, batch_sampler=batch_sampler, **loader_kwargs)\n    else:\n        train_sampler = (\n            DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)\n            if is_ddp else None\n        )\n        train_loader = DataLoader(\n            dataset, batch_size=batch_size, sampler=train_sampler,\n            shuffle=(train_sampler is None), drop_last=True, **loader_kwargs,\n        )\n\n    # Build validation loader only on rank 0 for evaluation\n    valid_dates = index.get_trading_dates(\n        config["split"]["valid_start"], config["split"]["valid_end"])\n    valid_dataset = SSFTDataset(\n        index, data_dir, valid_dates, selected_fields,\n        scaler, lookback, min_hist,\n        max_label_date=valid_dates[-1] if valid_dates else None,\n        ma_windows=config.get("finetune", {}).get("ma_windows", []),\n        barra_exposure_path=config["data"].get("barra_exposure_path") or None,\n        barra_neutralization_config=barra_config,\n    )\n    if len(valid_dataset) == 0:\n        raise RuntimeError(\n            "No validation samples remain after history/future-label filtering; "\n            "check validation dates and min_history_days."\n        )\n    if rank == 0:\n        valid_kwargs = {\n            "dataset": valid_dataset,\n            "num_workers": 2,\n            "pin_memory": True,\n            "collate_fn": collate_fn_sft,\n        }\n        if date_level_validation:\n            valid_loader = DataLoader(\n                batch_sampler=DateBatchSampler(\n                    valid_dataset, batch_size,\n                    shuffle=False, drop_last=False, seed=config["training"].get("seed", 42),\n                    full_date=True,\n                ),\n                **valid_kwargs,\n            )\n        else:\n            valid_loader = DataLoader(\n                batch_size=min(batch_size * 2, config["training"].get("valid_batch_size", 512)),\n                shuffle=False, drop_last=False, **valid_kwargs,\n            )\n    else:\n        valid_loader = None\n\n    # Build model\n    model_config = config.copy()\n    model_config["model"]["num_fields"] = len(selected_fields)\n    model_config["model"]["bars_per_day"] = config["data"].get("bars_per_day", 8)\n    model = SSPT30mModel(model_config).to(device)\n\n    if stage == "pretrain":\n        model.add_pretrain_heads(config["pretrain"])\n        model = model.to(device)\n\n    # Compile only the stock-local encoder.  The full date path contains\n    # ragged distributed collectives, which should remain eager; compiling\n    # this local portion still covers BarEncoder + intraday GRU + temporal\n    # Transformer and is safe for date-dependent local shard sizes.\n    if config["training"].get("compile", False):\n        compile_mode = config["training"].get("compile_mode", "reduce-overhead")\n        compile_dynamic = bool(config["training"].get("compile_dynamic", True))\n        try:\n            model.forward_encoder = torch.compile(\n                model.forward_encoder,\n                dynamic=compile_dynamic,\n                mode=compile_mode,\n            )\n            if rank == 0:\n                logger.info(\n                    "Compiled stock-local encoder: mode=%s dynamic=%s",\n                    compile_mode, compile_dynamic,\n                )\n        except Exception as exc:\n            if rank == 0:\n                logger.warning("torch.compile unavailable; using eager encoder: %s", exc)\n\n    # Load pretrained checkpoint if specified\n    pretrained_path = config["training"].get("pretrained_checkpoint", "")\n    if pretrained_path and not os.path.isabs(pretrained_path):\n        pretrained_path = os.path.join(config.get("_project_root", "."), pretrained_path)\n    if pretrained_path and os.path.exists(pretrained_path):\n        if rank == 0:\n            logger.info(f"Loading pretrained weights from {pretrained_path}")\n        checkpoint = torch.load(pretrained_path, map_location=device)\n        model_state = checkpoint.get("model_state_dict", checkpoint)\n        model.load_state_dict(model_state, strict=False)\n\n    if is_ddp:\n        model = DDP(model, device_ids=[local_rank], output_device=local_rank,\n                     find_unused_parameters=(stage == "pretrain"))\n\n    # Save resolved config\n    if rank == 0:\n        output_dir = os.path.join(config.get("_project_root", "."),\n                                   config["training"].get("output_dir", "outputs"),\n                                   config["training"].get("run_name", "run"))\n        os.makedirs(output_dir, exist_ok=True)\n        save_resolved_config(config, os.path.join(output_dir, "config_resolved.yaml"))\n\n    # Train\n    trainer = Trainer(model, config, device, train_loader, valid_loader,\n                       stage=stage, is_ddp=is_ddp, rank=rank)\n    trainer.train()\n\n    cleanup_ddp()\n\n\nif __name__ == "__main__":\n    main()\n'}


def _install_embedded_modules() -> None:
    for name, source in EMBEDDED_MODULE_SOURCES.items():
        sys.modules.pop(name, None)
        module = types.ModuleType(name)
        module.__file__ = str(PACKAGE_ROOT / (name + ".py"))
        module.__package__ = ""
        sys.modules[name] = module
        try:
            exec(compile(source, module.__file__, "exec"), module.__dict__)
        except Exception:
            sys.modules.pop(name, None)
            raise


_install_embedded_modules()


def main(datasources, start_date, end_date):
    """Official entrypoint: returns exactly date, instrument, score."""
    from inference import run_cloud_inference
    return run_cloud_inference(
        datasources=datasources,
        start_date=start_date,
        end_date=end_date,
        checkpoint_path="model_meta.pt",
        # 5m windows have six times more intraday cells than v4.  This keeps
        # CPU cloud inference bounded while preserving complete-date Latents.
        batch_size=64,
        # Reuse each 60-day history query across several adjacent signal days.
        # A one-day block repeated almost the same multi-million-row DAI query
        # for every public-board day and routinely timed out.
        query_chunk_days=10,
        instrument_query_chunk_size=128,
    )
